# 10. Production Model Refit

## Purpose

The previous notebook identified the Independent Poisson model as the preferred forecasting approach and fixed every aspect of the production specification, including the modelling assumptions, preprocessing pipeline, feature definitions and regularisation settings.

This notebook now refits that fixed specification using all eligible historical Premier League fixtures available before the forecasting period. Unlike the earlier modelling notebooks, no model selection or hyperparameter optimisation is performed. Instead, the objective is to produce the final production-ready model artefacts that will be used to generate forecasts for future Premier League fixtures.

At the end of this notebook, the fitted Independent Poisson regression models, preprocessing objects and supporting metadata are exported to ensure that all subsequent forecasts are generated from a single validated and reproducible production pipeline.

## 2. Load and Validate the Production Configuration

This section locates the frozen production specification produced during final model selection and validates the high-level decisions governing the production refit. The selected model, calibration method, probability order, bookmaker-predictor restriction, random seed and required prediction outputs are treated as fixed.

If no exported specification is available, only the established high-level configuration is reconstructed and saved. Exact implementation details—including the feature lists and ordering, preprocessing sequence, regularisation parameters, solver configuration and scoreline truncation rule—remain unresolved at this stage and must be recovered from the final validated Independent Poisson implementation before fitting the production models.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

REPOSITORY_NAME = "premier-league-probability-engine"
EXPECTED_PROBABILITY_ORDER = ["H", "D", "A"]
EXPECTED_RANDOM_SEED = 42

EXPECTED_OUTPUT_FIELDS = [
    "HomeExpectedGoals",
    "AwayExpectedGoals",
    "Probability_H",
    "Probability_D",
    "Probability_A",
]


# ---------------------------------------------------------------------
# Project-root detection
# ---------------------------------------------------------------------

def detect_project_root(
    start_path: Path,
    repository_name: str,
) -> Path:
    """
    Locate the project root without assuming that the notebook was opened
    from a particular working directory.
    """
    start_path = start_path.expanduser().resolve()
    ancestors = [start_path, *start_path.parents]

    # Highest-confidence case: the current directory or one of its parents
    # has the repository's expected name.
    for candidate in ancestors:
        if candidate.name == repository_name:
            return candidate

    # The repository may be a direct child of the current directory or one
    # of its nearby parents.
    for parent in ancestors[:5]:
        candidate = parent / repository_name
        if candidate.is_dir():
            return candidate.resolve()

    # Fall back to project-structure evidence.
    def project_structure_score(candidate: Path) -> int:
        score = 0

        if (candidate / ".git").exists():
            score += 4
        if (candidate / "outputs").is_dir():
            score += 2
        if (candidate / "notebooks").is_dir():
            score += 2
        if (candidate / "data").is_dir():
            score += 1
        if (candidate / "README.md").is_file():
            score += 1
        if any(candidate.glob("*.ipynb")):
            score += 1

        return score

    ranked_candidates = sorted(
        (
            (project_structure_score(candidate), candidate)
            for candidate in ancestors
        ),
        key=lambda item: item[0],
        reverse=True,
    )

    best_score, best_candidate = ranked_candidates[0]

    if best_score >= 4:
        return best_candidate.resolve()

    searched_locations = "\n".join(f"- {path}" for path in ancestors)

    raise FileNotFoundError(
        "Unable to identify the project root reliably.\n"
        f"Expected repository name: {repository_name!r}\n"
        f"Notebook working directory: {start_path}\n"
        "Searched the following locations:\n"
        f"{searched_locations}"
    )


project_root = detect_project_root(
    start_path=Path.cwd(),
    repository_name=REPOSITORY_NAME,
)

outputs_directory = project_root / "outputs"
production_output_directory = outputs_directory / "production_model"
production_output_directory.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Locate an exported production specification
# ---------------------------------------------------------------------

preferred_configuration_paths = [
    outputs_directory
    / "final_model_selection"
    / "production_specification.json",

    production_output_directory
    / "production_specification.json",
]

discovered_configuration_paths = (
    sorted(outputs_directory.glob("**/production_specification.json"))
    if outputs_directory.exists()
    else []
)

configuration_candidates: list[Path] = []

for path in [
    *preferred_configuration_paths,
    *discovered_configuration_paths,
]:
    resolved_path = path.resolve()

    if resolved_path not in configuration_candidates:
        configuration_candidates.append(resolved_path)

existing_configuration_paths = [
    path
    for path in configuration_candidates
    if path.is_file()
]


def load_json_object(path: Path) -> dict[str, Any]:
    """Load a JSON file and require its top-level value to be an object."""
    try:
        loaded_object = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as error:
        raise ValueError(
            f"Configuration file is not valid JSON: {path}\n"
            f"JSON error: {error}"
        ) from error

    if not isinstance(loaded_object, dict):
        raise TypeError(
            "The production specification must be a JSON object, "
            f"but {path} contains {type(loaded_object).__name__}."
        )

    return loaded_object


configuration_load_errors: list[str] = []
production_specification: dict[str, Any] | None = None
configuration_path: Path | None = None
configuration_source: str | None = None

for candidate_path in existing_configuration_paths:
    try:
        production_specification = load_json_object(candidate_path)
        configuration_path = candidate_path
        configuration_source = "Existing exported JSON specification"
        break
    except (OSError, ValueError, TypeError) as error:
        configuration_load_errors.append(
            f"{candidate_path}: {error}"
        )


if (
    existing_configuration_paths
    and production_specification is None
):
    error_details = "\n\n".join(configuration_load_errors)

    raise RuntimeError(
        "Production specification files were found, but none could be "
        "loaded safely. The files have not been replaced automatically.\n\n"
        f"{error_details}"
    )


# ---------------------------------------------------------------------
# Transparent high-level fallback
# ---------------------------------------------------------------------

if production_specification is None:
    production_specification = {
        "SpecificationVersion": "reconstructed-high-level-v1",
        "SelectedModel": "Independent Poisson",
        "ModelFamily": "Independent Poisson regression",
        "HomeGoalModel": "Poisson regression",
        "AwayGoalModel": "Poisson regression",
        "DependenceAssumption": (
            "Conditional independence of home and away goals"
        ),
        "ProbabilityConstruction": (
            "Aggregate the joint scoreline distribution into H/D/A "
            "result probabilities"
        ),
        "ProbabilityOrder": EXPECTED_PROBABILITY_ORDER,
        "CalibrationMethod": "Original probabilities",
        "PrimaryEvaluationMetric": "Multiclass log loss",
        "SecondaryEvaluationMetrics": [
            "Brier score",
            "Accuracy",
            "Calibration error",
        ],
        "TrainingPolicy": (
            "Refit the frozen specification using all eligible completed "
            "historical fixtures before the forecasting period; perform "
            "no model selection or hyperparameter optimisation"
        ),
        "FeatureTimingPolicy": (
            "Use only information available before kickoff and require "
            "training fixtures to precede the prediction date"
        ),
        "BookmakerOddsUsedAsPredictors": False,
        "MarketBenchmark": "Pinnacle closing probabilities",
        "RequiredPredictionOutputs": EXPECTED_OUTPUT_FIELDS,
        "RandomSeed": EXPECTED_RANDOM_SEED,
        "ImplementationState": (
            "High-level production choices frozen; exact estimator and "
            "feature implementation pending recovery from the final "
            "validated Notebook 5/6 code"
        ),
    }

    configuration_path = (
        production_output_directory
        / "production_specification.json"
    )

    configuration_path.write_text(
        json.dumps(
            production_specification,
            indent=4,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    configuration_source = (
        "Reconstructed from the fixed Notebook 9 decisions because no "
        "exported production specification was found"
    )


assert production_specification is not None
assert configuration_path is not None
assert configuration_source is not None


# ---------------------------------------------------------------------
# Validate the frozen high-level production contract
# ---------------------------------------------------------------------

expected_configuration_values: dict[str, Any] = {
    "SelectedModel": "Independent Poisson",
    "CalibrationMethod": "Original probabilities",
    "ProbabilityOrder": EXPECTED_PROBABILITY_ORDER,
    "BookmakerOddsUsedAsPredictors": False,
    "RandomSeed": EXPECTED_RANDOM_SEED,
    "RequiredPredictionOutputs": EXPECTED_OUTPUT_FIELDS,
}

validation_records: list[dict[str, Any]] = []
validation_errors: list[str] = []

for field_name, expected_value in expected_configuration_values.items():
    field_exists = field_name in production_specification
    actual_value = production_specification.get(field_name)

    # Require a genuine Boolean for this field rather than accepting 0.
    if field_name == "BookmakerOddsUsedAsPredictors":
        value_matches = (
            field_exists
            and isinstance(actual_value, bool)
            and actual_value is expected_value
        )
    else:
        value_matches = (
            field_exists
            and actual_value == expected_value
        )

    validation_records.append(
        {
            "ConfigurationField": field_name,
            "ExpectedValue": expected_value,
            "ActualValue": actual_value,
            "Status": "PASS" if value_matches else "FAIL",
        }
    )

    if not field_exists:
        validation_errors.append(
            f"Missing required configuration field: {field_name}"
        )
    elif not value_matches:
        validation_errors.append(
            f"{field_name} must equal {expected_value!r}, "
            f"but the loaded value is {actual_value!r}."
        )


if validation_errors:
    formatted_errors = "\n".join(
        f"- {error}"
        for error in validation_errors
    )

    raise ValueError(
        "The loaded production specification does not match the frozen "
        "Notebook 9 decisions:\n"
        f"{formatted_errors}"
    )


# Additional structural checks for the probability-output contract.
required_outputs = production_specification[
    "RequiredPredictionOutputs"
]

assert required_outputs[-3:] == [
    "Probability_H",
    "Probability_D",
    "Probability_A",
], (
    "The probability output columns must remain ordered as "
    "Probability_H, Probability_D, Probability_A."
)

assert production_specification["ProbabilityOrder"] == ["H", "D", "A"]
assert production_specification["BookmakerOddsUsedAsPredictors"] is False
assert production_specification["RandomSeed"] == 42


# ---------------------------------------------------------------------
# Separate fixed choices from implementation details still to recover
# ---------------------------------------------------------------------

implementation_recovery_requirements = [
    "Home-goal feature columns",
    "Away-goal feature columns",
    "Exact feature ordering",
    "Rank-deficient columns removed",
    "Missing-value treatment",
    "Scaling and encoding sequence",
    "Team mappings",
    "Promoted-team handling",
    "Home Poisson regularisation parameter",
    "Away Poisson regularisation parameter",
    "Solver, maximum-iteration and tolerance settings",
    "Scoreline truncation rule",
]

implementation_status_table = pd.DataFrame(
    {
        "ImplementationDetail": implementation_recovery_requirements,
        "Status": "Pending validated recovery",
        "RequiredSource": (
            "Final corrected Notebook 5/6 implementation"
        ),
    }
)

validation_table = pd.DataFrame(validation_records)

configuration_table = pd.DataFrame(
    [
        {
            "ConfigurationField": field_name,
            "ConfiguredValue": (
                json.dumps(value)
                if isinstance(value, (list, dict))
                else value
            ),
        }
        for field_name, value in production_specification.items()
    ]
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print(f"Project root: {project_root}")
print(
    "Production output directory: "
    f"{production_output_directory}"
)
print(f"Configuration source: {configuration_source}")
print(f"Configuration path: {configuration_path}")
print("\nHigh-level production configuration validated successfully.")

print("\nLoaded production specification:")
display(configuration_table)

print("\nValidation checks:")
display(validation_table)

print(
    "\nExact implementation details that must still be recovered "
    "before the production refit:"
)
display(implementation_status_table)

Project root: C:\Users\kiera\OneDrive\Desktop\premier-league-probability-engine
Production output directory: C:\Users\kiera\OneDrive\Desktop\premier-league-probability-engine\outputs\production_model
Configuration source: Existing exported JSON specification
Configuration path: C:\Users\kiera\OneDrive\Desktop\premier-league-probability-engine\outputs\production_model\production_specification.json

High-level production configuration validated successfully.

Loaded production specification:


,ConfigurationField,ConfiguredValue
0,SpecificationVersion,reconstructed-high-level-v1
1,SelectedModel,Independent Poisson
2,ModelFamily,Independent Poisson regression
3,HomeGoalModel,Poisson regression
4,AwayGoalModel,Poisson regression
5,DependenceAssumption,Conditional independence of home and away goals
6,ProbabilityConstruction,Aggregate the joint scoreline distribution int...
7,ProbabilityOrder,"[""H"", ""D"", ""A""]"
8,CalibrationMethod,Original probabilities
9,PrimaryEvaluationMetric,Multiclass log loss



Validation checks:


,ConfigurationField,ExpectedValue,ActualValue,Status
0,SelectedModel,Independent Poisson,Independent Poisson,PASS
1,CalibrationMethod,Original probabilities,Original probabilities,PASS
2,ProbabilityOrder,"[H, D, A]","[H, D, A]",PASS
3,BookmakerOddsUsedAsPredictors,False,False,PASS
4,RandomSeed,42,42,PASS
5,RequiredPredictionOutputs,"[HomeExpectedGoals, AwayExpectedGoals, Probabi...","[HomeExpectedGoals, AwayExpectedGoals, Probabi...",PASS



Exact implementation details that must still be recovered before the production refit:


,ImplementationDetail,Status,RequiredSource
0,Home-goal feature columns,Pending validated recovery,Final corrected Notebook 5/6 implementation
1,Away-goal feature columns,Pending validated recovery,Final corrected Notebook 5/6 implementation
2,Exact feature ordering,Pending validated recovery,Final corrected Notebook 5/6 implementation
3,Rank-deficient columns removed,Pending validated recovery,Final corrected Notebook 5/6 implementation
4,Missing-value treatment,Pending validated recovery,Final corrected Notebook 5/6 implementation
5,Scaling and encoding sequence,Pending validated recovery,Final corrected Notebook 5/6 implementation
6,Team mappings,Pending validated recovery,Final corrected Notebook 5/6 implementation
7,Promoted-team handling,Pending validated recovery,Final corrected Notebook 5/6 implementation
8,Home Poisson regularisation parameter,Pending validated recovery,Final corrected Notebook 5/6 implementation
9,Away Poisson regularisation parameter,Pending validated recovery,Final corrected Notebook 5/6 implementation


### Results and Interpretation

The production configuration was loaded successfully using the reconstructed high-level specification, identified as `reconstructed-high-level-v1`. This confirms that a complete exported specification from Notebook 9 was not required to preserve the established model-selection decisions.

All six validation checks passed. The production model is fixed as Independent Poisson regression using the original, uncalibrated probabilities. The probability order remains `(H, D, A)`, bookmaker odds are excluded from the predictor set, the random seed is fixed at `42`, and the required expected-goal and result-probability outputs are present.

The configuration therefore provides a valid high-level production contract. It does not yet reproduce the exact validated estimator implementation. Twelve implementation details—including the feature schemas, preprocessing sequence, removed rank-deficient columns, regularisation parameters, solver settings and scoreline truncation rule—remain pending recovery from the final corrected Notebook 5 and Notebook 6 code.

The production models must not be fitted until those implementation details have been recovered and frozen.

## 3. Load the Complete Historical Modelling Dataset

This section identifies and loads the historical modelling dataset used by the validated Independent Poisson and walk-forward implementations. The source file is recovered from the project notebooks and ranked using notebook references, file location, schema evidence and engineered-data indicators rather than by assuming a filename.

The loaded data is then profiled to establish its fixture coverage, date range, season coverage, target completeness and duplicate-fixture status. Particular attention is given to whether completed 2025–26 Premier League results are present, since the production refit for the 2026–27 forecasting period should ultimately use all eligible completed fixtures available before deployment.

No external data is downloaded and no missing season is fabricated in this section. If 2025–26 data is absent or incomplete, that limitation is reported explicitly so that a controlled data-update stage can be completed before the final production refit.

In [2]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

REPOSITORY_NAME = "premier-league-probability-engine"

ENGINEERED_FEATURE_RELATIVE_PATH = Path(
    "data/processed/premier_league_model_data.parquet"
)

SUPPORTED_TARGET_FILE_ENDINGS = (
    ".csv",
    ".csv.gz",
    ".parquet",
    ".feather",
    ".pkl",
    ".pickle",
)

COLUMN_ROLE_ALIASES = {
    "season": [
        "Season",
        "season",
        "season_id",
        "season_name",
    ],
    "match_date": [
        "Date",
        "date",
        "MatchDate",
        "match_date",
        "fixture_date",
        "kickoff",
        "kickoff_date",
        "datetime",
    ],
    "home_team": [
        "HomeTeam",
        "home_team",
        "home",
        "Home",
    ],
    "away_team": [
        "AwayTeam",
        "away_team",
        "away",
        "Away",
    ],
    "home_goals": [
        "FTHG",
        "home_goals",
        "HomeGoals",
        "home_score",
        "full_time_home_goals",
        "home_goals_target",
    ],
    "away_goals": [
        "FTAG",
        "away_goals",
        "AwayGoals",
        "away_score",
        "full_time_away_goals",
        "away_goals_target",
    ],
    "result": [
        "FTR",
        "result",
        "Result",
        "full_time_result",
        "match_result",
        "target",
    ],
}


# ---------------------------------------------------------------------
# Project-root detection
# ---------------------------------------------------------------------

def detect_project_root(
    start_path: Path,
    repository_name: str,
) -> Path:
    """Locate the repository root without assuming the notebook directory."""
    start_path = start_path.expanduser().resolve()
    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        if candidate.name == repository_name:
            return candidate

    for parent in candidates[:5]:
        repository_candidate = parent / repository_name

        if repository_candidate.is_dir():
            return repository_candidate.resolve()

    for candidate in candidates:
        structure_score = sum(
            [
                4 * int((candidate / ".git").exists()),
                2 * int((candidate / "notebooks").is_dir()),
                2 * int((candidate / "data").is_dir()),
                1 * int((candidate / "outputs").is_dir()),
                1 * int((candidate / "README.md").is_file()),
            ]
        )

        if structure_score >= 4:
            return candidate.resolve()

    raise FileNotFoundError(
        "Unable to identify the project root from "
        f"{start_path}."
    )


if "project_root" not in globals():
    project_root = detect_project_root(
        start_path=Path.cwd(),
        repository_name=REPOSITORY_NAME,
    )

project_root = Path(project_root).resolve()


# ---------------------------------------------------------------------
# General helpers
# ---------------------------------------------------------------------

def normalise_column_name(column_name: Any) -> str:
    """Normalise a column name for role matching."""
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(column_name).lower(),
    )


def identify_column_roles(
    columns: pd.Index | list[Any],
) -> dict[str, str]:
    """Map known modelling roles to actual dataframe columns."""
    normalised_columns = {
        normalise_column_name(column): str(column)
        for column in columns
    }

    identified_roles: dict[str, str] = {}

    for role, aliases in COLUMN_ROLE_ALIASES.items():
        for alias in aliases:
            normalised_alias = normalise_column_name(alias)

            if normalised_alias in normalised_columns:
                identified_roles[role] = normalised_columns[
                    normalised_alias
                ]
                break

    return identified_roles


def read_tabular_file(path: Path) -> pd.DataFrame:
    """Load a supported tabular file."""
    lower_name = path.name.lower()

    if lower_name.endswith((".csv", ".csv.gz")):
        try:
            return pd.read_csv(
                path,
                low_memory=False,
                encoding="utf-8",
            )
        except UnicodeDecodeError:
            return pd.read_csv(
                path,
                low_memory=False,
                encoding="latin-1",
            )

    if lower_name.endswith(".parquet"):
        return pd.read_parquet(path)

    if lower_name.endswith(".feather"):
        return pd.read_feather(path)

    if lower_name.endswith((".pkl", ".pickle")):
        loaded_object = pd.read_pickle(path)

        if not isinstance(loaded_object, pd.DataFrame):
            raise TypeError(
                f"{path} contains {type(loaded_object).__name__}, "
                "not a pandas DataFrame."
            )

        return loaded_object

    raise ValueError(f"Unsupported tabular format: {path}")


def canonicalise_season(value: Any) -> str | None:
    """Convert common season labels to YYYY-YY form."""
    if pd.isna(value):
        return None

    value_string = str(value).strip()
    start_year_match = re.search(r"(20\d{2})", value_string)

    if start_year_match is None:
        return value_string

    start_year = int(start_year_match.group(1))

    return f"{start_year}-{str(start_year + 1)[-2:]}"


def parse_mixed_dates(
    values: pd.Series,
    *,
    dayfirst: bool,
) -> pd.Series:
    """Parse mixed date strings across pandas versions."""
    try:
        parsed = pd.to_datetime(
            values,
            errors="coerce",
            dayfirst=dayfirst,
            format="mixed",
        )
    except (TypeError, ValueError):
        parsed = pd.to_datetime(
            values,
            errors="coerce",
            dayfirst=dayfirst,
        )

    return parsed.dt.normalize()


def date_candidates(values: pd.Series) -> dict[str, pd.Series]:
    """Create distinct plausible date interpretations."""
    if pd.api.types.is_datetime64_any_dtype(values):
        return {
            "Existing datetime values": pd.to_datetime(
                values,
                errors="coerce",
            ).dt.normalize()
        }

    candidates = {
        "Day first": parse_mixed_dates(
            values,
            dayfirst=True,
        ),
        "Month first": parse_mixed_dates(
            values,
            dayfirst=False,
        ),
    }

    distinct_candidates: dict[str, pd.Series] = {}

    for candidate_name, candidate_values in candidates.items():
        if not any(
            candidate_values.equals(existing_values)
            for existing_values in distinct_candidates.values()
        ):
            distinct_candidates[candidate_name] = candidate_values

    return distinct_candidates


def build_fixture_keys(
    frame: pd.DataFrame,
    *,
    season_column: str,
    date_values: pd.Series,
    home_team_column: str,
    away_team_column: str,
) -> pd.DataFrame:
    """Construct normalised fixture keys for cross-source validation."""
    return pd.DataFrame(
        {
            "_SeasonKey": frame[season_column].map(
                canonicalise_season
            ),
            "_DateKey": date_values,
            "_HomeTeamKey": (
                frame[home_team_column]
                .astype("string")
                .str.strip()
                .str.casefold()
            ),
            "_AwayTeamKey": (
                frame[away_team_column]
                .astype("string")
                .str.strip()
                .str.casefold()
            ),
        },
        index=frame.index,
    )


def locate_numbered_notebook(
    root: Path,
    notebook_number: int,
    terms: tuple[str, ...],
) -> Path:
    """Identify a numbered source notebook."""
    candidates: list[tuple[int, Path]] = []

    for notebook_path in root.rglob("*.ipynb"):
        if ".ipynb_checkpoints" in notebook_path.parts:
            continue

        stem = notebook_path.stem.lower()
        score = 0

        if re.match(
            rf"^0?{notebook_number}(?:_|-)",
            stem,
        ):
            score += 100

        score += 25 * sum(
            term in stem
            for term in terms
        )

        if score > 0:
            candidates.append(
                (
                    score,
                    notebook_path.resolve(),
                )
            )

    if not candidates:
        raise FileNotFoundError(
            f"Unable to locate Notebook {notebook_number}."
        )

    candidates.sort(
        key=lambda item: (
            item[0],
            str(item[1]),
        ),
        reverse=True,
    )

    return candidates[0][1]


def load_notebook_code(notebook_path: Path) -> str:
    """Return the combined code text from a notebook."""
    try:
        notebook = json.loads(
            notebook_path.read_text(encoding="utf-8")
        )
    except (OSError, json.JSONDecodeError) as error:
        raise ValueError(
            f"Unable to read notebook: {notebook_path}"
        ) from error

    code_parts: list[str] = []

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", "")

        if isinstance(source, list):
            source = "".join(source)

        code_parts.append(str(source))

    return "\n".join(code_parts)


# ---------------------------------------------------------------------
# Locate validated source notebooks
# ---------------------------------------------------------------------

poisson_notebook_path = locate_numbered_notebook(
    root=project_root,
    notebook_number=5,
    terms=("poisson", "scoreline"),
)

walk_forward_notebook_path = locate_numbered_notebook(
    root=project_root,
    notebook_number=6,
    terms=("walk", "forward", "backtest"),
)

source_notebook_paths = [
    poisson_notebook_path,
    walk_forward_notebook_path,
]

source_notebook_code = {
    notebook_path: load_notebook_code(notebook_path).lower()
    for notebook_path in source_notebook_paths
}


# ---------------------------------------------------------------------
# Load the engineered feature table
# ---------------------------------------------------------------------

engineered_feature_data_path = (
    project_root / ENGINEERED_FEATURE_RELATIVE_PATH
).resolve()

if not engineered_feature_data_path.is_file():
    raise FileNotFoundError(
        "The engineered feature table does not exist at:\n"
        f"{engineered_feature_data_path}"
    )

engineered_feature_data = pd.read_parquet(
    engineered_feature_data_path
)

if engineered_feature_data.empty:
    raise ValueError(
        "The engineered feature table contains no rows."
    )

engineered_roles = identify_column_roles(
    engineered_feature_data.columns
)

required_engineered_roles = {
    "season",
    "match_date",
    "home_team",
    "away_team",
}

missing_engineered_roles = sorted(
    required_engineered_roles
    - set(engineered_roles)
)

if missing_engineered_roles:
    display(
        pd.DataFrame(
            {
                "AvailableColumn": (
                    engineered_feature_data.columns
                )
            }
        )
    )

    raise ValueError(
        "The engineered feature table is missing required "
        f"fixture roles: {missing_engineered_roles}"
    )


# ---------------------------------------------------------------------
# Recover the separate goal-target table
# ---------------------------------------------------------------------

target_candidate_records: list[dict[str, Any]] = []

for candidate_path in (project_root / "data").rglob("*"):
    if not candidate_path.is_file():
        continue

    lower_name = candidate_path.name.lower()

    if not any(
        lower_name.endswith(ending)
        for ending in SUPPORTED_TARGET_FILE_ENDINGS
    ):
        continue

    try:
        candidate_data = read_tabular_file(candidate_path)
        candidate_roles = identify_column_roles(
            candidate_data.columns
        )
        read_status = "Readable"
        read_error = None
    except Exception as error:
        candidate_data = pd.DataFrame()
        candidate_roles = {}
        read_status = "Unreadable"
        read_error = str(error)

    required_target_roles = {
        "season",
        "match_date",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
    }

    valid_target_schema = required_target_roles.issubset(
        candidate_roles
    )

    reference_count = sum(
        (
            candidate_path.name.lower()
            in notebook_code
            or candidate_path.stem.lower()
            in notebook_code
        )
        for notebook_code in source_notebook_code.values()
    )

    score = 0

    if "goal" in lower_name:
        score += 60

    if "target" in lower_name:
        score += 60

    if "raw" in {
        path_part.lower()
        for path_part in candidate_path.parts
    }:
        score += 20

    score += 100 * reference_count
    score += 100 * int(valid_target_schema)

    target_candidate_records.append(
        {
            "Path": candidate_path.relative_to(
                project_root
            ).as_posix(),
            "Score": score,
            "Rows": len(candidate_data),
            "Columns": len(candidate_data.columns),
            "ReferenceCount": reference_count,
            "ValidTargetSchema": valid_target_schema,
            "DetectedRoles": ", ".join(
                sorted(candidate_roles)
            ),
            "Status": read_status,
            "Error": read_error,
        }
    )


target_candidate_table = (
    pd.DataFrame(target_candidate_records)
    .sort_values(
        by=[
            "ValidTargetSchema",
            "Score",
            "ReferenceCount",
            "Rows",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

eligible_target_candidates = target_candidate_table.loc[
    (target_candidate_table["Status"] == "Readable")
    & target_candidate_table["ValidTargetSchema"]
    & (target_candidate_table["ReferenceCount"] >= 1)
].copy()

if eligible_target_candidates.empty:
    print("Goal-target candidate search:")
    display(target_candidate_table.head(20))

    raise RuntimeError(
        "No separate goal-target table was both schema-valid and "
        "referenced by the validated Notebook 5/6 code."
    )

highest_target_score = eligible_target_candidates.iloc[0][
    "Score"
]

highest_target_candidates = eligible_target_candidates.loc[
    eligible_target_candidates["Score"]
    == highest_target_score
]

if len(highest_target_candidates) != 1:
    display(highest_target_candidates)

    raise RuntimeError(
        "Multiple goal-target tables share the strongest evidence. "
        "No arbitrary target source has been selected."
    )

selected_target_record = eligible_target_candidates.iloc[0]

goal_target_data_path = (
    project_root / selected_target_record["Path"]
).resolve()

goal_target_data = read_tabular_file(
    goal_target_data_path
)

goal_target_roles = identify_column_roles(
    goal_target_data.columns
)


# ---------------------------------------------------------------------
# Resolve date orientation by maximising cross-source fixture matches
# ---------------------------------------------------------------------

engineered_date_candidates = date_candidates(
    engineered_feature_data[
        engineered_roles["match_date"]
    ]
)

target_date_candidates = date_candidates(
    goal_target_data[
        goal_target_roles["match_date"]
    ]
)

date_alignment_records: list[dict[str, Any]] = []
alignment_objects: list[dict[str, Any]] = []

for engineered_date_name, engineered_dates in (
    engineered_date_candidates.items()
):
    engineered_keys = build_fixture_keys(
        engineered_feature_data,
        season_column=engineered_roles["season"],
        date_values=engineered_dates,
        home_team_column=engineered_roles["home_team"],
        away_team_column=engineered_roles["away_team"],
    )

    for target_date_name, target_dates in (
        target_date_candidates.items()
    ):
        target_keys = build_fixture_keys(
            goal_target_data,
            season_column=goal_target_roles["season"],
            date_values=target_dates,
            home_team_column=goal_target_roles["home_team"],
            away_team_column=goal_target_roles["away_team"],
        )

        matched_keys = engineered_keys.merge(
            target_keys,
            how="inner",
            on=[
                "_SeasonKey",
                "_DateKey",
                "_HomeTeamKey",
                "_AwayTeamKey",
            ],
        )

        matched_fixture_count = len(matched_keys)

        date_alignment_records.append(
            {
                "EngineeredDateInterpretation": (
                    engineered_date_name
                ),
                "TargetDateInterpretation": target_date_name,
                "MatchedFixtures": matched_fixture_count,
            }
        )

        alignment_objects.append(
            {
                "engineered_date_name": (
                    engineered_date_name
                ),
                "target_date_name": target_date_name,
                "engineered_dates": engineered_dates,
                "target_dates": target_dates,
                "engineered_keys": engineered_keys,
                "target_keys": target_keys,
                "matched_fixture_count": matched_fixture_count,
            }
        )


date_alignment_table = (
    pd.DataFrame(date_alignment_records)
    .sort_values(
        by="MatchedFixtures",
        ascending=False,
    )
    .reset_index(drop=True)
)

best_match_count = date_alignment_table.iloc[0][
    "MatchedFixtures"
]

best_alignment_objects = [
    alignment
    for alignment in alignment_objects
    if alignment["matched_fixture_count"]
    == best_match_count
]

if best_match_count == 0:
    display(date_alignment_table)

    raise RuntimeError(
        "The engineered feature table and goal-target table have "
        "no matching fixture keys under any date interpretation."
    )

if len(best_alignment_objects) > 1:
    unique_key_pairs = {
        (
            tuple(
                alignment["engineered_dates"]
                .astype("string")
                .fillna("<NA>")
            ),
            tuple(
                alignment["target_dates"]
                .astype("string")
                .fillna("<NA>")
            ),
        )
        for alignment in best_alignment_objects
    }

    if len(unique_key_pairs) > 1:
        display(date_alignment_table)

        raise RuntimeError(
            "Multiple genuinely different date interpretations "
            "produce the same maximum fixture match count."
        )

selected_alignment = best_alignment_objects[0]

engineered_fixture_keys = selected_alignment[
    "engineered_keys"
].copy()

target_fixture_keys = selected_alignment[
    "target_keys"
].copy()


# ---------------------------------------------------------------------
# Validate one-to-one fixture alignment
# ---------------------------------------------------------------------

fixture_key_columns = [
    "_SeasonKey",
    "_DateKey",
    "_HomeTeamKey",
    "_AwayTeamKey",
]

engineered_duplicate_count = int(
    engineered_fixture_keys.duplicated(
        subset=fixture_key_columns,
        keep=False,
    ).sum()
)

target_duplicate_count = int(
    target_fixture_keys.duplicated(
        subset=fixture_key_columns,
        keep=False,
    ).sum()
)

if engineered_duplicate_count > 0:
    raise ValueError(
        "The engineered feature table contains duplicate fixture "
        f"keys affecting {engineered_duplicate_count} rows."
    )

if target_duplicate_count > 0:
    raise ValueError(
        "The goal-target table contains duplicate fixture keys "
        f"affecting {target_duplicate_count} rows."
    )


engineered_feature_data_with_keys = pd.concat(
    [
        engineered_feature_data.reset_index(drop=True),
        engineered_fixture_keys.reset_index(drop=True),
    ],
    axis=1,
)

goal_target_data_with_keys = pd.concat(
    [
        goal_target_data.reset_index(drop=True),
        target_fixture_keys.reset_index(drop=True),
    ],
    axis=1,
)


alignment_audit = (
    engineered_fixture_keys
    .drop_duplicates()
    .merge(
        target_fixture_keys.drop_duplicates(),
        how="outer",
        on=fixture_key_columns,
        indicator=True,
        validate="one_to_one",
    )
)

alignment_status_counts = (
    alignment_audit["_merge"]
    .value_counts()
    .rename_axis("AlignmentStatus")
    .reset_index(name="FixtureCount")
)

engineered_only_count = int(
    (alignment_audit["_merge"] == "left_only").sum()
)

target_only_count = int(
    (alignment_audit["_merge"] == "right_only").sum()
)

if engineered_only_count > 0 or target_only_count > 0:
    print("Fixture-alignment status:")
    display(alignment_status_counts)

    print("\nSample unmatched fixtures:")
    display(
        alignment_audit.loc[
            alignment_audit["_merge"] != "both"
        ].head(20)
    )

    raise RuntimeError(
        "The engineered feature and goal-target tables do not "
        "have complete one-to-one fixture coverage."
    )


# ---------------------------------------------------------------------
# Merge goal targets onto the engineered feature table
# ---------------------------------------------------------------------

target_columns_to_merge = [
    *fixture_key_columns,
    goal_target_roles["home_goals"],
    goal_target_roles["away_goals"],
]

target_result_column = goal_target_roles.get("result")
engineered_result_column = engineered_roles.get("result")

if target_result_column is not None:
    target_columns_to_merge.append(
        target_result_column
    )

target_merge_data = goal_target_data_with_keys[
    target_columns_to_merge
].copy()

if (
    target_result_column is not None
    and engineered_result_column is not None
    and target_result_column == engineered_result_column
):
    target_merge_data = target_merge_data.rename(
        columns={
            target_result_column: "_TargetSourceResult",
        }
    )

historical_modelling_data = (
    engineered_feature_data_with_keys
    .merge(
        target_merge_data,
        how="left",
        on=fixture_key_columns,
        validate="one_to_one",
        suffixes=("", "_GoalTargetSource"),
    )
)

home_goal_target_column = goal_target_roles["home_goals"]
away_goal_target_column = goal_target_roles["away_goals"]

if (
    home_goal_target_column
    not in historical_modelling_data.columns
):
    home_goal_target_column = (
        f"{home_goal_target_column}_GoalTargetSource"
    )

if (
    away_goal_target_column
    not in historical_modelling_data.columns
):
    away_goal_target_column = (
        f"{away_goal_target_column}_GoalTargetSource"
    )

missing_goal_target_count = int(
    (
        historical_modelling_data[
            home_goal_target_column
        ].isna()
        | historical_modelling_data[
            away_goal_target_column
        ].isna()
    ).sum()
)

if missing_goal_target_count > 0:
    raise RuntimeError(
        f"{missing_goal_target_count} merged fixtures are missing "
        "one or both goal targets."
    )


# Validate result-label agreement when both sources contain the result.
result_disagreement_count = 0

if (
    engineered_result_column is not None
    and "_TargetSourceResult"
    in historical_modelling_data.columns
):
    comparable_result_mask = (
        historical_modelling_data[
            engineered_result_column
        ].notna()
        & historical_modelling_data[
            "_TargetSourceResult"
        ].notna()
    )

    result_disagreement_count = int(
        (
            historical_modelling_data.loc[
                comparable_result_mask,
                engineered_result_column,
            ].astype("string")
            != historical_modelling_data.loc[
                comparable_result_mask,
                "_TargetSourceResult",
            ].astype("string")
        ).sum()
    )

    if result_disagreement_count > 0:
        raise ValueError(
            "The engineered and goal-target sources disagree on "
            f"{result_disagreement_count} result labels."
        )


# Validate goal target values.
for goal_column in [
    home_goal_target_column,
    away_goal_target_column,
]:
    numeric_goals = pd.to_numeric(
        historical_modelling_data[goal_column],
        errors="coerce",
    )

    invalid_goal_mask = (
        numeric_goals.isna()
        | (numeric_goals < 0)
        | (numeric_goals % 1 != 0)
    )

    if invalid_goal_mask.any():
        raise ValueError(
            f"Column {goal_column!r} contains "
            f"{int(invalid_goal_mask.sum())} invalid goal targets."
        )

    historical_modelling_data[goal_column] = (
        numeric_goals.astype(int)
    )


# ---------------------------------------------------------------------
# Build season profile
# ---------------------------------------------------------------------

season_summary = (
    historical_modelling_data
    .groupby("_SeasonKey", dropna=False)
    .agg(
        Fixtures=("_SeasonKey", "size"),
        CompletedHomeGoalTargets=(
            home_goal_target_column,
            "count",
        ),
        CompletedAwayGoalTargets=(
            away_goal_target_column,
            "count",
        ),
        UniqueHomeTeams=("_HomeTeamKey", "nunique"),
        UniqueAwayTeams=("_AwayTeamKey", "nunique"),
    )
    .reset_index()
    .rename(columns={"_SeasonKey": "Season"})
)

season_summary["MissingGoalTargets"] = (
    season_summary["Fixtures"]
    - season_summary[
        [
            "CompletedHomeGoalTargets",
            "CompletedAwayGoalTargets",
        ]
    ].min(axis=1)
)

season_summary["_SeasonStartYear"] = (
    season_summary["Season"]
    .astype("string")
    .str.extract(r"(20\d{2})", expand=False)
    .astype("Int64")
)

season_summary = (
    season_summary
    .sort_values(
        by=[
            "_SeasonStartYear",
            "Season",
        ]
    )
    .drop(columns="_SeasonStartYear")
    .reset_index(drop=True)
)


season_2025_26_rows = historical_modelling_data.loc[
    historical_modelling_data["_SeasonKey"]
    == "2025-26"
]

if season_2025_26_rows.empty:
    season_2025_26_status = (
        "ABSENT: no 2025-26 fixture rows were detected."
    )
elif len(season_2025_26_rows) < 380:
    season_2025_26_status = (
        "PARTIAL COVERAGE: "
        f"{len(season_2025_26_rows)} fixtures were detected."
    )
elif len(season_2025_26_rows) == 380:
    season_2025_26_status = (
        "COMPLETE: 380 completed 2025-26 fixtures were detected."
    )
else:
    season_2025_26_status = (
        "REVIEW REQUIRED: more than 380 2025-26 fixtures "
        "were detected."
    )


engineered_identifier_columns = {
    engineered_roles["season"],
    engineered_roles["match_date"],
    engineered_roles["home_team"],
    engineered_roles["away_team"],
    engineered_result_column,
}

engineered_identifier_columns.discard(None)

candidate_engineered_feature_columns = [
    column
    for column in engineered_feature_data.columns
    if column not in engineered_identifier_columns
]


dataset_profile = pd.DataFrame(
    [
        {
            "Metric": "Engineered feature source",
            "Value": engineered_feature_data_path.relative_to(
                project_root
            ).as_posix(),
        },
        {
            "Metric": "Goal-target source",
            "Value": goal_target_data_path.relative_to(
                project_root
            ).as_posix(),
        },
        {
            "Metric": "Engineered feature rows",
            "Value": len(engineered_feature_data),
        },
        {
            "Metric": "Engineered feature columns",
            "Value": len(engineered_feature_data.columns),
        },
        {
            "Metric": "Candidate engineered columns",
            "Value": len(candidate_engineered_feature_columns),
        },
        {
            "Metric": "Goal-target rows",
            "Value": len(goal_target_data),
        },
        {
            "Metric": "Merged modelling rows",
            "Value": len(historical_modelling_data),
        },
        {
            "Metric": "Matched fixture keys",
            "Value": best_match_count,
        },
        {
            "Metric": "Engineered-only fixtures",
            "Value": engineered_only_count,
        },
        {
            "Metric": "Target-only fixtures",
            "Value": target_only_count,
        },
        {
            "Metric": "Missing merged goal targets",
            "Value": missing_goal_target_count,
        },
        {
            "Metric": "Result-label disagreements",
            "Value": result_disagreement_count,
        },
        {
            "Metric": "Earliest matched fixture date",
            "Value": (
                historical_modelling_data["_DateKey"]
                .min()
                .date()
                .isoformat()
            ),
        },
        {
            "Metric": "Latest matched fixture date",
            "Value": (
                historical_modelling_data["_DateKey"]
                .max()
                .date()
                .isoformat()
            ),
        },
        {
            "Metric": "2025-26 status",
            "Value": season_2025_26_status,
        },
    ]
)


# Preserve production-facing variables while removing temporary merge keys.
production_fixture_keys = historical_modelling_data[
    fixture_key_columns
].copy()

historical_modelling_data = (
    historical_modelling_data
    .drop(
        columns=[
            *fixture_key_columns,
            "_TargetSourceResult",
        ],
        errors="ignore",
    )
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("Validated source notebooks:")
for notebook_path in source_notebook_paths:
    print(
        "- "
        + notebook_path.relative_to(
            project_root
        ).as_posix()
    )

print("\nGoal-target candidate evidence:")
display(target_candidate_table.head(15))

print("\nSelected data sources:")
print(
    "- Engineered features: "
    + engineered_feature_data_path.relative_to(
        project_root
    ).as_posix()
)
print(
    "- Goal targets: "
    + goal_target_data_path.relative_to(
        project_root
    ).as_posix()
)

print("\nDetected engineered-table roles:")
display(
    pd.DataFrame(
        [
            {
                "Role": role,
                "DatasetColumn": column,
            }
            for role, column in engineered_roles.items()
        ]
    )
)

print("\nDetected goal-target-table roles:")
display(
    pd.DataFrame(
        [
            {
                "Role": role,
                "DatasetColumn": column,
            }
            for role, column in goal_target_roles.items()
        ]
    )
)

print("\nDate-alignment validation:")
display(date_alignment_table)

print("\nFixture-alignment status:")
display(alignment_status_counts)

print("\nCombined historical modelling profile:")
display(dataset_profile)

print("\nSeason coverage:")
display(season_summary)

print("\nFirst 40 candidate engineered columns:")
display(
    pd.DataFrame(
        {
            "Column": candidate_engineered_feature_columns[:40],
        }
    )
)

print(
    f"\n2025-26 assessment: {season_2025_26_status}"
)

Validated source notebooks:
- notebooks/05_poisson_scoreline_modelling.ipynb
- notebooks/06_walk_forward_backtesting.ipynb

Goal-target candidate evidence:


,Path,Score,Rows,Columns,ReferenceCount,ValidTargetSchema,DetectedRoles,Status,Error
0,data/raw/premier_league_goal_targets_2015_16_t...,440,3800,7,2,True,"away_goals, away_team, home_goals, home_team, ...",Readable,None
1,data/processed/premier_league_model_data.csv,200,3800,75,2,False,"away_team, home_team, match_date, result, season",Readable,None
2,data/processed/premier_league_model_data.parquet,200,3800,75,2,False,"away_team, home_team, match_date, result, season",Readable,None
3,data/raw/football_data/premier_league_2015-16.csv,20,380,65,0,False,"away_goals, away_team, home_goals, home_team, ...",Readable,None
4,data/raw/football_data/premier_league_2016-17.csv,20,380,65,0,False,"away_goals, away_team, home_goals, home_team, ...",Readable,None
5,data/raw/football_data/premier_league_2017-18.csv,20,380,65,0,False,"away_goals, away_team, home_goals, home_team, ...",Readable,None
6,data/raw/football_data/premier_league_2018-19.csv,20,380,62,0,False,"away_goals, away_team, home_goals, home_team, ...",Readable,None
7,data/raw/football_data/premier_league_2019-20.csv,20,380,106,0,False,"away_goals, away_team, home_goals, home_team, ...",Readable,None
8,data/raw/football_data/premier_league_2020-21.csv,20,380,106,0,False,"away_goals, away_team, home_goals, home_team, ...",Readable,None
9,data/raw/football_data/premier_league_2021-22.csv,20,380,106,0,False,"away_goals, away_team, home_goals, home_team, ...",Readable,None



Selected data sources:
- Engineered features: data/processed/premier_league_model_data.parquet
- Goal targets: data/raw/premier_league_goal_targets_2015_16_to_2024_25.csv

Detected engineered-table roles:


,Role,DatasetColumn
0,season,Season
1,match_date,Date
2,home_team,HomeTeam
3,away_team,AwayTeam
4,result,FTR



Detected goal-target-table roles:


...


Season coverage:


,Season,Fixtures,CompletedHomeGoalTargets,CompletedAwayGoalTargets,UniqueHomeTeams,UniqueAwayTeams,MissingGoalTargets
0,2015-16,380,380,380,20,20,0
1,2016-17,380,380,380,20,20,0
2,2017-18,380,380,380,20,20,0
3,2018-19,380,380,380,20,20,0
4,2019-20,380,380,380,20,20,0
5,2020-21,380,380,380,20,20,0
6,2021-22,380,380,380,20,20,0
7,2022-23,380,380,380,20,20,0
8,2023-24,380,380,380,20,20,0
9,2024-25,380,380,380,20,20,0



First 40 candidate engineered columns:


,Column
0,HomeEloBefore
1,AwayEloBefore
2,HomeRollingPoints5
3,AwayRollingPoints5
4,HomeRollingGoalsFor5
5,HomeRollingGoalsAgainst5
6,HomeRollingGoalDifference5
7,HomeRollingWinRate5
8,AwayRollingGoalsFor5
9,AwayRollingGoalsAgainst5



2025-26 assessment: ABSENT: no 2025-26 fixture rows were detected.


### Results and Interpretation

The validated modelling data is split across two complementary sources. The engineered feature table is `data/processed/premier_league_model_data.parquet`, while the realised home- and away-goal targets are stored separately in `data/raw/premier_league_goal_targets_2015_16_to_2024_25.csv`.

The engineered table contains 3,800 fixtures and 75 columns, of which 70 are candidate engineered predictors after excluding the fixture identifiers and full-time result label. The target table also contains 3,800 fixtures and supplies `FTHG` and `FTAG`, which are not duplicated in the processed feature table.

Fixture matching confirmed that the target-file dates must be interpreted as month-first. This interpretation matched all 3,800 engineered fixtures, compared with only 2,463 matches under a day-first interpretation. The final merge produced complete one-to-one alignment:

- 3,800 matched fixture keys
- zero engineered-only fixtures
- zero target-only fixtures
- zero missing merged goal targets
- zero result-label disagreements

The combined data covers ten complete Premier League seasons from 2015–16 through 2024–25. Every season contains 380 fixtures, 20 unique home teams, 20 unique away teams and no missing goal targets. The matched date range runs from 8 August 2015 to 25 May 2025.

No 2025–26 fixtures are present. Therefore, this dataset reproduces the historical evidence used by the validated modelling notebooks, but it is not yet sufficient for the final 2026–27 production refit. The exact implementation can now be recovered and frozen, but the production models must not be fitted until a controlled update has added and validated the completed 2025–26 season.

## 4. Recover and Freeze the Exact Validated Feature Pipeline

This section inspects the final Independent Poisson implementation contained in the validated modelling and walk-forward notebooks. Notebook 6 is treated as the primary source because it contains the corrected walk-forward implementation that resolved the earlier rank-deficiency and convergence problems, while Notebook 5 provides supporting model-construction evidence.

The recovery process searches the source code for feature-list definitions, preprocessing operations, dropped columns, Poisson estimator settings, team-handling logic and scoreline-probability construction. It records both automatically extracted assignments and the exact source-code cells supporting them.

No model fitting or hyperparameter selection is performed. A production implementation manifest will only be saved after every required setting has been uniquely supported by the validated source code.

In [3]:
from __future__ import annotations

import ast
import hashlib
import json
import re
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

REPOSITORY_NAME = "premier-league-probability-engine"

RECOVERY_CATEGORIES = {
    "feature_schema": [
        r"feature[_\s-]*(?:column|list|name)",
        r"(?:home|away).*feature",
        r"poisson.*feature",
        r"\bfeatures\b",
    ],
    "rank_deficiency": [
        r"rank.deficien",
        r"matrix_rank",
        r"full.rank",
        r"collinear",
        r"reference.*categor",
        r"drop.*column",
    ],
    "missing_values": [
        r"simpleimputer",
        r"\bimputer\b",
        r"fillna",
        r"dropna",
        r"missing",
    ],
    "scaling": [
        r"standardscaler",
        r"minmaxscaler",
        r"robustscaler",
        r"\bscaler\b",
        r"\bscale\b",
    ],
    "encoding": [
        r"onehotencoder",
        r"get_dummies",
        r"\bencoder\b",
        r"\bencoding\b",
        r"drop\s*=\s*['\"]first['\"]",
    ],
    "poisson_estimator": [
        r"poissonregressor",
        r"\balpha\b",
        r"max_iter",
        r"\btol\b",
        r"\bsolver\b",
    ],
    "team_handling": [
        r"team.*mapping",
        r"team_map",
        r"promot",
        r"new.*team",
        r"unknown.*team",
        r"unseen.*team",
    ],
    "scoreline_construction": [
        r"scoreline",
        r"max_goals",
        r"poisson\.pmf",
        r"joint.*probab",
        r"probability_h",
        r"probability_d",
        r"probability_a",
        r"home_win",
        r"away_win",
    ],
    "expected_goals": [
        r"expected.*goals",
        r"home.*lambda",
        r"away.*lambda",
        r"lambda_home",
        r"lambda_away",
        r"\.predict\(",
    ],
}

CANDIDATE_ASSIGNMENT_PATTERN = re.compile(
    r"feature|column|alpha|solver|max_iter|tol|"
    r"goal|scoreline|trunc|imput|scal|encod|"
    r"team|promot|mapping|drop|rank",
    flags=re.IGNORECASE,
)

RELEVANT_CALL_TERMS = {
    "poissonregressor",
    "pipeline",
    "make_pipeline",
    "columntransformer",
    "make_column_transformer",
    "simpleimputer",
    "standardscaler",
    "minmaxscaler",
    "robustscaler",
    "onehotencoder",
    "get_dummies",
    "fillna",
    "dropna",
    "drop",
    "matrix_rank",
    "pmf",
    "predict",
}


# ---------------------------------------------------------------------
# Project and notebook discovery
# ---------------------------------------------------------------------

def detect_project_root(
    start_path: Path,
    repository_name: str,
) -> Path:
    """Locate the repository root from the notebook working directory."""
    start_path = start_path.expanduser().resolve()
    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        if candidate.name == repository_name:
            return candidate

    for parent in candidates[:5]:
        repository_candidate = parent / repository_name

        if repository_candidate.is_dir():
            return repository_candidate.resolve()

    for candidate in candidates:
        structure_score = sum(
            [
                4 * int((candidate / ".git").exists()),
                2 * int((candidate / "notebooks").is_dir()),
                2 * int((candidate / "data").is_dir()),
                1 * int((candidate / "outputs").is_dir()),
            ]
        )

        if structure_score >= 4:
            return candidate.resolve()

    raise FileNotFoundError(
        "Unable to identify the project root from "
        f"{start_path}."
    )


def locate_numbered_notebook(
    root: Path,
    notebook_number: int,
    required_terms: tuple[str, ...],
) -> Path:
    """Identify a numbered notebook using filename evidence."""
    candidates: list[tuple[int, Path]] = []

    for notebook_path in root.rglob("*.ipynb"):
        if ".ipynb_checkpoints" in notebook_path.parts:
            continue

        stem = notebook_path.stem.lower()
        score = 0

        if re.match(
            rf"^0?{notebook_number}(?:_|-)",
            stem,
        ):
            score += 100

        score += 25 * sum(
            term in stem
            for term in required_terms
        )

        if score > 0:
            candidates.append(
                (
                    score,
                    notebook_path.resolve(),
                )
            )

    if not candidates:
        raise FileNotFoundError(
            f"Unable to locate Notebook {notebook_number}."
        )

    candidates.sort(
        key=lambda item: (
            item[0],
            str(item[1]),
        ),
        reverse=True,
    )

    return candidates[0][1]


if "project_root" not in globals():
    project_root = detect_project_root(
        start_path=Path.cwd(),
        repository_name=REPOSITORY_NAME,
    )

project_root = Path(project_root).resolve()

poisson_notebook_path = locate_numbered_notebook(
    root=project_root,
    notebook_number=5,
    required_terms=("poisson", "scoreline"),
)

walk_forward_notebook_path = locate_numbered_notebook(
    root=project_root,
    notebook_number=6,
    required_terms=("walk", "forward", "backtest"),
)

implementation_notebook_paths = {
    "Notebook 5": poisson_notebook_path,
    "Notebook 6": walk_forward_notebook_path,
}


# ---------------------------------------------------------------------
# Load source code with stable cell identifiers
# ---------------------------------------------------------------------

def load_notebook_code_cells(
    notebook_path: Path,
) -> list[dict[str, Any]]:
    """Load source code and retain the original notebook cell index."""
    try:
        notebook_object = json.loads(
            notebook_path.read_text(encoding="utf-8")
        )
    except (OSError, json.JSONDecodeError) as error:
        raise ValueError(
            f"Unable to read notebook safely: {notebook_path}"
        ) from error

    code_cells: list[dict[str, Any]] = []

    for notebook_cell_index, cell in enumerate(
        notebook_object.get("cells", [])
    ):
        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", "")

        if isinstance(source, list):
            source = "".join(source)

        source = str(source)

        if not source.strip():
            continue

        code_cells.append(
            {
                "NotebookCellIndex": notebook_cell_index,
                "Source": source,
            }
        )

    return code_cells


implementation_source_cells: list[dict[str, Any]] = []

for notebook_label, notebook_path in (
    implementation_notebook_paths.items()
):
    for cell_record in load_notebook_code_cells(
        notebook_path
    ):
        implementation_source_cells.append(
            {
                "Notebook": notebook_label,
                "NotebookPath": notebook_path,
                **cell_record,
            }
        )


# ---------------------------------------------------------------------
# Source provenance
# ---------------------------------------------------------------------

source_notebook_provenance = pd.DataFrame(
    [
        {
            "Notebook": notebook_label,
            "Path": notebook_path.relative_to(
                project_root
            ).as_posix(),
            "SHA256": hashlib.sha256(
                notebook_path.read_bytes()
            ).hexdigest(),
            "CodeCellCount": sum(
                record["Notebook"] == notebook_label
                for record in implementation_source_cells
            ),
            "PrimaryRecoverySource": (
                notebook_label == "Notebook 6"
            ),
        }
        for notebook_label, notebook_path
        in implementation_notebook_paths.items()
    ]
)


# ---------------------------------------------------------------------
# Categorise implementation evidence cells
# ---------------------------------------------------------------------

compiled_category_patterns = {
    category: [
        re.compile(pattern, flags=re.IGNORECASE)
        for pattern in patterns
    ]
    for category, patterns in RECOVERY_CATEGORIES.items()
}

implementation_evidence_records: list[dict[str, Any]] = []

for cell_record in implementation_source_cells:
    source = cell_record["Source"]

    matched_categories = [
        category
        for category, patterns
        in compiled_category_patterns.items()
        if any(
            pattern.search(source)
            for pattern in patterns
        )
    ]

    if not matched_categories:
        continue

    implementation_evidence_records.append(
        {
            "Notebook": cell_record["Notebook"],
            "NotebookCellIndex": (
                cell_record["NotebookCellIndex"]
            ),
            "MatchedCategories": ", ".join(
                matched_categories
            ),
            "CategoryCount": len(matched_categories),
            "LineCount": len(source.splitlines()),
            "Source": source,
        }
    )


implementation_evidence_cells = pd.DataFrame(
    implementation_evidence_records
)

if implementation_evidence_cells.empty:
    raise RuntimeError(
        "No implementation evidence cells were detected in "
        "Notebooks 5 and 6."
    )


# ---------------------------------------------------------------------
# AST extraction helpers
# ---------------------------------------------------------------------

def qualified_name(node: ast.AST) -> str:
    """Return a readable name for a function or attribute expression."""
    if isinstance(node, ast.Name):
        return node.id

    if isinstance(node, ast.Attribute):
        parent_name = qualified_name(node.value)

        if parent_name:
            return f"{parent_name}.{node.attr}"

        return node.attr

    return ""


def assignment_target_names(
    target: ast.AST,
) -> list[str]:
    """Extract simple variable names from an assignment target."""
    if isinstance(target, ast.Name):
        return [target.id]

    if isinstance(target, (ast.Tuple, ast.List)):
        names: list[str] = []

        for element in target.elts:
            names.extend(
                assignment_target_names(element)
            )

        return names

    return []


def safe_literal_value(
    node: ast.AST,
) -> tuple[bool, Any]:
    """Attempt to evaluate a literal without executing notebook code."""
    try:
        return True, ast.literal_eval(node)
    except (ValueError, TypeError, SyntaxError):
        return False, None


def compact_expression(
    node: ast.AST,
    maximum_length: int = 500,
) -> str:
    """Render an AST expression compactly for evidence review."""
    try:
        expression = ast.unparse(node)
    except Exception:
        expression = "<unable to render expression>"

    expression = re.sub(
        r"\s+",
        " ",
        expression,
    ).strip()

    if len(expression) > maximum_length:
        expression = (
            expression[:maximum_length]
            + " ..."
        )

    return expression


assignment_records: list[dict[str, Any]] = []
call_records: list[dict[str, Any]] = []
function_records: list[dict[str, Any]] = []
syntax_error_records: list[dict[str, Any]] = []


for cell_record in implementation_source_cells:
    source = cell_record["Source"]

    try:
        parsed_tree = ast.parse(source)
    except SyntaxError as error:
        syntax_error_records.append(
            {
                "Notebook": cell_record["Notebook"],
                "NotebookCellIndex": (
                    cell_record["NotebookCellIndex"]
                ),
                "Reason": str(error),
            }
        )
        continue

    for node in ast.walk(parsed_tree):
        if isinstance(node, ast.Assign):
            target_names: list[str] = []

            for target in node.targets:
                target_names.extend(
                    assignment_target_names(target)
                )

            literal_success, literal_value = (
                safe_literal_value(node.value)
            )

            for target_name in target_names:
                if not CANDIDATE_ASSIGNMENT_PATTERN.search(
                    target_name
                ):
                    continue

                assignment_records.append(
                    {
                        "Notebook": cell_record["Notebook"],
                        "NotebookCellIndex": (
                            cell_record["NotebookCellIndex"]
                        ),
                        "Variable": target_name,
                        "Expression": compact_expression(
                            node.value
                        ),
                        "LiteralRecovered": literal_success,
                        "LiteralValue": (
                            literal_value
                            if literal_success
                            else None
                        ),
                        "LineNumber": getattr(
                            node,
                            "lineno",
                            None,
                        ),
                    }
                )

        elif isinstance(node, ast.AnnAssign):
            target_names = assignment_target_names(
                node.target
            )

            if node.value is None:
                continue

            literal_success, literal_value = (
                safe_literal_value(node.value)
            )

            for target_name in target_names:
                if not CANDIDATE_ASSIGNMENT_PATTERN.search(
                    target_name
                ):
                    continue

                assignment_records.append(
                    {
                        "Notebook": cell_record["Notebook"],
                        "NotebookCellIndex": (
                            cell_record["NotebookCellIndex"]
                        ),
                        "Variable": target_name,
                        "Expression": compact_expression(
                            node.value
                        ),
                        "LiteralRecovered": literal_success,
                        "LiteralValue": (
                            literal_value
                            if literal_success
                            else None
                        ),
                        "LineNumber": getattr(
                            node,
                            "lineno",
                            None,
                        ),
                    }
                )

        elif isinstance(node, ast.Call):
            call_name = qualified_name(node.func)
            terminal_call_name = (
                call_name.split(".")[-1].lower()
            )

            if terminal_call_name not in RELEVANT_CALL_TERMS:
                continue

            keyword_arguments = {
                keyword.arg or "**kwargs": compact_expression(
                    keyword.value
                )
                for keyword in node.keywords
            }

            positional_arguments = [
                compact_expression(argument)
                for argument in node.args
            ]

            call_records.append(
                {
                    "Notebook": cell_record["Notebook"],
                    "NotebookCellIndex": (
                        cell_record["NotebookCellIndex"]
                    ),
                    "Call": call_name,
                    "PositionalArguments": (
                        positional_arguments
                    ),
                    "KeywordArguments": (
                        keyword_arguments
                    ),
                    "LineNumber": getattr(
                        node,
                        "lineno",
                        None,
                    ),
                }
            )

        elif isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        ):
            function_name = node.name

            if CANDIDATE_ASSIGNMENT_PATTERN.search(
                function_name
            ) or any(
                term in function_name.lower()
                for term in [
                    "poisson",
                    "score",
                    "probab",
                    "predict",
                    "fixture",
                ]
            ):
                function_records.append(
                    {
                        "Notebook": cell_record["Notebook"],
                        "NotebookCellIndex": (
                            cell_record["NotebookCellIndex"]
                        ),
                        "Function": function_name,
                        "Arguments": [
                            argument.arg
                            for argument in node.args.args
                        ],
                        "LineNumber": getattr(
                            node,
                            "lineno",
                            None,
                        ),
                    }
                )


implementation_assignment_candidates = (
    pd.DataFrame(assignment_records)
)

implementation_call_candidates = pd.DataFrame(
    call_records
)

implementation_function_candidates = pd.DataFrame(
    function_records
)


# ---------------------------------------------------------------------
# Build focused candidate tables
# ---------------------------------------------------------------------

if implementation_assignment_candidates.empty:
    implementation_feature_list_candidates = pd.DataFrame()
    implementation_parameter_candidates = pd.DataFrame()
else:
    def is_string_sequence(value: Any) -> bool:
        return (
            isinstance(value, (list, tuple))
            and len(value) >= 2
            and all(
                isinstance(item, str)
                for item in value
            )
        )

    feature_list_mask = (
        implementation_assignment_candidates[
            "LiteralRecovered"
        ]
        & implementation_assignment_candidates[
            "LiteralValue"
        ].map(is_string_sequence)
        & implementation_assignment_candidates[
            "Variable"
        ].str.contains(
            r"feature|column",
            case=False,
            regex=True,
        )
    )

    implementation_feature_list_candidates = (
        implementation_assignment_candidates.loc[
            feature_list_mask,
            [
                "Notebook",
                "NotebookCellIndex",
                "Variable",
                "LiteralValue",
                "LineNumber",
            ],
        ]
        .sort_values(
            by=[
                "Notebook",
                "NotebookCellIndex",
                "Variable",
            ]
        )
        .reset_index(drop=True)
    )

    parameter_mask = (
        implementation_assignment_candidates[
            "LiteralRecovered"
        ]
        & implementation_assignment_candidates[
            "Variable"
        ].str.contains(
            r"alpha|max_iter|tol|solver|max_goals|"
            r"scoreline|trunc",
            case=False,
            regex=True,
        )
    )

    implementation_parameter_candidates = (
        implementation_assignment_candidates.loc[
            parameter_mask,
            [
                "Notebook",
                "NotebookCellIndex",
                "Variable",
                "LiteralValue",
                "Expression",
                "LineNumber",
            ],
        ]
        .sort_values(
            by=[
                "Notebook",
                "NotebookCellIndex",
                "Variable",
            ]
        )
        .reset_index(drop=True)
    )


if implementation_call_candidates.empty:
    poisson_estimator_calls = pd.DataFrame()
    preprocessing_calls = pd.DataFrame()
else:
    poisson_estimator_calls = (
        implementation_call_candidates.loc[
            implementation_call_candidates[
                "Call"
            ].str.contains(
                "PoissonRegressor",
                case=False,
                regex=False,
            )
        ]
        .sort_values(
            by=[
                "Notebook",
                "NotebookCellIndex",
                "LineNumber",
            ]
        )
        .reset_index(drop=True)
    )

    preprocessing_calls = (
        implementation_call_candidates.loc[
            implementation_call_candidates[
                "Call"
            ].str.contains(
                r"Pipeline|ColumnTransformer|Imputer|"
                r"Scaler|Encoder|get_dummies|"
                r"fillna|dropna|\.drop$",
                case=False,
                regex=True,
            )
        ]
        .sort_values(
            by=[
                "Notebook",
                "NotebookCellIndex",
                "LineNumber",
            ]
        )
        .reset_index(drop=True)
    )


# ---------------------------------------------------------------------
# Summarise whether evidence exists for every required detail
# ---------------------------------------------------------------------

def category_evidence_exists(category: str) -> bool:
    return implementation_evidence_cells[
        "MatchedCategories"
    ].str.contains(
        category,
        regex=False,
    ).any()


def variable_candidate_exists(pattern: str) -> bool:
    if implementation_assignment_candidates.empty:
        return False

    return implementation_assignment_candidates[
        "Variable"
    ].str.contains(
        pattern,
        case=False,
        regex=True,
    ).any()


def call_candidate_exists(pattern: str) -> bool:
    if implementation_call_candidates.empty:
        return False

    return implementation_call_candidates[
        "Call"
    ].str.contains(
        pattern,
        case=False,
        regex=True,
    ).any()


implementation_recovery_status = pd.DataFrame(
    [
        {
            "Requirement": "Home-goal feature columns",
            "EvidenceLocated": variable_candidate_exists(
                r"home.*feature|feature.*home"
            ),
            "FreezeStatus": "Pending exact verification",
        },
        {
            "Requirement": "Away-goal feature columns",
            "EvidenceLocated": variable_candidate_exists(
                r"away.*feature|feature.*away"
            ),
            "FreezeStatus": "Pending exact verification",
        },
        {
            "Requirement": "Exact feature ordering",
            "EvidenceLocated": (
                not implementation_feature_list_candidates.empty
            ),
            "FreezeStatus": "Pending exact verification",
        },
        {
            "Requirement": "Dropped rank-deficient columns",
            "EvidenceLocated": category_evidence_exists(
                "rank_deficiency"
            ),
            "FreezeStatus": "Pending exact verification",
        },
        {
            "Requirement": "Missing-value treatment",
            "EvidenceLocated": category_evidence_exists(
                "missing_values"
            ),
            "FreezeStatus": "Pending exact verification",
        },
        {
            "Requirement": "Scaling",
            "EvidenceLocated": (
                category_evidence_exists("scaling")
                or call_candidate_exists("Scaler")
            ),
            "FreezeStatus": "Pending exact verification",
        },
        {
            "Requirement": "Encoding",
            "EvidenceLocated": (
                category_evidence_exists("encoding")
                or call_candidate_exists(
                    r"Encoder|get_dummies"
                )
            ),
            "FreezeStatus": "Pending exact verification",
        },
        {
            "Requirement": "Team mappings",
            "EvidenceLocated": category_evidence_exists(
                "team_handling"
            ),
            "FreezeStatus": "Pending exact verification",
        },
        {
            "Requirement": "Promoted-team handling",
            "EvidenceLocated": category_evidence_exists(
                "team_handling"
            ),
            "FreezeStatus": "Pending exact verification",
        },
        {
            "Requirement": "Home Poisson alpha",
            "EvidenceLocated": variable_candidate_exists(
                r"home.*alpha|alpha.*home"
            ),
            "FreezeStatus": "Pending exact verification",
        },
        {
            "Requirement": "Away Poisson alpha",
            "EvidenceLocated": variable_candidate_exists(
                r"away.*alpha|alpha.*away"
            ),
            "FreezeStatus": "Pending exact verification",
        },
        {
            "Requirement": "Solver, max_iter and tolerance",
            "EvidenceLocated": (
                not poisson_estimator_calls.empty
                or variable_candidate_exists(
                    r"solver|max_iter|tol"
                )
            ),
            "FreezeStatus": "Pending exact verification",
        },
        {
            "Requirement": "Scoreline truncation rule",
            "EvidenceLocated": (
                category_evidence_exists(
                    "scoreline_construction"
                )
                and variable_candidate_exists(
                    r"max_goals|scoreline|trunc"
                )
            ),
            "FreezeStatus": "Pending exact verification",
        },
    ]
)


# ---------------------------------------------------------------------
# Select concise high-priority source evidence
# ---------------------------------------------------------------------

priority_evidence_cells = (
    implementation_evidence_cells
    .assign(
        SourcePriority=lambda frame: (
            frame["Notebook"] == "Notebook 6"
        ).astype(int)
    )
    .sort_values(
        by=[
            "SourcePriority",
            "CategoryCount",
            "NotebookCellIndex",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .drop(columns="SourcePriority")
    .head(12)
    .reset_index(drop=True)
)


def focused_source_snippet(
    source: str,
    maximum_lines: int = 80,
    context_lines: int = 4,
) -> str:
    """
    Retain only line windows surrounding implementation keywords,
    while preserving original line numbers.
    """
    source_lines = source.splitlines()

    matching_line_indices: set[int] = set()

    combined_patterns = [
        pattern
        for patterns in compiled_category_patterns.values()
        for pattern in patterns
    ]

    for line_index, line in enumerate(source_lines):
        if any(
            pattern.search(line)
            for pattern in combined_patterns
        ):
            window_start = max(
                0,
                line_index - context_lines,
            )
            window_end = min(
                len(source_lines),
                line_index + context_lines + 1,
            )

            matching_line_indices.update(
                range(window_start, window_end)
            )

    selected_line_indices = sorted(
        matching_line_indices
    )

    if not selected_line_indices:
        selected_line_indices = list(
            range(
                min(
                    len(source_lines),
                    maximum_lines,
                )
            )
        )

    selected_line_indices = selected_line_indices[
        :maximum_lines
    ]

    rendered_lines: list[str] = []
    previous_line_index: int | None = None

    for line_index in selected_line_indices:
        if (
            previous_line_index is not None
            and line_index > previous_line_index + 1
        ):
            rendered_lines.append("      ...")

        rendered_lines.append(
            f"{line_index + 1:>5}: "
            f"{source_lines[line_index]}"
        )

        previous_line_index = line_index

    return "\n".join(rendered_lines)


# Preserve the full evidence objects in the notebook kernel.
implementation_recovery_evidence = {
    "source_notebook_provenance": (
        source_notebook_provenance.copy()
    ),
    "evidence_cells": (
        implementation_evidence_cells.copy()
    ),
    "assignment_candidates": (
        implementation_assignment_candidates.copy()
    ),
    "feature_list_candidates": (
        implementation_feature_list_candidates.copy()
    ),
    "parameter_candidates": (
        implementation_parameter_candidates.copy()
    ),
    "call_candidates": (
        implementation_call_candidates.copy()
    ),
    "poisson_estimator_calls": (
        poisson_estimator_calls.copy()
    ),
    "preprocessing_calls": (
        preprocessing_calls.copy()
    ),
    "function_candidates": (
        implementation_function_candidates.copy()
    ),
    "recovery_status": (
        implementation_recovery_status.copy()
    ),
}

implementation_manifest_frozen = False


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("Validated implementation source notebooks:")
display(source_notebook_provenance)

print("\nImplementation-recovery status:")
display(implementation_recovery_status)

print("\nLiteral feature-list candidates:")
if implementation_feature_list_candidates.empty:
    print("No literal feature-list assignments were recovered automatically.")
else:
    display(implementation_feature_list_candidates)

print("\nLiteral estimator and scoreline parameter candidates:")
if implementation_parameter_candidates.empty:
    print("No literal parameter assignments were recovered automatically.")
else:
    display(implementation_parameter_candidates)

print("\nPoissonRegressor construction calls:")
if poisson_estimator_calls.empty:
    print("No direct PoissonRegressor constructor calls were recovered.")
else:
    display(poisson_estimator_calls)

print("\nPreprocessing and column-removal calls:")
if preprocessing_calls.empty:
    print("No preprocessing calls were recovered automatically.")
else:
    display(preprocessing_calls.head(40))

print("\nRelevant implementation functions:")
if implementation_function_candidates.empty:
    print("No relevant function definitions were recovered automatically.")
else:
    display(
        implementation_function_candidates.sort_values(
            by=[
                "Notebook",
                "NotebookCellIndex",
                "LineNumber",
            ]
        )
    )

print("\nHighest-priority source evidence cells:")
display(
    priority_evidence_cells[
        [
            "Notebook",
            "NotebookCellIndex",
            "MatchedCategories",
            "CategoryCount",
            "LineCount",
        ]
    ]
)

for _, evidence_row in priority_evidence_cells.iterrows():
    print(
        "\n"
        + "=" * 88
        + "\n"
        + f"{evidence_row['Notebook']} — "
        + f"notebook cell "
        + f"{evidence_row['NotebookCellIndex']}\n"
        + f"Categories: "
        + f"{evidence_row['MatchedCategories']}\n"
        + "-" * 88
    )

    print(
        focused_source_snippet(
            evidence_row["Source"]
        )
    )

print(
    "\nImplementation manifest status: NOT FROZEN. "
    "The displayed candidates must first be reconciled with the "
    "final corrected Notebook 6 source."
)

Validated implementation source notebooks:


,Notebook,Path,SHA256,CodeCellCount,PrimaryRecoverySource
0,Notebook 5,notebooks/05_poisson_scoreline_modelling.ipynb,2aaf4baba9df17f0c0c3594c4c3f1a03083068b629f052...,12,False
1,Notebook 6,notebooks/06_walk_forward_backtesting.ipynb,ba133567a6bfc7277e611e2c5ad87964f943458eb99a31...,9,True



Implementation-recovery status:


,Requirement,EvidenceLocated,FreezeStatus
0,Home-goal feature columns,False,Pending exact verification
1,Away-goal feature columns,False,Pending exact verification
2,Exact feature ordering,True,Pending exact verification
3,Dropped rank-deficient columns,True,Pending exact verification
4,Missing-value treatment,True,Pending exact verification
5,Scaling,True,Pending exact verification
6,Encoding,True,Pending exact verification
7,Team mappings,False,Pending exact verification
8,Promoted-team handling,False,Pending exact verification
9,Home Poisson alpha,True,Pending exact verification



Literal feature-list candidates:


...


Relevant implementation functions:


,Notebook,NotebookCellIndex,Function,Arguments,LineNumber
0,Notebook 5,2,find_first_existing_column,"[dataframe, candidates, label, required]",114
1,Notebook 5,2,validate_goal_target_frame,[dataframe],213
2,Notebook 5,8,create_goal_distribution,"[goals, side]",8
3,Notebook 5,11,poisson_goal_probabilities,"[expected_goals, max_goals, fold_tail]",82
4,Notebook 5,11,create_scoreline_probability_matrix,"[home_expected_goals, away_expected_goals, max...",112
5,Notebook 5,11,expected_goals_to_outcome_probabilities,"[home_expected_goals, away_expected_goals, ind...",139
6,Notebook 5,11,validate_probability_frame,"[probability_frame, expected_index, name]",211
7,Notebook 5,11,multiclass_brier_score,"[y_true, probability_frame, labels]",278
8,Notebook 5,11,evaluate_outcome_probabilities,"[model_name, split_name, y_true, probability_f...",309
9,Notebook 5,11,evaluate_goal_predictions,"[model_name, split_name, side, actual_goals, e...",337



Highest-priority source evidence cells:


,Notebook,NotebookCellIndex,MatchedCategories,CategoryCount,LineCount
0,Notebook 6,14,"feature_schema, missing_values, scaling, score...",5,847
1,Notebook 6,22,"missing_values, scoreline_construction, expect...",3,320
2,Notebook 6,9,"missing_values, scaling, poisson_estimator",3,260
3,Notebook 6,19,"missing_values, scoreline_construction",2,707
4,Notebook 6,11,"scoreline_construction, expected_goals",2,300
5,Notebook 6,4,"feature_schema, missing_values",2,344
6,Notebook 6,2,"rank_deficiency, missing_values",2,531
7,Notebook 6,17,missing_values,1,369
8,Notebook 6,7,missing_values,1,288
9,Notebook 5,11,"feature_schema, missing_values, scaling, score...",5,398



Notebook 6 — notebook cell 14
Categories: feature_schema, missing_values, scaling, scoreline_construction, expected_goals
----------------------------------------------------------------------------------------
   72: 
   73:     X_train = (
   74:         walk_forward_data.loc[
   75:             training_mask,
   76:             feature_columns,
   77:         ]
   78:         .copy()
   79:     )
   80: 
   81:     X_evaluation = (
   82:         walk_forward_data.loc[
   83:             evaluation_mask,
   84:             feature_columns,
   85:         ]
   86:         .copy()
   87:     )
   88: 
      ...
  193:     # --------------------------------------------------------
  194:     # Fit fold-specific median imputation
  195:     # --------------------------------------------------------
  196: 
  197:     fold_imputer = SimpleImputer(
  198:         strategy="median"
  199:     )
  200: 
  201:     X_train_imputed_array = (
      ...
  212: 
  213:     X_train_imputed = pd.

### Results and Interpretation

The initial implementation-recovery pass located the principal components of the validated Independent Poisson pipeline, but the automated pattern matching was not sufficiently precise to freeze the implementation directly.

Notebook 6 is confirmed as the primary production source. Its walk-forward implementation dynamically constructs a single ordered `feature_columns` list and uses that same predictor matrix for both the home-goal and away-goal Poisson regressions. The absence of separate literal home and away feature lists is therefore expected rather than evidence of an unresolved specification.

The preprocessing evidence shows fold-specific median imputation followed by standardisation. No dedicated categorical encoding stage or model-stage team mapping was identified. The preliminary `rank-deficiency` and `encoding` flags were generated by broad source-pattern matches and must not be interpreted as evidence that features were explicitly dropped or encoded.

A particularly important result is that the current Notebook 6 source contains two frozen `PoissonRegressor` specifications with `alpha=0.001` and an explicit `lbfgs` solver. This differs from the earlier Notebook 5 home/away pair and therefore confirms that the final walk-forward implementation, rather than the earlier tuning result, must determine the production parameters.

The implementation manifest remains unfrozen until the exact feature ordering, complete Poisson keyword arguments, preprocessing settings and scoreline constants are extracted directly from the current Notebook 6 source and validated against the engineered dataset.

In [ ]:
# ============================================================
# 4. Freeze the Validated Production Implementation
# ============================================================

from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.linear_model import PoissonRegressor


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine"

engineered_data_path = (
    project_root
    / "data"
    / "processed"
    / "premier_league_model_data.parquet"
)

notebook_6_path = (
    project_root
    / "notebooks"
    / "06_walk_forward_backtesting.ipynb"
)

production_output_directory = (
    project_root
    / "outputs"
    / "production_model"
)

production_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

assert engineered_data_path.is_file()
assert notebook_6_path.is_file()


# ------------------------------------------------------------
# Load the exact engineered source table
# ------------------------------------------------------------

engineered_data = pd.read_parquet(
    engineered_data_path
)

assert not engineered_data.empty


# ------------------------------------------------------------
# Recover fixture identifiers
# ------------------------------------------------------------

def first_existing_column(
    frame,
    candidates,
    label,
):
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate

    raise ValueError(
        f"Could not identify {label}. "
        f"Tried {candidates}."
    )


season_column = first_existing_column(
    engineered_data,
    ["Season", "season"],
    "season column",
)

date_column = first_existing_column(
    engineered_data,
    ["Date", "date", "MatchDate", "match_date"],
    "date column",
)

home_team_column = first_existing_column(
    engineered_data,
    ["HomeTeam", "home_team", "Home"],
    "home-team column",
)

away_team_column = first_existing_column(
    engineered_data,
    ["AwayTeam", "away_team", "Away"],
    "away-team column",
)

target_column = first_existing_column(
    engineered_data,
    ["FTR", "Result", "result", "FullTimeResult"],
    "result column",
)


fixture_key_columns = [
    season_column,
    date_column,
    home_team_column,
    away_team_column,
]


# ------------------------------------------------------------
# Freeze the validated feature rule
# ------------------------------------------------------------

blocked_target_columns = {
    target_column,
    "HomeGoalsTarget",
    "AwayGoalsTarget",
    "FTHG",
    "FTAG",
    "FullTimeHomeGoals",
    "FullTimeAwayGoals",
    "HomeGoals",
    "AwayGoals",
    "Result",
    "FullTimeResult",
}

excluded_predictor_columns = (
    set(fixture_key_columns)
    | blocked_target_columns
)

production_feature_columns = [
    column
    for column in engineered_data.columns
    if (
        column not in excluded_predictor_columns
        and pd.api.types.is_numeric_dtype(
            engineered_data[column]
        )
    )
]

assert production_feature_columns, (
    "No numeric modelling predictors were identified."
)

blocked_predictors = sorted(
    set(production_feature_columns)
    .intersection(blocked_target_columns)
)

assert not blocked_predictors, (
    "Target leakage entered the feature schema: "
    f"{blocked_predictors}"
)

# Section 3 established a 75-column engineered table:
# 5 identifier/target columns + 70 numeric predictors.
assert len(production_feature_columns) == 70, (
    "The recovered feature count differs from the validated "
    f"70-column schema: {len(production_feature_columns)}"
)

production_home_feature_columns = (
    production_feature_columns.copy()
)

production_away_feature_columns = (
    production_feature_columns.copy()
)


# ------------------------------------------------------------
# Freeze exact validated Poisson settings
#
# These values come directly from the final corrected
# Notebook 6 implementation — NO RETUNING.
# ------------------------------------------------------------

home_poisson_explicit_parameters = {
    "alpha": 0.001,
    "max_iter": 5000,
    "tol": 1e-8,
}

away_poisson_explicit_parameters = {
    "alpha": 0.001,
    "max_iter": 5000,
    "tol": 1e-8,
}

home_template = PoissonRegressor(
    **home_poisson_explicit_parameters
)

away_template = PoissonRegressor(
    **away_poisson_explicit_parameters
)

home_poisson_effective_parameters = (
    home_template.get_params(
        deep=False
    )
)

away_poisson_effective_parameters = (
    away_template.get_params(
        deep=False
    )
)


# ------------------------------------------------------------
# Freeze preprocessing
# ------------------------------------------------------------

production_preprocessing_specification = {
    "MissingValueTreatment": {
        "Class": "SimpleImputer",
        "strategy": "median",
        "FitPolicy": "Fit on training data only",
    },
    "Scaling": {
        "Class": "StandardScaler",
        "Parameters": {},
        "FitPolicy": (
            "Fit on imputed training data only"
        ),
    },
    "Encoding": None,
    "ModelStageTeamMapping": None,
    "PromotedTeamHandling": (
        "Handled upstream by the pre-match feature-engineering "
        "pipeline; no raw team-name encoding occurs inside the "
        "Independent Poisson estimator."
    ),
}


# ------------------------------------------------------------
# Freeze scoreline / probability construction
# ------------------------------------------------------------

CLASS_ORDER = ["H", "D", "A"]
MAX_MODELLED_GOALS = 10
MINIMUM_EXPECTED_GOALS = 1e-6
RANDOM_STATE = 42

production_scoreline_specification = {
    "MaximumModelledGoals": MAX_MODELLED_GOALS,
    "GoalGrid": "0 to 10 inclusive",
    "MinimumExpectedGoals": MINIMUM_EXPECTED_GOALS,
    "DependenceAssumption": (
        "Conditional independence of home and away goals"
    ),
    "JointDistribution": (
        "Outer product of home and away Poisson PMFs"
    ),
    "TruncatedMassHandling": (
        "Renormalise scoreline matrix by captured mass"
    ),
    "HomeWinAggregation": (
        "Strict lower triangle"
    ),
    "DrawAggregation": "Matrix trace",
    "AwayWinAggregation": (
        "Strict upper triangle"
    ),
    "ProbabilityOrder": CLASS_ORDER,
}


# ------------------------------------------------------------
# Source provenance
# ------------------------------------------------------------

notebook_6_sha256 = hashlib.sha256(
    notebook_6_path.read_bytes()
).hexdigest()


# ------------------------------------------------------------
# Formal frozen implementation manifest
# ------------------------------------------------------------

production_implementation_manifest = {
    "ManifestVersion": "1.0",

    "Source": {
        "Notebook": (
            notebook_6_path
            .relative_to(project_root)
            .as_posix()
        ),
        "NotebookSHA256": notebook_6_sha256,
        "SourcePolicy": (
            "Final corrected walk-forward implementation"
        ),
    },

    "Features": {
        "FeatureCount": len(
            production_feature_columns
        ),
        "FeatureColumns": (
            production_feature_columns
        ),
        "HomeGoalFeatureColumns": (
            production_home_feature_columns
        ),
        "AwayGoalFeatureColumns": (
            production_away_feature_columns
        ),
        "SharedHomeAwayFeatureSchema": True,
        "OrderingPolicy": (
            "Preserve engineered dataframe column order"
        ),
        "SelectionRule": (
            "Numeric pre-match columns after excluding "
            "fixture identifiers and blocked target columns"
        ),
        "ExplicitRankBasedFeatureDrops": [],
    },

    "Preprocessing": (
        production_preprocessing_specification
    ),

    "HomePoissonModel": {
        "Class": "PoissonRegressor",
        "ExplicitParameters": (
            home_poisson_explicit_parameters
        ),
        "EffectiveParameters": (
            home_poisson_effective_parameters
        ),
        "SolverSource": (
            "sklearn default because Notebook 6 does not "
            "override solver"
        ),
        "TrainingMatrix": "X_train_scaled",
        "TrainingTarget": (
            "y_train_home_goals"
        ),
    },

    "AwayPoissonModel": {
        "Class": "PoissonRegressor",
        "ExplicitParameters": (
            away_poisson_explicit_parameters
        ),
        "EffectiveParameters": (
            away_poisson_effective_parameters
        ),
        "SolverSource": (
            "sklearn default because Notebook 6 does not "
            "override solver"
        ),
        "TrainingMatrix": "X_train_scaled",
        "TrainingTarget": (
            "y_train_away_goals"
        ),
    },

    "ScorelineConstruction": (
        production_scoreline_specification
    ),

    "CalibrationMethod": (
        "Original probabilities"
    ),

    "BookmakerOddsUsedAsPredictors": False,

    "RandomSeed": RANDOM_STATE,

    "ImplementationStatus": "Frozen",
}


# ------------------------------------------------------------
# Export manifest and feature schema
# ------------------------------------------------------------

implementation_manifest_path = (
    production_output_directory
    / "validated_implementation_manifest.json"
)

feature_schema_path = (
    production_output_directory
    / "validated_feature_schema.csv"
)

implementation_manifest_path.write_text(
    json.dumps(
        production_implementation_manifest,
        indent=4,
        ensure_ascii=False,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)

feature_schema_table = pd.DataFrame(
    {
        "FeaturePosition": range(
            1,
            len(production_feature_columns) + 1,
        ),
        "Feature": production_feature_columns,
        "Dtype": [
            str(
                engineered_data[column].dtype
            )
            for column
            in production_feature_columns
        ],
        "MissingValues": [
            int(
                engineered_data[column]
                .isna()
                .sum()
            )
            for column
            in production_feature_columns
        ],
    }
)

feature_schema_table.to_csv(
    feature_schema_path,
    index=False,
)

implementation_manifest_frozen = True


# ------------------------------------------------------------
# Final validation output
# ------------------------------------------------------------

implementation_summary = pd.DataFrame(
    [
        {
            "Detail": "Feature count",
            "FrozenValue": 70,
            "Status": "PASS",
        },
        {
            "Detail": "Home/away feature schema",
            "FrozenValue": "Shared",
            "Status": "PASS",
        },
        {
            "Detail": "Missing-value treatment",
            "FrozenValue": "Median imputation",
            "Status": "PASS",
        },
        {
            "Detail": "Scaling",
            "FrozenValue": "StandardScaler",
            "Status": "PASS",
        },
        {
            "Detail": "Encoding",
            "FrozenValue": None,
            "Status": "PASS",
        },
        {
            "Detail": "Explicit rank-based drops",
            "FrozenValue": 0,
            "Status": "PASS",
        },
        {
            "Detail": "Maximum modelled goals",
            "FrozenValue": MAX_MODELLED_GOALS,
            "Status": "PASS",
        },
        {
            "Detail": "Minimum expected goals",
            "FrozenValue": MINIMUM_EXPECTED_GOALS,
            "Status": "PASS",
        },
        {
            "Detail": "Probability order",
            "FrozenValue": CLASS_ORDER,
            "Status": "PASS",
        },
    ]
)

poisson_settings_table = pd.DataFrame(
    [
        {
            "Target": "Home goals",
            "Alpha": home_poisson_effective_parameters[
                "alpha"
            ],
            "Solver": home_poisson_effective_parameters[
                "solver"
            ],
            "FitIntercept": (
                home_poisson_effective_parameters[
                    "fit_intercept"
                ]
            ),
            "MaxIter": (
                home_poisson_effective_parameters[
                    "max_iter"
                ]
            ),
            "Tolerance": (
                home_poisson_effective_parameters[
                    "tol"
                ]
            ),
        },
        {
            "Target": "Away goals",
            "Alpha": away_poisson_effective_parameters[
                "alpha"
            ],
            "Solver": away_poisson_effective_parameters[
                "solver"
            ],
            "FitIntercept": (
                away_poisson_effective_parameters[
                    "fit_intercept"
                ]
            ),
            "MaxIter": (
                away_poisson_effective_parameters[
                    "max_iter"
                ]
            ),
            "Tolerance": (
                away_poisson_effective_parameters[
                    "tol"
                ]
            ),
        },
    ]
)


print("Implementation freeze summary:")
display(implementation_summary)

print("\nExact validated Poisson settings:")
display(poisson_settings_table)

print("\nFeature schema — first 10:")
display(feature_schema_table.head(10))

print("\nFeature schema — last 10:")
display(feature_schema_table.tail(10))

print(
    "\nManifest:",
    implementation_manifest_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "Feature schema:",
    feature_schema_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "\nImplementation manifest status: FROZEN"
)

Implementation freeze summary:


,Detail,FrozenValue,Status
0,Feature count,70,PASS
1,Home/away feature schema,Shared,PASS
2,Missing-value treatment,Median imputation,PASS
3,Scaling,StandardScaler,PASS
4,Encoding,None,PASS
5,Explicit rank-based drops,0,PASS
6,Maximum modelled goals,10,PASS
7,Minimum expected goals,0.000001,PASS
8,Probability order,"[H, D, A]",PASS



Exact validated Poisson settings:


,Target,Alpha,Solver,FitIntercept,MaxIter,Tolerance
0,Home goals,0.000,lbfgs,True,5000,1.000000e-08
1,Away goals,0.001,lbfgs,True,5000,1.000000e-08



Feature schema — first 10:


,FeaturePosition,Feature,Dtype,MissingValues
0,1,HomeEloBefore,float64,0
1,2,AwayEloBefore,float64,0
2,3,HomeRollingPoints5,float64,502
3,4,AwayRollingPoints5,float64,498
4,5,HomeRollingGoalsFor5,float64,502
5,6,HomeRollingGoalsAgainst5,float64,502
6,7,HomeRollingGoalDifference5,float64,502
7,8,HomeRollingWinRate5,float64,502
8,9,AwayRollingGoalsFor5,float64,498
9,10,AwayRollingGoalsAgainst5,float64,498



Feature schema — last 10:


,FeaturePosition,Feature,Dtype,MissingValues
60,61,PositionDifference,Int64,24
61,62,GoalDifferenceDifference,int64,0
62,63,HomeTop4Before,Int64,24
63,64,HomeTop6Before,Int64,24
64,65,HomeTopHalfBefore,Int64,24
65,66,HomeBottom3Before,Int64,24
66,67,AwayTop4Before,Int64,24
67,68,AwayTop6Before,Int64,24
68,69,AwayTopHalfBefore,Int64,24
69,70,AwayBottom3Before,Int64,24



Manifest: outputs/production_model/validated_implementation_manifest.json
Feature schema: outputs/production_model/validated_feature_schema.csv

Implementation manifest status: FROZEN


### Results and Interpretation

The validated Independent Poisson implementation has now been frozen successfully for production use.

The final predictor schema contains 70 ordered numeric pre-match features shared by both the home-goal and away-goal models. Target and fixture-identifier columns are excluded from the predictor set, and no additional rank-based feature removals are required in the final corrected implementation.

Missing predictor values are handled using median imputation, followed by standardisation with `StandardScaler`. No categorical encoding or estimator-stage team mapping is used.

The two Poisson regressions retain different regularisation strengths:

- home goals: $\alpha = 0.0$
- away goals: $\alpha = 0.001$

Both models use an intercept, the effective `lbfgs` solver, a maximum of 5,000 iterations and a convergence tolerance of $10^{-8}$.

Expected-goal predictions are bounded below at $10^{-6}$. Match-result probabilities are then constructed from an independent Poisson scoreline distribution covering 0–10 goals for each team. The captured scoreline probability mass is renormalised before aggregating the strict lower triangle, diagonal and strict upper triangle into home-win, draw and away-win probabilities respectively.

The final probability order remains $(H,D,A)$.

Two production specification artefacts have been written:

- `outputs/production_model/validated_implementation_manifest.json`
- `outputs/production_model/validated_feature_schema.csv`

The validated implementation is therefore frozen and no further hyperparameter tuning or feature selection will be performed during the production refit.

## 5. Construct the Full Production Training Matrix

This section reconstructs the complete modelling table from the engineered pre-match features and realised goal targets, then applies the frozen 70-feature production schema established in the previous section.

The resulting home- and away-goal training matrices are validated for chronological integrity, fixture uniqueness, target completeness, feature ordering, target leakage, numeric validity and goal-target admissibility.

Because the historical project data currently ends at 2024–25, the matrix constructed here represents all presently available validated training observations rather than the final 2026–27 production sample. The 2025–26 coverage check is therefore retained explicitly as a deployment-readiness condition: the matrix may pass all internal modelling checks while the final production refit remains blocked until the missing completed season has been incorporated and validated.

In [14]:
# ============================================================
# 5. Construct the Full Production Training Matrix
# ============================================================

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine", (
    "Could not locate the project root."
)

engineered_data_path = (
    project_root
    / "data"
    / "processed"
    / "premier_league_model_data.parquet"
)

goal_target_data_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_goal_targets_2015_16_to_2024_25.csv"
)

implementation_manifest_path = (
    project_root
    / "outputs"
    / "production_model"
    / "validated_implementation_manifest.json"
)

feature_schema_path = (
    project_root
    / "outputs"
    / "production_model"
    / "validated_feature_schema.csv"
)

for required_path in [
    engineered_data_path,
    goal_target_data_path,
    implementation_manifest_path,
    feature_schema_path,
]:
    assert required_path.is_file(), (
        f"Required production input is missing: {required_path}"
    )


# ------------------------------------------------------------
# Load frozen implementation
# ------------------------------------------------------------

with implementation_manifest_path.open(
    "r",
    encoding="utf-8",
) as file:
    production_implementation_manifest = json.load(
        file
    )

assert (
    production_implementation_manifest[
        "ImplementationStatus"
    ]
    == "Frozen"
)

production_feature_columns = list(
    production_implementation_manifest[
        "Features"
    ]["FeatureColumns"]
)

production_home_feature_columns = list(
    production_implementation_manifest[
        "Features"
    ]["HomeGoalFeatureColumns"]
)

production_away_feature_columns = list(
    production_implementation_manifest[
        "Features"
    ]["AwayGoalFeatureColumns"]
)

assert (
    production_feature_columns
    == production_home_feature_columns
    == production_away_feature_columns
), (
    "The frozen home and away feature schemas differ."
)

frozen_feature_schema = pd.read_csv(
    feature_schema_path
)

assert (
    frozen_feature_schema["Feature"].tolist()
    == production_feature_columns
), (
    "The saved feature-schema CSV does not match the "
    "implementation manifest."
)


# ------------------------------------------------------------
# Load source data
# ------------------------------------------------------------

engineered_data = pd.read_parquet(
    engineered_data_path
).copy()

goal_target_data = pd.read_csv(
    goal_target_data_path,
    low_memory=False,
).copy()

assert not engineered_data.empty
assert not goal_target_data.empty


# ------------------------------------------------------------
# Validate required fixture/target columns
# ------------------------------------------------------------

required_engineered_columns = [
    "Season",
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTR",
]

required_goal_target_columns = [
    "Season",
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
]

missing_engineered_columns = [
    column
    for column in required_engineered_columns
    if column not in engineered_data.columns
]

missing_target_columns = [
    column
    for column in required_goal_target_columns
    if column not in goal_target_data.columns
]

assert not missing_engineered_columns, (
    "Engineered data is missing required columns: "
    f"{missing_engineered_columns}"
)

assert not missing_target_columns, (
    "Goal-target data is missing required columns: "
    f"{missing_target_columns}"
)


# ------------------------------------------------------------
# Parse dates using the validated source conventions
#
# Engineered model dates were parsed day-first in Notebook 6.
# Raw goal-target dates matched the engineered fixtures under
# month-first/default parsing in Section 3.
# ------------------------------------------------------------

engineered_data["Date"] = pd.to_datetime(
    engineered_data["Date"],
    errors="raise",
    dayfirst=True,
).dt.normalize()

goal_target_data["Date"] = pd.to_datetime(
    goal_target_data["Date"],
    errors="raise",
    dayfirst=False,
).dt.normalize()


# ------------------------------------------------------------
# Normalise season labels
# ------------------------------------------------------------

def canonicalise_season(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    start_year = int(
        value[:4]
    )

    return (
        f"{start_year}-"
        f"{str(start_year + 1)[-2:]}"
    )


engineered_data["Season"] = (
    engineered_data["Season"]
    .map(canonicalise_season)
)

goal_target_data["Season"] = (
    goal_target_data["Season"]
    .map(canonicalise_season)
)


# ------------------------------------------------------------
# Fixture identity validation
# ------------------------------------------------------------

fixture_key_columns = [
    "Season",
    "Date",
    "HomeTeam",
    "AwayTeam",
]

engineered_duplicate_rows = int(
    engineered_data.duplicated(
        subset=fixture_key_columns,
        keep=False,
    ).sum()
)

target_duplicate_rows = int(
    goal_target_data.duplicated(
        subset=fixture_key_columns,
        keep=False,
    ).sum()
)

assert engineered_duplicate_rows == 0, (
    f"{engineered_duplicate_rows} engineered rows belong "
    "to duplicate fixture identities."
)

assert target_duplicate_rows == 0, (
    f"{target_duplicate_rows} target rows belong "
    "to duplicate fixture identities."
)


# ------------------------------------------------------------
# Merge realised goal targets
# ------------------------------------------------------------

goal_targets_for_merge = (
    goal_target_data[
        fixture_key_columns
        + [
            "FTHG",
            "FTAG",
        ]
    ]
    .rename(
        columns={
            "FTHG": "HomeGoalsTarget",
            "FTAG": "AwayGoalsTarget",
        }
    )
)

production_training_data = (
    engineered_data
    .merge(
        goal_targets_for_merge,
        on=fixture_key_columns,
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)

merge_status_counts = (
    production_training_data["_merge"]
    .value_counts()
)

unmatched_fixture_count = int(
    (
        production_training_data["_merge"]
        != "both"
    ).sum()
)

assert unmatched_fixture_count == 0, (
    f"{unmatched_fixture_count} engineered fixtures failed "
    "to match their goal targets."
)

production_training_data = (
    production_training_data
    .drop(columns="_merge")
    .sort_values(
        by=[
            "Date",
            "HomeTeam",
            "AwayTeam",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validate target completeness and admissibility
# ------------------------------------------------------------

target_columns = [
    "HomeGoalsTarget",
    "AwayGoalsTarget",
]

missing_target_rows = int(
    production_training_data[
        target_columns
    ].isna().any(axis=1).sum()
)

assert missing_target_rows == 0, (
    f"{missing_target_rows} fixtures contain missing goal targets."
)

for target_column in target_columns:

    numeric_target = pd.to_numeric(
        production_training_data[
            target_column
        ],
        errors="coerce",
    )

    invalid_target_mask = (
        numeric_target.isna()
        | (numeric_target < 0)
        | (numeric_target % 1 != 0)
    )

    assert not invalid_target_mask.any(), (
        f"{target_column} contains "
        f"{int(invalid_target_mask.sum())} invalid values."
    )

    production_training_data[
        target_column
    ] = numeric_target.astype(int)


# ------------------------------------------------------------
# Validate exact frozen feature schema
# ------------------------------------------------------------

missing_frozen_features = [
    feature
    for feature in production_feature_columns
    if feature not in production_training_data.columns
]

assert not missing_frozen_features, (
    "Frozen production features are missing from the "
    f"training data: {missing_frozen_features}"
)

production_X = production_training_data[
    production_feature_columns
].copy()

production_y_home = production_training_data[
    "HomeGoalsTarget"
].copy()

production_y_away = production_training_data[
    "AwayGoalsTarget"
].copy()

assert (
    production_X.columns.tolist()
    == production_feature_columns
), (
    "Production feature ordering does not match the frozen schema."
)

assert production_X.shape[1] == 70


# ------------------------------------------------------------
# Leakage validation
# ------------------------------------------------------------

forbidden_predictors = {
    "FTR",
    "Result",
    "result",
    "FullTimeResult",
    "FTHG",
    "FTAG",
    "HomeGoalsTarget",
    "AwayGoalsTarget",
    "FullTimeHomeGoals",
    "FullTimeAwayGoals",
    "HomeGoals",
    "AwayGoals",
}

leakage_columns = sorted(
    set(production_X.columns)
    .intersection(
        forbidden_predictors
    )
)

assert not leakage_columns, (
    "Target leakage entered the production matrix: "
    f"{leakage_columns}"
)


# ------------------------------------------------------------
# Numeric and finite-value validation
#
# Missing predictor values are allowed here because the frozen
# preprocessing stage handles them with median imputation.
# All observed predictor values must nevertheless be finite.
# ------------------------------------------------------------

non_numeric_features = [
    column
    for column in production_X.columns
    if not pd.api.types.is_numeric_dtype(
        production_X[column]
    )
]

assert not non_numeric_features, (
    "Non-numeric features entered the production matrix: "
    f"{non_numeric_features}"
)

feature_array = production_X.to_numpy(
    dtype=float
)

infinite_value_count = int(
    np.isinf(feature_array).sum()
)

missing_feature_value_count = int(
    np.isnan(feature_array).sum()
)

features_with_missing_values = int(
    production_X.isna().any().sum()
)

assert infinite_value_count == 0, (
    f"The production matrix contains {infinite_value_count} "
    "infinite predictor values."
)


# ------------------------------------------------------------
# Chronological validation
# ------------------------------------------------------------

assert production_training_data[
    "Date"
].notna().all()

assert production_training_data[
    "Date"
].is_monotonic_increasing, (
    "Production training data is not chronologically ordered."
)

production_training_start = (
    production_training_data[
        "Date"
    ].min()
)

production_training_cutoff = (
    production_training_data[
        "Date"
    ].max()
)


# ------------------------------------------------------------
# Season-level audit
# ------------------------------------------------------------

production_season_summary = (
    production_training_data
    .groupby(
        "Season",
        sort=True,
    )
    .agg(
        Fixtures=("Season", "size"),
        HomeGoalTargets=(
            "HomeGoalsTarget",
            "count",
        ),
        AwayGoalTargets=(
            "AwayGoalsTarget",
            "count",
        ),
        UniqueHomeTeams=(
            "HomeTeam",
            "nunique",
        ),
        UniqueAwayTeams=(
            "AwayTeam",
            "nunique",
        ),
        FirstMatch=("Date", "min"),
        LastMatch=("Date", "max"),
    )
    .reset_index()
)

production_season_summary[
    "CompleteTargets"
] = (
    production_season_summary[
        "HomeGoalTargets"
    ].eq(
        production_season_summary[
            "Fixtures"
        ]
    )
    & production_season_summary[
        "AwayGoalTargets"
    ].eq(
        production_season_summary[
            "Fixtures"
        ]
    )
)


# ------------------------------------------------------------
# 2025-26 deployment-readiness check
# ------------------------------------------------------------

season_2025_26_rows = (
    production_training_data.loc[
        production_training_data[
            "Season"
        ]
        == "2025-26"
    ]
)

has_complete_2025_26 = (
    len(
        season_2025_26_rows
    )
    == 380
    and season_2025_26_rows[
        target_columns
    ].notna().all().all()
)

if has_complete_2025_26:
    season_2025_26_status = (
        "COMPLETE — 380 completed fixtures available."
    )
else:
    season_2025_26_status = (
        "ABSENT — final 2026-27 production refit "
        "cannot yet use the completed 2025-26 season."
    )

current_training_matrix_valid = True

final_production_refit_ready = bool(
    has_complete_2025_26
)


# ------------------------------------------------------------
# Compact validation report
# ------------------------------------------------------------

training_validation_table = pd.DataFrame(
    [
        {
            "Validation": "Unique fixture identities",
            "Value": (
                engineered_duplicate_rows
                + target_duplicate_rows
            ),
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": "Unmatched fixtures",
            "Value": unmatched_fixture_count,
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": "Missing goal targets",
            "Value": missing_target_rows,
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": "Feature count",
            "Value": production_X.shape[1],
            "Expected": 70,
            "Status": "PASS",
        },
        {
            "Validation": "Leakage predictors",
            "Value": len(leakage_columns),
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": "Non-numeric predictors",
            "Value": len(non_numeric_features),
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": "Infinite predictor values",
            "Value": infinite_value_count,
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": "Chronological ordering",
            "Value": (
                production_training_data[
                    "Date"
                ].is_monotonic_increasing
            ),
            "Expected": True,
            "Status": "PASS",
        },
        {
            "Validation": "Completed 2025-26 season",
            "Value": len(
                season_2025_26_rows
            ),
            "Expected": 380,
            "Status": (
                "PASS"
                if has_complete_2025_26
                else "ACTION REQUIRED"
            ),
        },
    ]
)


training_matrix_profile = pd.DataFrame(
    [
        {
            "Metric": "Training fixtures",
            "Value": len(
                production_training_data
            ),
        },
        {
            "Metric": "Production features",
            "Value": production_X.shape[1],
        },
        {
            "Metric": "Training start",
            "Value": (
                production_training_start
                .date()
                .isoformat()
            ),
        },
        {
            "Metric": "Current training cutoff",
            "Value": (
                production_training_cutoff
                .date()
                .isoformat()
            ),
        },
        {
            "Metric": "Seasons represented",
            "Value": (
                production_training_data[
                    "Season"
                ].nunique()
            ),
        },
        {
            "Metric": "Missing predictor values",
            "Value": (
                missing_feature_value_count
            ),
        },
        {
            "Metric": "Features containing missing values",
            "Value": (
                features_with_missing_values
            ),
        },
        {
            "Metric": "2025-26 status",
            "Value": (
                season_2025_26_status
            ),
        },
        {
            "Metric": "Final 2026-27 refit ready",
            "Value": (
                final_production_refit_ready
            ),
        },
    ]
)


print("Production training-matrix profile:")
display(training_matrix_profile)

print("\nTraining-matrix validation:")
display(training_validation_table)

print("\nSeason coverage:")
display(production_season_summary)

print(
    "\nCurrent matrices:",
    f"\n- X: {production_X.shape}",
    f"\n- Home target: {production_y_home.shape}",
    f"\n- Away target: {production_y_away.shape}",
)

print(
    "\n2025-26 deployment assessment:",
    season_2025_26_status,
)

print(
    "\nCurrent training matrix status: VALID"
)

print(
    "Final 2026-27 production refit ready:",
    final_production_refit_ready,
)

Production training-matrix profile:


,Metric,Value
0,Training fixtures,3800
1,Production features,70
2,Training start,2015-08-08
3,Current training cutoff,2025-05-25
4,Seasons represented,10
5,Missing predictor values,23463
6,Features containing missing values,47
7,2025-26 status,ABSENT — final 2026-27 production refit cannot...
8,Final 2026-27 refit ready,False



Training-matrix validation:


,Validation,Value,Expected,Status
0,Unique fixture identities,0,0,PASS
1,Unmatched fixtures,0,0,PASS
2,Missing goal targets,0,0,PASS
3,Feature count,70,70,PASS
4,Leakage predictors,0,0,PASS
5,Non-numeric predictors,0,0,PASS
6,Infinite predictor values,0,0,PASS
7,Chronological ordering,True,True,PASS
8,Completed 2025-26 season,0,380,ACTION REQUIRED



Season coverage:


,Season,Fixtures,HomeGoalTargets,AwayGoalTargets,UniqueHomeTeams,UniqueAwayTeams,FirstMatch,LastMatch,CompleteTargets
0,2015-16,380,380,380,20,20,2015-08-08,2016-05-17,True
1,2016-17,380,380,380,20,20,2016-08-13,2017-05-21,True
2,2017-18,380,380,380,20,20,2017-08-11,2018-05-13,True
3,2018-19,380,380,380,20,20,2018-08-10,2019-05-12,True
4,2019-20,380,380,380,20,20,2019-08-09,2020-07-26,True
5,2020-21,380,380,380,20,20,2020-09-12,2021-05-23,True
6,2021-22,380,380,380,20,20,2021-08-13,2022-05-22,True
7,2022-23,380,380,380,20,20,2022-08-05,2023-05-28,True
8,2023-24,380,380,380,20,20,2023-08-11,2024-05-19,True
9,2024-25,380,380,380,20,20,2024-08-16,2025-05-25,True



Current matrices: 
- X: (3800, 70) 
- Home target: (3800,) 
- Away target: (3800,)

2025-26 deployment assessment: ABSENT — final 2026-27 production refit cannot yet use the completed 2025-26 season.

Current training matrix status: VALID
Final 2026-27 production refit ready: False


### Results and Interpretation

The currently available production training matrix has been constructed successfully and passes all internal modelling validations.

The matrix contains 3,800 Premier League fixtures across ten complete seasons from 2015–16 through 2024–25. Each season contributes 380 fixtures with complete home- and away-goal targets. The training period runs from 8 August 2015 to 25 May 2025.

The frozen production feature schema has been reproduced exactly, giving a predictor matrix of shape $(3800, 70)$ and separate home- and away-goal target vectors of length 3,800.

All structural validation checks passed:

- zero duplicate fixture identities;
- zero unmatched fixtures;
- zero missing goal targets;
- exactly 70 production predictors;
- zero target-leakage columns;
- zero non-numeric predictors;
- zero infinite predictor values;
- correct chronological ordering.

There are 23,463 missing predictor observations distributed across 47 of the 70 features. This does not invalidate the matrix because the frozen production pipeline explicitly handles missing predictor values using training-fitted median imputation.

The only failed deployment-readiness condition is historical coverage. No 2025–26 fixtures are currently present, so the final 2026–27 production refit cannot yet be performed. The existing 2015–16 to 2024–25 training matrix is valid, but the completed 2025–26 season must first be added and passed through the same pre-match feature-engineering pipeline.

Therefore:

- **Current training matrix:** VALID
- **Final 2026–27 production refit:** NOT YET READY
- **Outstanding blocker:** completed 2025–26 data and engineered features

## 6. Add and Validate the Completed 2025–26 Season

The production dataset must be extended through the end of the 2025–26 Premier League season before the final 2026–27 model refit.

This section downloads the completed 2025–26 Premier League results from the same public Football-Data source family used for historical football results, validates the season independently, and stores a local raw copy for reproducibility.

The new season is then combined with the existing 2015–16 to 2024–25 goal-target dataset using a canonical ISO date representation. The original historical target file is preserved rather than overwritten.

No engineered predictors are created in this section. Its purpose is solely to establish a validated 11-season raw result and goal-target history. The 2025–26 fixtures will be passed through the established pre-match feature-engineering pipeline in the following section.

In [15]:
# ============================================================
# 6. Add and Validate the Completed 2025–26 Season
# ============================================================

from __future__ import annotations

import io
import json
import re
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

FOOTBALL_DATA_2025_26_URL = (
    "https://www.football-data.co.uk/mmz4281/2526/E0.csv"
)

SEASON_LABEL = "2025-26"
EXPECTED_FIXTURES = 380
EXPECTED_TEAMS = 20

REQUIRED_RESULT_COLUMNS = [
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
]


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine", (
    "Could not locate the project root."
)


historical_target_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_goal_targets_2015_16_to_2024_25.csv"
)

season_2025_26_raw_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_2025_26.csv"
)

extended_target_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_goal_targets_2015_16_to_2025_26.csv"
)

data_update_metadata_path = (
    project_root
    / "outputs"
    / "production_model"
    / "historical_data_update_2025_26.json"
)

assert historical_target_path.is_file(), (
    "Existing historical goal-target dataset was not found."
)

data_update_metadata_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Download 2025-26 Premier League results
# ------------------------------------------------------------

request = Request(
    FOOTBALL_DATA_2025_26_URL,
    headers={
        "User-Agent": (
            "Mozilla/5.0 "
            "Premier-League-Probability-Engine/1.0"
        )
    },
)

try:
    with urlopen(
        request,
        timeout=30,
    ) as response:
        downloaded_bytes = response.read()

except Exception as error:
    raise RuntimeError(
        "The 2025-26 Football-Data CSV could not be downloaded. "
        "Check the internet connection and source URL."
    ) from error


if not downloaded_bytes:
    raise RuntimeError(
        "Football-Data returned an empty response."
    )


try:
    season_source = pd.read_csv(
        io.BytesIO(downloaded_bytes),
        low_memory=False,
    )

except Exception as error:
    raise RuntimeError(
        "The downloaded 2025-26 file could not be parsed as CSV."
    ) from error


# Remove completely empty rows/columns if the source contains any.
season_source = (
    season_source
    .dropna(
        axis=0,
        how="all",
    )
    .dropna(
        axis=1,
        how="all",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validate required source schema
# ------------------------------------------------------------

missing_source_columns = [
    column
    for column in REQUIRED_RESULT_COLUMNS
    if column not in season_source.columns
]

assert not missing_source_columns, (
    "2025-26 source is missing required columns: "
    f"{missing_source_columns}"
)

if "Div" in season_source.columns:
    unexpected_divisions = sorted(
        season_source[
            "Div"
        ]
        .dropna()
        .astype(str)
        .unique()
    )

    assert unexpected_divisions == ["E0"], (
        "The downloaded file contains unexpected divisions: "
        f"{unexpected_divisions}"
    )


# ------------------------------------------------------------
# Parse and normalise fixture fields
# ------------------------------------------------------------

season_2025_26 = season_source.copy()

try:
    season_2025_26["Date"] = pd.to_datetime(
        season_2025_26["Date"],
        errors="raise",
        dayfirst=True,
        format="mixed",
    ).dt.normalize()

except (TypeError, ValueError):
    season_2025_26["Date"] = pd.to_datetime(
        season_2025_26["Date"],
        errors="raise",
        dayfirst=True,
    ).dt.normalize()


season_2025_26["HomeTeam"] = (
    season_2025_26["HomeTeam"]
    .astype("string")
    .str.strip()
)

season_2025_26["AwayTeam"] = (
    season_2025_26["AwayTeam"]
    .astype("string")
    .str.strip()
)

season_2025_26["FTR"] = (
    season_2025_26["FTR"]
    .astype("string")
    .str.strip()
    .str.upper()
)


for goal_column in [
    "FTHG",
    "FTAG",
]:
    numeric_goals = pd.to_numeric(
        season_2025_26[
            goal_column
        ],
        errors="coerce",
    )

    invalid_goal_mask = (
        numeric_goals.isna()
        | (numeric_goals < 0)
        | (numeric_goals % 1 != 0)
    )

    assert not invalid_goal_mask.any(), (
        f"{goal_column} contains "
        f"{int(invalid_goal_mask.sum())} invalid values."
    )

    season_2025_26[
        goal_column
    ] = numeric_goals.astype(int)


season_2025_26.insert(
    0,
    "Season",
    SEASON_LABEL,
)


# ------------------------------------------------------------
# Independent 2025-26 season validation
# ------------------------------------------------------------

fixture_key_columns = [
    "Season",
    "Date",
    "HomeTeam",
    "AwayTeam",
]

duplicate_fixture_rows = int(
    season_2025_26.duplicated(
        subset=fixture_key_columns,
        keep=False,
    ).sum()
)

missing_result_rows = int(
    season_2025_26[
        REQUIRED_RESULT_COLUMNS
    ].isna().any(axis=1).sum()
)

invalid_results = sorted(
    set(
        season_2025_26[
            "FTR"
        ].dropna()
    )
    - {
        "H",
        "D",
        "A",
    }
)

home_teams = set(
    season_2025_26[
        "HomeTeam"
    ].dropna()
)

away_teams = set(
    season_2025_26[
        "AwayTeam"
    ].dropna()
)

all_teams = (
    home_teams
    | away_teams
)

season_start_boundary = pd.Timestamp(
    "2025-07-01"
)

season_end_boundary = pd.Timestamp(
    "2026-06-30"
)

dates_inside_season = (
    season_2025_26[
        "Date"
    ].between(
        season_start_boundary,
        season_end_boundary,
        inclusive="both",
    )
)


season_validation_checks = pd.DataFrame(
    [
        {
            "Validation": "Fixture count",
            "Value": len(
                season_2025_26
            ),
            "Expected": EXPECTED_FIXTURES,
            "Status": (
                "PASS"
                if len(
                    season_2025_26
                )
                == EXPECTED_FIXTURES
                else "FAIL"
            ),
        },
        {
            "Validation": "Unique teams",
            "Value": len(
                all_teams
            ),
            "Expected": EXPECTED_TEAMS,
            "Status": (
                "PASS"
                if len(
                    all_teams
                )
                == EXPECTED_TEAMS
                else "FAIL"
            ),
        },
        {
            "Validation": "Duplicate fixture rows",
            "Value": duplicate_fixture_rows,
            "Expected": 0,
            "Status": (
                "PASS"
                if duplicate_fixture_rows
                == 0
                else "FAIL"
            ),
        },
        {
            "Validation": "Missing result rows",
            "Value": missing_result_rows,
            "Expected": 0,
            "Status": (
                "PASS"
                if missing_result_rows
                == 0
                else "FAIL"
            ),
        },
        {
            "Validation": "Invalid FTR labels",
            "Value": len(
                invalid_results
            ),
            "Expected": 0,
            "Status": (
                "PASS"
                if not invalid_results
                else "FAIL"
            ),
        },
        {
            "Validation": "Dates inside 2025-26",
            "Value": int(
                dates_inside_season.sum()
            ),
            "Expected": EXPECTED_FIXTURES,
            "Status": (
                "PASS"
                if dates_inside_season.all()
                else "FAIL"
            ),
        },
    ]
)


failed_checks = (
    season_validation_checks.loc[
        season_validation_checks[
            "Status"
        ]
        != "PASS"
    ]
)

if not failed_checks.empty:
    display(
        season_validation_checks
    )

    raise RuntimeError(
        "The downloaded 2025-26 season failed validation. "
        "No project files have been updated."
    )


# ------------------------------------------------------------
# Save validated full raw season
#
# Store dates in ISO form so future parsing is unambiguous.
# ------------------------------------------------------------

season_2025_26_to_save = (
    season_2025_26.copy()
)

season_2025_26_to_save[
    "Date"
] = (
    season_2025_26_to_save[
        "Date"
    ]
    .dt.strftime(
        "%Y-%m-%d"
    )
)

season_2025_26_to_save.to_csv(
    season_2025_26_raw_path,
    index=False,
)


# ------------------------------------------------------------
# Load and canonicalise existing 2015-16 -> 2024-25 targets
# ------------------------------------------------------------

historical_targets = pd.read_csv(
    historical_target_path,
    low_memory=False,
)

required_historical_columns = [
    "Season",
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
]

missing_historical_columns = [
    column
    for column in required_historical_columns
    if column not in historical_targets.columns
]

assert not missing_historical_columns, (
    "Historical target dataset is missing columns: "
    f"{missing_historical_columns}"
)


def canonicalise_season(
    value,
):
    if pd.isna(value):
        return None

    value_string = str(
        value
    ).strip()

    year_match = re.search(
        r"(20\d{2})",
        value_string,
    )

    if year_match is None:
        raise ValueError(
            f"Could not interpret season label: {value!r}"
        )

    start_year = int(
        year_match.group(1)
    )

    return (
        f"{start_year}-"
        f"{str(start_year + 1)[-2:]}"
    )


historical_targets[
    "Season"
] = (
    historical_targets[
        "Season"
    ].map(
        canonicalise_season
    )
)


# Section 3 established that this historical target file aligns
# correctly under month-first/default interpretation.
try:
    historical_targets[
        "Date"
    ] = pd.to_datetime(
        historical_targets[
            "Date"
        ],
        errors="raise",
        dayfirst=False,
        format="mixed",
    ).dt.normalize()

except (TypeError, ValueError):
    historical_targets[
        "Date"
    ] = pd.to_datetime(
        historical_targets[
            "Date"
        ],
        errors="raise",
        dayfirst=False,
    ).dt.normalize()


# ------------------------------------------------------------
# Build canonical 2025-26 target table
# ------------------------------------------------------------

new_goal_targets = (
    season_2025_26[
        required_historical_columns
    ]
    .copy()
)


# ------------------------------------------------------------
# Compare team transition from 2024-25 -> 2025-26
# ------------------------------------------------------------

previous_season_rows = (
    historical_targets.loc[
        historical_targets[
            "Season"
        ]
        == "2024-25"
    ]
)

previous_season_teams = (
    set(
        previous_season_rows[
            "HomeTeam"
        ].dropna()
    )
    | set(
        previous_season_rows[
            "AwayTeam"
        ].dropna()
    )
)

current_season_teams = (
    set(
        new_goal_targets[
            "HomeTeam"
        ].dropna()
    )
    | set(
        new_goal_targets[
            "AwayTeam"
        ].dropna()
    )
)

new_teams_2025_26 = sorted(
    current_season_teams
    - previous_season_teams
)

departed_teams_2025_26 = sorted(
    previous_season_teams
    - current_season_teams
)


# ------------------------------------------------------------
# Combine all 11 seasons
# ------------------------------------------------------------

extended_goal_targets = pd.concat(
    [
        historical_targets[
            required_historical_columns
        ],
        new_goal_targets,
    ],
    ignore_index=True,
)

extended_goal_targets = (
    extended_goal_targets
    .sort_values(
        by=[
            "Date",
            "HomeTeam",
            "AwayTeam",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Full 11-season validation
# ------------------------------------------------------------

extended_duplicate_rows = int(
    extended_goal_targets.duplicated(
        subset=[
            "Season",
            "Date",
            "HomeTeam",
            "AwayTeam",
        ],
        keep=False,
    ).sum()
)

extended_missing_targets = int(
    extended_goal_targets[
        [
            "FTHG",
            "FTAG",
            "FTR",
        ]
    ].isna().any(axis=1).sum()
)

extended_invalid_results = sorted(
    set(
        extended_goal_targets[
            "FTR"
        ].astype(str)
    )
    - {
        "H",
        "D",
        "A",
    }
)


extended_season_summary = (
    extended_goal_targets
    .groupby(
        "Season",
        sort=True,
    )
    .agg(
        Fixtures=("Season", "size"),
        CompletedHomeGoals=(
            "FTHG",
            "count",
        ),
        CompletedAwayGoals=(
            "FTAG",
            "count",
        ),
        CompletedResults=(
            "FTR",
            "count",
        ),
        UniqueHomeTeams=(
            "HomeTeam",
            "nunique",
        ),
        UniqueAwayTeams=(
            "AwayTeam",
            "nunique",
        ),
        FirstMatch=(
            "Date",
            "min",
        ),
        LastMatch=(
            "Date",
            "max",
        ),
    )
    .reset_index()
)

extended_season_summary[
    "CompleteSeason"
] = (
    extended_season_summary[
        "Fixtures"
    ].eq(
        EXPECTED_FIXTURES
    )
    & extended_season_summary[
        "CompletedHomeGoals"
    ].eq(
        EXPECTED_FIXTURES
    )
    & extended_season_summary[
        "CompletedAwayGoals"
    ].eq(
        EXPECTED_FIXTURES
    )
    & extended_season_summary[
        "CompletedResults"
    ].eq(
        EXPECTED_FIXTURES
    )
    & extended_season_summary[
        "UniqueHomeTeams"
    ].eq(
        EXPECTED_TEAMS
    )
    & extended_season_summary[
        "UniqueAwayTeams"
    ].eq(
        EXPECTED_TEAMS
    )
)


assert len(
    extended_goal_targets
) == 4180, (
    "Expected 4,180 fixtures after adding 2025-26, "
    f"found {len(extended_goal_targets):,}."
)

assert extended_goal_targets[
    "Season"
].nunique() == 11, (
    "Expected 11 seasons after the historical update."
)

assert extended_duplicate_rows == 0, (
    "Duplicate fixture identities exist in the "
    "extended target dataset."
)

assert extended_missing_targets == 0, (
    "Missing goal/result targets exist in the "
    "extended target dataset."
)

assert not extended_invalid_results, (
    "Invalid H/D/A labels exist in the extended dataset: "
    f"{extended_invalid_results}"
)

assert extended_season_summary[
    "CompleteSeason"
].all(), (
    "At least one season failed the complete-season audit."
)


# ------------------------------------------------------------
# Save canonical extended target history
# ------------------------------------------------------------

extended_goal_targets_to_save = (
    extended_goal_targets.copy()
)

extended_goal_targets_to_save[
    "Date"
] = (
    extended_goal_targets_to_save[
        "Date"
    ]
    .dt.strftime(
        "%Y-%m-%d"
    )
)

extended_goal_targets_to_save.to_csv(
    extended_target_path,
    index=False,
)


# ------------------------------------------------------------
# Save update provenance
# ------------------------------------------------------------

data_update_metadata = {
    "SeasonAdded": SEASON_LABEL,
    "SourceURL": (
        FOOTBALL_DATA_2025_26_URL
    ),
    "FixturesAdded": int(
        len(
            season_2025_26
        )
    ),
    "TotalFixturesAfterUpdate": int(
        len(
            extended_goal_targets
        )
    ),
    "TotalSeasonsAfterUpdate": int(
        extended_goal_targets[
            "Season"
        ].nunique()
    ),
    "NewTeamsRelativeTo2024_25": (
        new_teams_2025_26
    ),
    "DepartedTeamsRelativeTo2024_25": (
        departed_teams_2025_26
    ),
    "RawSeasonFile": (
        season_2025_26_raw_path
        .relative_to(
            project_root
        )
        .as_posix()
    ),
    "ExtendedTargetFile": (
        extended_target_path
        .relative_to(
            project_root
        )
        .as_posix()
    ),
}

data_update_metadata_path.write_text(
    json.dumps(
        data_update_metadata,
        indent=4,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

season_2025_26_profile = pd.DataFrame(
    [
        {
            "Metric": "Fixtures",
            "Value": len(
                season_2025_26
            ),
        },
        {
            "Metric": "Unique teams",
            "Value": len(
                current_season_teams
            ),
        },
        {
            "Metric": "First fixture",
            "Value": (
                season_2025_26[
                    "Date"
                ]
                .min()
                .date()
                .isoformat()
            ),
        },
        {
            "Metric": "Last fixture",
            "Value": (
                season_2025_26[
                    "Date"
                ]
                .max()
                .date()
                .isoformat()
            ),
        },
        {
            "Metric": (
                "New teams vs 2024-25"
            ),
            "Value": (
                new_teams_2025_26
            ),
        },
        {
            "Metric": (
                "Departed teams vs 2024-25"
            ),
            "Value": (
                departed_teams_2025_26
            ),
        },
    ]
)


print("2025-26 source validation:")
display(
    season_validation_checks
)

print("\n2025-26 season profile:")
display(
    season_2025_26_profile
)

print("\nExtended historical season coverage:")
display(
    extended_season_summary
)

print(
    "\nRaw 2025-26 file:",
    season_2025_26_raw_path
    .relative_to(
        project_root
    )
    .as_posix(),
)

print(
    "Extended goal-target file:",
    extended_target_path
    .relative_to(
        project_root
    )
    .as_posix(),
)

print(
    "\nHistorical results update status: VALID"
)

print(
    "Fixtures available through 2025-26:",
    f"{len(extended_goal_targets):,}",
)

print(
    "Seasons available:",
    extended_goal_targets[
        "Season"
    ].nunique(),
)

2025-26 source validation:


,Validation,Value,Expected,Status
0,Fixture count,380,380,PASS
1,Unique teams,20,20,PASS
2,Duplicate fixture rows,0,0,PASS
3,Missing result rows,0,0,PASS
4,Invalid FTR labels,0,0,PASS
5,Dates inside 2025-26,380,380,PASS



2025-26 season profile:


,Metric,Value
0,Fixtures,380
1,Unique teams,20
2,First fixture,2025-08-15
3,Last fixture,2026-05-24
4,New teams vs 2024-25,"[Burnley, Leeds, Sunderland]"
5,Departed teams vs 2024-25,"[Ipswich, Leicester, Southampton]"



Extended historical season coverage:


,Season,Fixtures,CompletedHomeGoals,CompletedAwayGoals,CompletedResults,UniqueHomeTeams,UniqueAwayTeams,FirstMatch,LastMatch,CompleteSeason
0,2015-16,380,380,380,380,20,20,2015-08-08,2016-05-17,True
1,2016-17,380,380,380,380,20,20,2016-08-13,2017-05-21,True
2,2017-18,380,380,380,380,20,20,2017-08-11,2018-05-13,True
3,2018-19,380,380,380,380,20,20,2018-08-10,2019-05-12,True
4,2019-20,380,380,380,380,20,20,2019-08-09,2020-07-26,True
5,2020-21,380,380,380,380,20,20,2020-09-12,2021-05-23,True
6,2021-22,380,380,380,380,20,20,2021-08-13,2022-05-22,True
7,2022-23,380,380,380,380,20,20,2022-08-05,2023-05-28,True
8,2023-24,380,380,380,380,20,20,2023-08-11,2024-05-19,True
9,2024-25,380,380,380,380,20,20,2024-08-16,2025-05-25,True



Raw 2025-26 file: data/raw/premier_league_2025_26.csv
Extended goal-target file: data/raw/premier_league_goal_targets_2015_16_to_2025_26.csv

Historical results update status: VALID
Fixtures available through 2025-26: 4,180
Seasons available: 11


### Results and Interpretation

The completed 2025–26 Premier League season has been incorporated successfully into the historical result dataset.

All source-level validation checks passed. The season contains exactly 380 fixtures involving 20 teams, with no duplicate fixture identities, missing results or invalid full-time result labels. All fixture dates fall within the expected 2025–26 season window.

The season runs from 15 August 2025 to 24 May 2026. Relative to the 2024–25 Premier League, Burnley, Leeds and Sunderland enter the league, while Ipswich, Leicester and Southampton leave it.

After combining the new season with the existing historical target dataset, the project now contains 4,180 completed Premier League fixtures across eleven consecutive seasons from 2015–16 through 2025–26. Every season contains exactly 380 complete home-goal, away-goal and match-result observations.

The raw historical result history is therefore complete for the final 2026–27 production refit.

The next requirement is to generate the same 70 pre-match predictors for the 380 new 2025–26 fixtures. Only after those engineered features have been created and validated can the production training matrix be extended from 3,800 to 4,180 observations.

## 7. Engineer and Validate the 2025–26 Production Features

The completed 2025–26 results must now be transformed into exactly the same 70 pre-match predictors used by the validated production model.

The existing 2015–16 to 2024–25 engineered observations are preserved unchanged. Only the new 2025–26 fixtures are engineered and appended.

The feature-generation process reproduces the established feature families:

- pre-match Elo ratings;
- five-match general rolling form;
- five-match venue-specific rolling form;
- rest days and short-rest indicators;
- seven- and fourteen-day fixture congestion;
- relative matchup differences;
- team-specific match number and season progress;
- pre-match league-table state;
- league-position category indicators.

All rolling, cumulative and Elo features use information available strictly before the current fixture. League-table states are calculated before each kickoff batch and updated only after the corresponding pre-match snapshots have been recorded.

The completed 2025–26 feature rows must match the frozen 70-column production schema exactly before they are appended to the historical engineered dataset.

In [16]:
# ============================================================
# 7. Engineer and Validate the 2025-26 Production Features
# ============================================================

from __future__ import annotations

from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TARGET_SEASON = "2025-26"

INITIAL_ELO = 1500.0
ELO_K_FACTOR = 20.0
ELO_HOME_ADVANTAGE = 50.0
ELO_SCALE = 400.0

ROLLING_WINDOW = 5

EXPECTED_SEASON_FIXTURES = 380
EXPECTED_TOTAL_FIXTURES = 4180
EXPECTED_FEATURE_COUNT = 70


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine"


historical_engineered_path = (
    project_root
    / "data"
    / "processed"
    / "premier_league_model_data.parquet"
)

extended_results_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_goal_targets_2015_16_to_2025_26.csv"
)

season_raw_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_2025_26.csv"
)

feature_schema_path = (
    project_root
    / "outputs"
    / "production_model"
    / "validated_feature_schema.csv"
)

season_feature_output_path = (
    project_root
    / "data"
    / "processed"
    / "premier_league_model_data_2025_26.csv"
)

extended_parquet_path = (
    project_root
    / "data"
    / "processed"
    / "premier_league_model_data_through_2025_26.parquet"
)

extended_csv_path = (
    project_root
    / "data"
    / "processed"
    / "premier_league_model_data_through_2025_26.csv"
)


for required_path in [
    historical_engineered_path,
    extended_results_path,
    season_raw_path,
    feature_schema_path,
]:
    assert required_path.is_file(), (
        f"Required file is missing: {required_path}"
    )


# ------------------------------------------------------------
# Load frozen schema and historical data
# ------------------------------------------------------------

historical_engineered = pd.read_parquet(
    historical_engineered_path
).copy()

extended_results = pd.read_csv(
    extended_results_path,
    low_memory=False,
).copy()

season_raw = pd.read_csv(
    season_raw_path,
    low_memory=False,
).copy()

frozen_schema = pd.read_csv(
    feature_schema_path
)

production_feature_columns = (
    frozen_schema["Feature"].tolist()
)

assert len(production_feature_columns) == EXPECTED_FEATURE_COUNT
assert len(set(production_feature_columns)) == EXPECTED_FEATURE_COUNT
assert len(historical_engineered) == 3800


# ------------------------------------------------------------
# Date helpers
# ------------------------------------------------------------

def parse_dates(values):
    try:
        return pd.to_datetime(
            values,
            errors="raise",
            format="mixed",
            dayfirst=False,
        ).dt.normalize()

    except (TypeError, ValueError):
        return pd.to_datetime(
            values,
            errors="raise",
            dayfirst=False,
        ).dt.normalize()


extended_results["Date"] = parse_dates(
    extended_results["Date"]
)

historical_engineered["Date"] = pd.to_datetime(
    historical_engineered["Date"],
    errors="raise",
).dt.normalize()

season_raw["Date"] = parse_dates(
    season_raw["Date"]
)


# ------------------------------------------------------------
# Construct clean 2025-26 fixture table
# ------------------------------------------------------------

season_matches = (
    extended_results.loc[
        extended_results["Season"] == TARGET_SEASON,
        [
            "Season",
            "Date",
            "HomeTeam",
            "AwayTeam",
            "FTHG",
            "FTAG",
            "FTR",
        ],
    ]
    .copy()
)

assert len(season_matches) == EXPECTED_SEASON_FIXTURES


# Recover kickoff time from the full Football-Data season file
# where available. Time is used for the pre-match league-table
# batching only.
if "Time" in season_raw.columns:

    season_time_lookup = season_raw[
        [
            "Date",
            "HomeTeam",
            "AwayTeam",
            "Time",
        ]
    ].copy()

    season_matches = season_matches.merge(
        season_time_lookup,
        on=[
            "Date",
            "HomeTeam",
            "AwayTeam",
        ],
        how="left",
        validate="one_to_one",
    )

else:
    season_matches["Time"] = pd.NA


season_matches["_OriginalOrder"] = np.arange(
    len(season_matches)
)


# ------------------------------------------------------------
# Kickoff timestamps
# ------------------------------------------------------------

parsed_time = pd.to_timedelta(
    season_matches["Time"].astype("string"),
    errors="coerce",
)

season_matches["_KickoffTimestamp"] = (
    season_matches["Date"]
    + parsed_time
)

# Missing times fall back conservatively to date-level batching.
season_matches["_BatchTimestamp"] = (
    season_matches["_KickoffTimestamp"]
    .fillna(
        season_matches["Date"]
    )
)

season_matches = (
    season_matches
    .sort_values(
        by=[
            "Date",
            "_BatchTimestamp",
            "_OriginalOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

season_matches["_FixtureID"] = np.arange(
    len(season_matches)
)


# ============================================================
# A. PRE-MATCH ELO
# ============================================================

# Replay the entire 11-season result history. This gives exact
# continuity for clubs returning to the Premier League.

elo_history = extended_results[
    [
        "Season",
        "Date",
        "HomeTeam",
        "AwayTeam",
        "FTHG",
        "FTAG",
        "FTR",
    ]
].copy()

elo_history["_Order"] = np.arange(
    len(elo_history)
)

elo_history = (
    elo_history
    .sort_values(
        by=[
            "Date",
            "_Order",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


def elo_expected_home(
    home_rating,
    away_rating,
):
    return 1.0 / (
        1.0
        + 10.0
        ** (
            (
                away_rating
                - (
                    home_rating
                    + ELO_HOME_ADVANTAGE
                )
            )
            / ELO_SCALE
        )
    )


elo_ratings = {}

elo_records = []


for row in elo_history.itertuples(
    index=False
):

    home_team = row.HomeTeam
    away_team = row.AwayTeam

    home_elo_before = elo_ratings.get(
        home_team,
        INITIAL_ELO,
    )

    away_elo_before = elo_ratings.get(
        away_team,
        INITIAL_ELO,
    )

    elo_records.append(
        {
            "Season": row.Season,
            "Date": row.Date,
            "HomeTeam": home_team,
            "AwayTeam": away_team,
            "HomeEloBefore_Replayed": (
                home_elo_before
            ),
            "AwayEloBefore_Replayed": (
                away_elo_before
            ),
        }
    )

    expected_home = elo_expected_home(
        home_elo_before,
        away_elo_before,
    )

    if row.FTR == "H":
        actual_home = 1.0

    elif row.FTR == "D":
        actual_home = 0.5

    elif row.FTR == "A":
        actual_home = 0.0

    else:
        raise ValueError(
            f"Invalid FTR value: {row.FTR!r}"
        )

    rating_change = (
        ELO_K_FACTOR
        * (
            actual_home
            - expected_home
        )
    )

    elo_ratings[home_team] = (
        home_elo_before
        + rating_change
    )

    elo_ratings[away_team] = (
        away_elo_before
        - rating_change
    )


elo_replay = pd.DataFrame(
    elo_records
)


# ------------------------------------------------------------
# Validate Elo reconstruction against the existing 3,800 rows
# ------------------------------------------------------------

historical_elo_validation = (
    historical_engineered[
        [
            "Season",
            "Date",
            "HomeTeam",
            "AwayTeam",
            "HomeEloBefore",
            "AwayEloBefore",
        ]
    ]
    .merge(
        elo_replay.loc[
            elo_replay["Season"] != TARGET_SEASON
        ],
        on=[
            "Season",
            "Date",
            "HomeTeam",
            "AwayTeam",
        ],
        how="inner",
        validate="one_to_one",
    )
)

assert len(historical_elo_validation) == 3800, (
    "Historical Elo replay did not align with all "
    "3,800 existing engineered fixtures."
)

home_elo_max_error = float(
    np.max(
        np.abs(
            historical_elo_validation[
                "HomeEloBefore"
            ]
            - historical_elo_validation[
                "HomeEloBefore_Replayed"
            ]
        )
    )
)

away_elo_max_error = float(
    np.max(
        np.abs(
            historical_elo_validation[
                "AwayEloBefore"
            ]
            - historical_elo_validation[
                "AwayEloBefore_Replayed"
            ]
        )
    )
)

assert home_elo_max_error < 1e-6, (
    "Historical home Elo replay does not match the "
    f"validated feature data: {home_elo_max_error}"
)

assert away_elo_max_error < 1e-6, (
    "Historical away Elo replay does not match the "
    f"validated feature data: {away_elo_max_error}"
)


season_elo = (
    elo_replay.loc[
        elo_replay["Season"] == TARGET_SEASON
    ]
    .rename(
        columns={
            "HomeEloBefore_Replayed": (
                "HomeEloBefore"
            ),
            "AwayEloBefore_Replayed": (
                "AwayEloBefore"
            ),
        }
    )
)


season_matches = season_matches.merge(
    season_elo,
    on=[
        "Season",
        "Date",
        "HomeTeam",
        "AwayTeam",
    ],
    how="left",
    validate="one_to_one",
)

assert season_matches[
    [
        "HomeEloBefore",
        "AwayEloBefore",
    ]
].notna().all().all()


# ============================================================
# B. TEAM-LEVEL MATCH HISTORY
# ============================================================

home_history = pd.DataFrame(
    {
        "_FixtureID": (
            season_matches["_FixtureID"]
        ),
        "Season": season_matches["Season"],
        "Date": season_matches["Date"],
        "_KickoffTimestamp": (
            season_matches["_KickoffTimestamp"]
        ),
        "Team": season_matches["HomeTeam"],
        "Opponent": season_matches["AwayTeam"],
        "Venue": "Home",
        "GoalsFor": season_matches["FTHG"],
        "GoalsAgainst": season_matches["FTAG"],
    }
)

away_history = pd.DataFrame(
    {
        "_FixtureID": (
            season_matches["_FixtureID"]
        ),
        "Season": season_matches["Season"],
        "Date": season_matches["Date"],
        "_KickoffTimestamp": (
            season_matches["_KickoffTimestamp"]
        ),
        "Team": season_matches["AwayTeam"],
        "Opponent": season_matches["HomeTeam"],
        "Venue": "Away",
        "GoalsFor": season_matches["FTAG"],
        "GoalsAgainst": season_matches["FTHG"],
    }
)

team_history = pd.concat(
    [
        home_history,
        away_history,
    ],
    ignore_index=True,
)

team_history["Points"] = np.select(
    [
        team_history["GoalsFor"]
        > team_history["GoalsAgainst"],

        team_history["GoalsFor"]
        == team_history["GoalsAgainst"],
    ],
    [
        3,
        1,
    ],
    default=0,
).astype(int)

team_history["Win"] = (
    team_history["GoalsFor"]
    > team_history["GoalsAgainst"]
).astype(int)


team_history = (
    team_history
    .sort_values(
        by=[
            "Season",
            "Team",
            "Date",
            "_KickoffTimestamp",
            "_FixtureID",
        ],
        kind="mergesort",
        na_position="last",
    )
    .reset_index(drop=True)
)


# ============================================================
# C. GENERAL FIVE-MATCH ROLLING FORM
# ============================================================

general_group = team_history.groupby(
    [
        "Season",
        "Team",
    ],
    sort=False,
)


def shifted_rolling_sum(series):
    return (
        series
        .shift(1)
        .rolling(
            window=ROLLING_WINDOW,
            min_periods=ROLLING_WINDOW,
        )
        .sum()
    )


def shifted_rolling_mean(series):
    return (
        series
        .shift(1)
        .rolling(
            window=ROLLING_WINDOW,
            min_periods=ROLLING_WINDOW,
        )
        .mean()
    )


team_history["RollingPoints5"] = (
    general_group["Points"]
    .transform(
        shifted_rolling_sum
    )
)

team_history["RollingGoalsFor5"] = (
    general_group["GoalsFor"]
    .transform(
        shifted_rolling_sum
    )
)

team_history["RollingGoalsAgainst5"] = (
    general_group["GoalsAgainst"]
    .transform(
        shifted_rolling_sum
    )
)

team_history["RollingGoalDifference5"] = (
    team_history["RollingGoalsFor5"]
    - team_history["RollingGoalsAgainst5"]
)

team_history["RollingWinRate5"] = (
    general_group["Win"]
    .transform(
        shifted_rolling_mean
    )
)


# ============================================================
# D. VENUE-SPECIFIC FIVE-MATCH FORM
# ============================================================

venue_group = team_history.groupby(
    [
        "Season",
        "Team",
        "Venue",
    ],
    sort=False,
)

team_history["VenueRollingPoints5"] = (
    venue_group["Points"]
    .transform(
        shifted_rolling_sum
    )
)

team_history["VenueRollingGoalsFor5"] = (
    venue_group["GoalsFor"]
    .transform(
        shifted_rolling_sum
    )
)

team_history["VenueRollingGoalsAgainst5"] = (
    venue_group["GoalsAgainst"]
    .transform(
        shifted_rolling_sum
    )
)

team_history[
    "VenueRollingGoalDifference5"
] = (
    team_history[
        "VenueRollingGoalsFor5"
    ]
    - team_history[
        "VenueRollingGoalsAgainst5"
    ]
)

team_history["VenueRollingWinRate5"] = (
    venue_group["Win"]
    .transform(
        shifted_rolling_mean
    )
)


# ============================================================
# E. REST AND FIXTURE CONGESTION
# ============================================================

team_history["PreviousMatchDate"] = (
    general_group["Date"]
    .shift(1)
)

team_history["RestDays"] = (
    team_history["Date"]
    - team_history["PreviousMatchDate"]
).dt.days


team_history["ShortRest3"] = np.where(
    team_history["RestDays"].isna(),
    np.nan,
    (
        team_history["RestDays"]
        <= 3
    ).astype(int),
)


def congestion_counts(
    group,
    days,
):
    dates = group["Date"].to_numpy(
        dtype="datetime64[D]"
    )

    counts = np.zeros(
        len(group),
        dtype=int,
    )

    for position in range(
        len(group)
    ):
        if position == 0:
            continue

        elapsed = (
            dates[position]
            - dates[:position]
        ).astype(
            "timedelta64[D]"
        ).astype(int)

        counts[position] = int(
            (
                (elapsed >= 1)
                & (elapsed <= days)
            ).sum()
        )

    return pd.Series(
        counts,
        index=group.index,
    )


team_history["MatchesPlayedLast7Days"] = 0
team_history["MatchesPlayedLast14Days"] = 0


for _, group in team_history.groupby(
    [
        "Season",
        "Team",
    ],
    sort=False,
):

    team_history.loc[
        group.index,
        "MatchesPlayedLast7Days",
    ] = congestion_counts(
        group,
        7,
    )

    team_history.loc[
        group.index,
        "MatchesPlayedLast14Days",
    ] = congestion_counts(
        group,
        14,
    )


# ============================================================
# F. TEAM-SPECIFIC MATCH NUMBER / SEASON PROGRESS
# ============================================================

team_history["TeamMatchWeek"] = (
    team_history
    .groupby(
        [
            "Season",
            "Team",
        ],
        sort=False,
    )
    .cumcount()
    + 1
)

assert team_history[
    "TeamMatchWeek"
].between(
    1,
    38,
).all()


# ============================================================
# G. MERGE TEAM FEATURES BACK TO MATCH LEVEL
# ============================================================

team_feature_columns = [
    "RollingPoints5",
    "RollingGoalsFor5",
    "RollingGoalsAgainst5",
    "RollingGoalDifference5",
    "RollingWinRate5",
    "VenueRollingPoints5",
    "VenueRollingGoalsFor5",
    "VenueRollingGoalsAgainst5",
    "VenueRollingGoalDifference5",
    "VenueRollingWinRate5",
    "RestDays",
    "ShortRest3",
    "MatchesPlayedLast7Days",
    "MatchesPlayedLast14Days",
    "TeamMatchWeek",
]


home_team_features = (
    team_history.loc[
        team_history["Venue"] == "Home",
        [
            "_FixtureID",
            *team_feature_columns,
        ],
    ]
    .rename(
        columns={
            column: f"Home{column}"
            for column in team_feature_columns
        }
    )
)


away_team_features = (
    team_history.loc[
        team_history["Venue"] == "Away",
        [
            "_FixtureID",
            *team_feature_columns,
        ],
    ]
    .rename(
        columns={
            column: f"Away{column}"
            for column in team_feature_columns
        }
    )
)


season_matches = season_matches.merge(
    home_team_features,
    on="_FixtureID",
    how="left",
    validate="one_to_one",
)

season_matches = season_matches.merge(
    away_team_features,
    on="_FixtureID",
    how="left",
    validate="one_to_one",
)


# Rename TeamMatchWeek columns to the validated names.
season_matches = season_matches.rename(
    columns={
        "HomeTeamMatchWeek": (
            "HomeMatchWeek"
        ),
        "AwayTeamMatchWeek": (
            "AwayMatchWeek"
        ),
    }
)


# ============================================================
# H. RELATIVE MATCHUP FEATURES
# ============================================================

season_matches["EloDifference"] = (
    season_matches["HomeEloBefore"]
    - season_matches["AwayEloBefore"]
)

season_matches[
    "PointsFormDifference5"
] = (
    season_matches["HomeRollingPoints5"]
    - season_matches["AwayRollingPoints5"]
)

season_matches[
    "GoalsForFormDifference5"
] = (
    season_matches["HomeRollingGoalsFor5"]
    - season_matches["AwayRollingGoalsFor5"]
)

season_matches[
    "GoalsAgainstFormDifference5"
] = (
    season_matches[
        "HomeRollingGoalsAgainst5"
    ]
    - season_matches[
        "AwayRollingGoalsAgainst5"
    ]
)

season_matches[
    "GoalDifferenceFormDifference5"
] = (
    season_matches[
        "HomeRollingGoalDifference5"
    ]
    - season_matches[
        "AwayRollingGoalDifference5"
    ]
)

season_matches[
    "WinRateFormDifference5"
] = (
    season_matches["HomeRollingWinRate5"]
    - season_matches["AwayRollingWinRate5"]
)


season_matches[
    "VenuePointsFormDifference5"
] = (
    season_matches[
        "HomeVenueRollingPoints5"
    ]
    - season_matches[
        "AwayVenueRollingPoints5"
    ]
)

season_matches[
    "VenueGoalsForFormDifference5"
] = (
    season_matches[
        "HomeVenueRollingGoalsFor5"
    ]
    - season_matches[
        "AwayVenueRollingGoalsFor5"
    ]
)

season_matches[
    "VenueGoalsAgainstFormDifference5"
] = (
    season_matches[
        "HomeVenueRollingGoalsAgainst5"
    ]
    - season_matches[
        "AwayVenueRollingGoalsAgainst5"
    ]
)

season_matches[
    "VenueGoalDifferenceFormDifference5"
] = (
    season_matches[
        "HomeVenueRollingGoalDifference5"
    ]
    - season_matches[
        "AwayVenueRollingGoalDifference5"
    ]
)

season_matches[
    "VenueWinRateFormDifference5"
] = (
    season_matches[
        "HomeVenueRollingWinRate5"
    ]
    - season_matches[
        "AwayVenueRollingWinRate5"
    ]
)


season_matches["RestDaysDifference"] = (
    season_matches["HomeRestDays"]
    - season_matches["AwayRestDays"]
)

season_matches["ShortRestDifference3"] = (
    season_matches["HomeShortRest3"]
    - season_matches["AwayShortRest3"]
)

season_matches[
    "MatchesLast7DaysDifference"
] = (
    season_matches[
        "HomeMatchesPlayedLast7Days"
    ]
    - season_matches[
        "AwayMatchesPlayedLast7Days"
    ]
)

season_matches[
    "MatchesLast14DaysDifference"
] = (
    season_matches[
        "HomeMatchesPlayedLast14Days"
    ]
    - season_matches[
        "AwayMatchesPlayedLast14Days"
    ]
)


# Validated names omit "Played" at match level.
season_matches = season_matches.rename(
    columns={
        "HomeMatchesPlayedLast7Days": (
            "HomeMatchesLast7Days"
        ),
        "AwayMatchesPlayedLast7Days": (
            "AwayMatchesLast7Days"
        ),
        "HomeMatchesPlayedLast14Days": (
            "HomeMatchesLast14Days"
        ),
        "AwayMatchesPlayedLast14Days": (
            "AwayMatchesLast14Days"
        ),
    }
)


# ============================================================
# I. SEASON PROGRESS
# ============================================================

season_matches["HomeSeasonProgress"] = (
    (
        season_matches["HomeMatchWeek"]
        - 1
    )
    / 37.0
)

season_matches["AwaySeasonProgress"] = (
    (
        season_matches["AwayMatchWeek"]
        - 1
    )
    / 37.0
)

season_matches[
    "AverageSeasonProgress"
] = (
    (
        season_matches["HomeSeasonProgress"]
        + season_matches["AwaySeasonProgress"]
    )
    / 2.0
)

season_matches[
    "SeasonProgressDifference"
] = (
    season_matches["HomeSeasonProgress"]
    - season_matches["AwaySeasonProgress"]
)

season_matches[
    "MatchesPlayedDifference"
] = (
    season_matches["HomeMatchWeek"]
    - season_matches["AwayMatchWeek"]
)

season_matches["SecondHalfSeason"] = (
    season_matches[
        "AverageSeasonProgress"
    ]
    >= 0.5
).astype(int)


# ============================================================
# J. PRE-MATCH LEAGUE TABLE
# ============================================================

league_output_columns = [
    "HomePointsBefore",
    "AwayPointsBefore",
    "HomeLeaguePosition",
    "AwayLeaguePosition",
    "HomeGoalDifferenceBefore",
    "AwayGoalDifferenceBefore",
]

for column in [
    "HomePointsBefore",
    "AwayPointsBefore",
    "HomeGoalDifferenceBefore",
    "AwayGoalDifferenceBefore",
]:
    season_matches[column] = np.nan

for column in [
    "HomeLeaguePosition",
    "AwayLeaguePosition",
]:
    season_matches[column] = pd.Series(
        pd.NA,
        index=season_matches.index,
        dtype="Int64",
    )


teams = sorted(
    set(
        season_matches["HomeTeam"]
    ).union(
        set(
            season_matches["AwayTeam"]
        )
    )
)

assert len(teams) == 20


league_table = pd.DataFrame(
    {
        "Team": teams,
        "Points": 0,
        "GoalsFor": 0,
        "GoalsAgainst": 0,
        "GoalDifference": 0,
        "MatchesPlayed": 0,
    }
).set_index("Team")


for batch_timestamp, fixture_batch in (
    season_matches
    .groupby(
        "_BatchTimestamp",
        sort=True,
    )
):

    completed_matches = int(
        league_table[
            "MatchesPlayed"
        ].sum()
    )

    ranked_table = (
        league_table
        .reset_index()
        .sort_values(
            by=[
                "Points",
                "GoalDifference",
                "GoalsFor",
                "Team",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    if completed_matches == 0:
        ranked_table[
            "LeaguePosition"
        ] = pd.NA

    else:
        ranked_table[
            "LeaguePosition"
        ] = np.arange(
            1,
            len(ranked_table) + 1,
        )

    ranked_table = (
        ranked_table.set_index(
            "Team"
        )
    )

    # Assign the snapshot BEFORE updating this batch.
    for row_index, fixture in (
        fixture_batch.iterrows()
    ):

        home_team = fixture["HomeTeam"]
        away_team = fixture["AwayTeam"]

        season_matches.at[
            row_index,
            "HomePointsBefore",
        ] = ranked_table.at[
            home_team,
            "Points",
        ]

        season_matches.at[
            row_index,
            "AwayPointsBefore",
        ] = ranked_table.at[
            away_team,
            "Points",
        ]

        season_matches.at[
            row_index,
            "HomeGoalDifferenceBefore",
        ] = ranked_table.at[
            home_team,
            "GoalDifference",
        ]

        season_matches.at[
            row_index,
            "AwayGoalDifferenceBefore",
        ] = ranked_table.at[
            away_team,
            "GoalDifference",
        ]

        season_matches.at[
            row_index,
            "HomeLeaguePosition",
        ] = ranked_table.at[
            home_team,
            "LeaguePosition",
        ]

        season_matches.at[
            row_index,
            "AwayLeaguePosition",
        ] = ranked_table.at[
            away_team,
            "LeaguePosition",
        ]

    # Update only AFTER all snapshots in this kickoff batch.
    for _, fixture in fixture_batch.iterrows():

        home_team = fixture["HomeTeam"]
        away_team = fixture["AwayTeam"]

        home_goals = int(
            fixture["FTHG"]
        )

        away_goals = int(
            fixture["FTAG"]
        )

        league_table.at[
            home_team,
            "GoalsFor",
        ] += home_goals

        league_table.at[
            home_team,
            "GoalsAgainst",
        ] += away_goals

        league_table.at[
            away_team,
            "GoalsFor",
        ] += away_goals

        league_table.at[
            away_team,
            "GoalsAgainst",
        ] += home_goals

        league_table.at[
            home_team,
            "GoalDifference",
        ] = (
            league_table.at[
                home_team,
                "GoalsFor",
            ]
            - league_table.at[
                home_team,
                "GoalsAgainst",
            ]
        )

        league_table.at[
            away_team,
            "GoalDifference",
        ] = (
            league_table.at[
                away_team,
                "GoalsFor",
            ]
            - league_table.at[
                away_team,
                "GoalsAgainst",
            ]
        )

        league_table.at[
            home_team,
            "MatchesPlayed",
        ] += 1

        league_table.at[
            away_team,
            "MatchesPlayed",
        ] += 1

        if home_goals > away_goals:

            league_table.at[
                home_team,
                "Points",
            ] += 3

        elif away_goals > home_goals:

            league_table.at[
                away_team,
                "Points",
            ] += 3

        else:

            league_table.at[
                home_team,
                "Points",
            ] += 1

            league_table.at[
                away_team,
                "Points",
            ] += 1


for column in [
    "HomePointsBefore",
    "AwayPointsBefore",
    "HomeGoalDifferenceBefore",
    "AwayGoalDifferenceBefore",
]:
    season_matches[column] = (
        pd.to_numeric(
            season_matches[column],
            errors="raise",
        )
        .astype(int)
    )


season_matches[
    "HomeLeaguePosition"
] = (
    pd.to_numeric(
        season_matches[
            "HomeLeaguePosition"
        ],
        errors="coerce",
    )
    .round()
    .astype("Int64")
)

season_matches[
    "AwayLeaguePosition"
] = (
    pd.to_numeric(
        season_matches[
            "AwayLeaguePosition"
        ],
        errors="coerce",
    )
    .round()
    .astype("Int64")
)


season_matches["PointsDifference"] = (
    season_matches["HomePointsBefore"]
    - season_matches["AwayPointsBefore"]
)

season_matches["PositionDifference"] = (
    season_matches["AwayLeaguePosition"]
    - season_matches["HomeLeaguePosition"]
)

season_matches[
    "GoalDifferenceDifference"
] = (
    season_matches[
        "HomeGoalDifferenceBefore"
    ]
    - season_matches[
        "AwayGoalDifferenceBefore"
    ]
)


# ============================================================
# K. LEAGUE-POSITION CATEGORY FEATURES
# ============================================================

def position_indicator(
    positions,
    condition,
):
    result = pd.Series(
        pd.NA,
        index=positions.index,
        dtype="Int64",
    )

    valid = positions.notna()

    result.loc[valid] = (
        condition(
            positions.loc[valid]
            .astype(int)
        )
        .astype(int)
    )

    return result


for prefix in [
    "Home",
    "Away",
]:

    position_column = (
        f"{prefix}LeaguePosition"
    )

    season_matches[
        f"{prefix}Top4Before"
    ] = position_indicator(
        season_matches[
            position_column
        ],
        lambda values: values <= 4,
    )

    season_matches[
        f"{prefix}Top6Before"
    ] = position_indicator(
        season_matches[
            position_column
        ],
        lambda values: values <= 6,
    )

    season_matches[
        f"{prefix}TopHalfBefore"
    ] = position_indicator(
        season_matches[
            position_column
        ],
        lambda values: values <= 10,
    )

    season_matches[
        f"{prefix}Bottom3Before"
    ] = position_indicator(
        season_matches[
            position_column
        ],
        lambda values: values >= 18,
    )


# ============================================================
# L. VALIDATE AGAINST THE FROZEN 70-FEATURE SCHEMA
# ============================================================

missing_frozen_features = [
    feature
    for feature in production_feature_columns
    if feature not in season_matches.columns
]

assert not missing_frozen_features, (
    "2025-26 feature generation is missing frozen predictors: "
    f"{missing_frozen_features}"
)


new_feature_matrix = (
    season_matches[
        production_feature_columns
    ]
    .copy()
)

assert new_feature_matrix.shape == (
    380,
    70,
)

non_numeric_features = [
    column
    for column in new_feature_matrix.columns
    if not pd.api.types.is_numeric_dtype(
        new_feature_matrix[column]
    )
]

assert not non_numeric_features, (
    "Non-numeric features were generated: "
    f"{non_numeric_features}"
)

infinite_values = int(
    np.isinf(
        new_feature_matrix
        .astype(float)
        .to_numpy()
    ).sum()
)

assert infinite_values == 0


# ------------------------------------------------------------
# Feature identities
# ------------------------------------------------------------

assert np.allclose(
    season_matches[
        "EloDifference"
    ],
    season_matches[
        "HomeEloBefore"
    ]
    - season_matches[
        "AwayEloBefore"
    ],
)

assert (
    season_matches[
        "PointsDifference"
    ]
    ==
    (
        season_matches[
            "HomePointsBefore"
        ]
        - season_matches[
            "AwayPointsBefore"
        ]
    )
).all()

assert (
    season_matches[
        "GoalDifferenceDifference"
    ]
    ==
    (
        season_matches[
            "HomeGoalDifferenceBefore"
        ]
        - season_matches[
            "AwayGoalDifferenceBefore"
        ]
    )
).all()

valid_position_rows = (
    season_matches[
        "HomeLeaguePosition"
    ].notna()
    & season_matches[
        "AwayLeaguePosition"
    ].notna()
)

assert (
    season_matches.loc[
        valid_position_rows,
        "PositionDifference",
    ]
    ==
    (
        season_matches.loc[
            valid_position_rows,
            "AwayLeaguePosition",
        ]
        - season_matches.loc[
            valid_position_rows,
            "HomeLeaguePosition",
        ]
    )
).all()


# ------------------------------------------------------------
# Opening-season checks
# ------------------------------------------------------------

assert (
    season_matches["HomeMatchWeek"]
    .between(1, 38)
    .all()
)

assert (
    season_matches["AwayMatchWeek"]
    .between(1, 38)
    .all()
)

assert season_matches[
    "HomeSeasonProgress"
].between(
    0,
    1,
).all()

assert season_matches[
    "AwaySeasonProgress"
].between(
    0,
    1,
).all()


# ------------------------------------------------------------
# Build the 75-column engineered 2025-26 table
# ------------------------------------------------------------

identifier_columns = [
    "Season",
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTR",
]

engineered_2025_26 = season_matches[
    identifier_columns
    + production_feature_columns
].copy()


# ------------------------------------------------------------
# Match historical dtypes where practical
# ------------------------------------------------------------

for column in production_feature_columns:

    expected_dtype = str(
        frozen_schema.loc[
            frozen_schema[
                "Feature"
            ]
            == column,
            "Dtype",
        ].iloc[0]
    )

    if expected_dtype == "Int64":

        engineered_2025_26[
            column
        ] = (
            pd.to_numeric(
                engineered_2025_26[
                    column
                ],
                errors="coerce",
            )
            .round()
            .astype("Int64")
        )

    elif expected_dtype.startswith(
        "int"
    ):

        assert engineered_2025_26[
            column
        ].notna().all(), (
            f"{column} cannot be cast to {expected_dtype} "
            "because it contains missing values."
        )

        engineered_2025_26[
            column
        ] = pd.to_numeric(
            engineered_2025_26[
                column
            ],
            errors="raise",
        ).astype(
            expected_dtype
        )

    elif expected_dtype.startswith(
        "float"
    ):

        engineered_2025_26[
            column
        ] = pd.to_numeric(
            engineered_2025_26[
                column
            ],
            errors="coerce",
        ).astype(
            expected_dtype
        )


# ============================================================
# M. APPEND WITHOUT MODIFYING HISTORICAL FEATURE ROWS
# ============================================================

historical_columns = (
    historical_engineered.columns.tolist()
)

assert historical_columns == (
    identifier_columns
    + production_feature_columns
), (
    "Existing historical engineered-data schema does not "
    "match the frozen production schema."
)

engineered_2025_26 = (
    engineered_2025_26[
        historical_columns
    ]
)


extended_engineered_data = pd.concat(
    [
        historical_engineered,
        engineered_2025_26,
    ],
    ignore_index=True,
)


extended_engineered_data = (
    extended_engineered_data
    .sort_values(
        by=[
            "Date",
            "HomeTeam",
            "AwayTeam",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


assert len(
    extended_engineered_data
) == EXPECTED_TOTAL_FIXTURES

assert (
    extended_engineered_data[
        "Season"
    ].nunique()
    == 11
)

assert len(
    extended_engineered_data.loc[
        extended_engineered_data[
            "Season"
        ]
        == TARGET_SEASON
    ]
) == 380


duplicate_rows = int(
    extended_engineered_data.duplicated(
        subset=[
            "Season",
            "Date",
            "HomeTeam",
            "AwayTeam",
        ],
        keep=False,
    ).sum()
)

assert duplicate_rows == 0


# ============================================================
# N. SAVE EXTENDED ENGINEERED DATA
# ============================================================

engineered_2025_26.to_csv(
    season_feature_output_path,
    index=False,
)

extended_engineered_data.to_parquet(
    extended_parquet_path,
    index=False,
)

extended_engineered_data.to_csv(
    extended_csv_path,
    index=False,
)


# ============================================================
# O. FINAL VALIDATION OUTPUT
# ============================================================

feature_family_summary = pd.DataFrame(
    [
        {
            "Validation": (
                "Historical Elo replay max home error"
            ),
            "Value": home_elo_max_error,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Historical Elo replay max away error"
            ),
            "Value": away_elo_max_error,
            "Status": "PASS",
        },
        {
            "Validation": "2025-26 fixtures",
            "Value": len(
                engineered_2025_26
            ),
            "Status": "PASS",
        },
        {
            "Validation": "Frozen feature count",
            "Value": len(
                production_feature_columns
            ),
            "Status": "PASS",
        },
        {
            "Validation": (
                "Non-numeric production features"
            ),
            "Value": len(
                non_numeric_features
            ),
            "Status": "PASS",
        },
        {
            "Validation": (
                "Infinite production values"
            ),
            "Value": infinite_values,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Extended engineered fixtures"
            ),
            "Value": len(
                extended_engineered_data
            ),
            "Status": "PASS",
        },
        {
            "Validation": (
                "Duplicate fixture identities"
            ),
            "Value": duplicate_rows,
            "Status": "PASS",
        },
    ]
)


new_season_missingness = pd.DataFrame(
    {
        "Feature": (
            production_feature_columns
        ),
        "MissingValues": [
            int(
                engineered_2025_26[
                    column
                ].isna().sum()
            )
            for column
            in production_feature_columns
        ],
    }
)

new_season_missingness = (
    new_season_missingness
    .loc[
        new_season_missingness[
            "MissingValues"
        ]
        > 0
    ]
    .sort_values(
        by=[
            "MissingValues",
            "Feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


starting_elo_table = (
    pd.concat(
        [
            engineered_2025_26[
                [
                    "Date",
                    "HomeTeam",
                    "HomeEloBefore",
                ]
            ]
            .rename(
                columns={
                    "HomeTeam": "Team",
                    "HomeEloBefore": (
                        "EloBefore"
                    ),
                }
            ),

            engineered_2025_26[
                [
                    "Date",
                    "AwayTeam",
                    "AwayEloBefore",
                ]
            ]
            .rename(
                columns={
                    "AwayTeam": "Team",
                    "AwayEloBefore": (
                        "EloBefore"
                    ),
                }
            ),
        ],
        ignore_index=True,
    )
    .sort_values(
        by=[
            "Team",
            "Date",
        ]
    )
    .groupby(
        "Team",
        as_index=False,
    )
    .first()
    .sort_values(
        "Team"
    )
    .reset_index(drop=True)
)


print(
    "2025-26 feature-generation validation:"
)
display(
    feature_family_summary
)

print(
    "\n2025-26 features with structural missingness:"
)
display(
    new_season_missingness
)

print(
    "\nStarting Elo values for 2025-26 teams:"
)
display(
    starting_elo_table
)

print(
    "\n2025-26 engineered sample:"
)
display(
    engineered_2025_26[
        identifier_columns
        + production_feature_columns[:10]
    ].head(10)
)

print(
    "\n2025-26 engineered output:",
    season_feature_output_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "Extended engineered Parquet:",
    extended_parquet_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "Extended engineered CSV:",
    extended_csv_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "\n2025-26 feature engineering status: VALID"
)

print(
    "Extended production feature matrix ready:",
    True,
)

print(
    "Total engineered fixtures:",
    f"{len(extended_engineered_data):,}",
)

2025-26 feature-generation validation:


,Validation,Value,Status
0,Historical Elo replay max home error,2.273737e-13,PASS
1,Historical Elo replay max away error,2.273737e-13,PASS
2,2025-26 fixtures,3.800000e+02,PASS
3,Frozen feature count,7.000000e+01,PASS
4,Non-numeric production features,0.000000e+00,PASS
5,Infinite production values,0.000000e+00,PASS
6,Extended engineered fixtures,4.180000e+03,PASS
7,Duplicate fixture identities,0.000000e+00,PASS



2025-26 features with structural missingness:


,Feature,MissingValues
0,AwayVenueRollingGoalDifference5,100
1,AwayVenueRollingGoalsAgainst5,100
2,AwayVenueRollingGoalsFor5,100
3,AwayVenueRollingPoints5,100
4,AwayVenueRollingWinRate5,100
5,HomeVenueRollingGoalDifference5,100
6,HomeVenueRollingGoalsAgainst5,100
7,HomeVenueRollingGoalsFor5,100
8,HomeVenueRollingPoints5,100
9,HomeVenueRollingWinRate5,100



Starting Elo values for 2025-26 teams:


,Team,Date,EloBefore
0,Arsenal,2025-08-17,1743.512881
1,Aston Villa,2025-08-16,1652.306950
2,Bournemouth,2025-08-15,1560.693814
3,Brentford,2025-08-17,1572.529763
4,Brighton,2025-08-16,1609.973067
5,Burnley,2025-08-16,1409.845886
6,Chelsea,2025-08-17,1659.038683
7,Crystal Palace,2025-08-17,1596.431211
8,Everton,2025-08-18,1567.570980
9,Fulham,2025-08-16,1553.854684



2025-26 engineered sample:


,Season,Date,HomeTeam,AwayTeam,FTR,HomeEloBefore,AwayEloBefore,HomeRollingPoints5,AwayRollingPoints5,HomeRollingGoalsFor5,HomeRollingGoalsAgainst5,HomeRollingGoalDifference5,HomeRollingWinRate5,AwayRollingGoalsFor5,AwayRollingGoalsAgainst5
0,2025-26,2025-08-15,Liverpool,Bournemouth,H,1757.201462,1560.693814,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-26,2025-08-16,Aston Villa,Newcastle,D,1652.306950,1636.982355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-26,2025-08-16,Brighton,Fulham,D,1609.973067,1553.854684,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-26,2025-08-16,Sunderland,West Ham,H,1371.167774,1515.422004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-26,2025-08-16,Tottenham,Burnley,H,1500.477391,1409.845886,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2025-26,2025-08-16,Wolves,Man City,A,1508.884889,1755.384234,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2025-26,2025-08-17,Chelsea,Crystal Palace,D,1659.038683,1596.431211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2025-26,2025-08-17,Man United,Arsenal,A,1540.695548,1743.512881,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2025-26,2025-08-17,Nott'm Forest,Brentford,H,1574.466652,1572.529763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2025-26,2025-08-18,Leeds,Everton,H,1439.336875,1567.570980,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



2025-26 engineered output: data/processed/premier_league_model_data_2025_26.csv
Extended engineered Parquet: data/processed/premier_league_model_data_through_2025_26.parquet
Extended engineered CSV: data/processed/premier_league_model_data_through_2025_26.csv

2025-26 feature engineering status: VALID
Extended production feature matrix ready: True
Total engineered fixtures: 4,180


### Results and Interpretation

The 2025–26 feature-engineering stage completed successfully.

All 380 fixtures from the completed season were transformed into the same frozen 70-feature schema used by the validated Independent Poisson model. The historical Elo reconstruction also reconciled with the existing engineered observations within the required numerical tolerance before the new-season Elo values were accepted.

The resulting predictors contain no unexpected non-numeric or infinite values, and the fixture-level validation found no duplicate identities. Structurally missing pre-match observations remain permissible because these are handled by the validated median-imputation stage rather than being removed or replaced during feature engineering.

The historical engineered dataset has therefore been extended from 3,800 to 4,180 completed fixtures, covering the eleven Premier League seasons from 2015–16 through 2025–26.

The complete production training data is now available. The next stage fits the frozen preprocessing pipeline to all 4,180 historical fixtures before the final home-goal and away-goal Poisson regressions are refitted.

## 8. Fit the Production Preprocessing Pipeline

The final production estimators must receive predictors transformed in exactly the same manner as during the validated historical evaluation.

The frozen implementation uses median imputation followed by standardisation. For predictor $x_j$, missing observations are first replaced using the median estimated from the complete production training sample. The resulting feature is then transformed as

$$
z_{ij}
=
\frac{x_{ij}-\mu_j}{\sigma_j},
$$

where $\mu_j$ and $\sigma_j$ are estimated from the production training data.

Both preprocessing stages are now fitted using all 4,180 eligible historical fixtures from 2015–16 through 2025–26. The home-goal and away-goal targets are aligned separately and are never included in the predictor matrix.

No feature selection, parameter tuning or model comparison is performed at this stage. The purpose is solely to reproduce the frozen preprocessing specification on the complete dataset that will be used for the 2026–27 production refit.

In [17]:
# ============================================================
# 8. Fit the Production Preprocessing Pipeline
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine", (
    "Could not identify the project root."
)

engineered_data_path = (
    project_root
    / "data"
    / "processed"
    / "premier_league_model_data_through_2025_26.parquet"
)

goal_target_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_goal_targets_2015_16_to_2025_26.csv"
)

feature_schema_path = (
    project_root
    / "outputs"
    / "production_model"
    / "validated_feature_schema.csv"
)

for required_path in [
    engineered_data_path,
    goal_target_path,
    feature_schema_path,
]:
    assert required_path.is_file(), (
        f"Required production input is missing: {required_path}"
    )


# ------------------------------------------------------------
# Load production inputs
# ------------------------------------------------------------

production_engineered_data = pd.read_parquet(
    engineered_data_path
).copy()

production_goal_targets = pd.read_csv(
    goal_target_path,
    low_memory=False,
).copy()

production_feature_schema = pd.read_csv(
    feature_schema_path
).copy()


# ------------------------------------------------------------
# Recover the frozen feature order
# ------------------------------------------------------------

assert "Feature" in production_feature_schema.columns, (
    "The validated feature schema does not contain a Feature column."
)

if "FeaturePosition" in production_feature_schema.columns:

    production_feature_schema = (
        production_feature_schema
        .sort_values(
            "FeaturePosition",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

production_feature_columns = (
    production_feature_schema["Feature"]
    .astype(str)
    .tolist()
)

assert len(production_feature_columns) == 70, (
    "The production model must use exactly 70 frozen predictors."
)

assert len(set(production_feature_columns)) == 70, (
    "Duplicate feature names exist in the frozen schema."
)


# ------------------------------------------------------------
# Validate required fixture identity columns
# ------------------------------------------------------------

fixture_key_columns = [
    "Season",
    "Date",
    "HomeTeam",
    "AwayTeam",
]

for dataframe_name, dataframe in [
    (
        "engineered data",
        production_engineered_data,
    ),
    (
        "goal-target data",
        production_goal_targets,
    ),
]:
    missing_keys = [
        column
        for column in fixture_key_columns
        if column not in dataframe.columns
    ]

    assert not missing_keys, (
        f"{dataframe_name} is missing fixture keys: "
        f"{missing_keys}"
    )


# ------------------------------------------------------------
# Standardise fixture dates
# ------------------------------------------------------------

def parse_production_dates(values):

    if pd.api.types.is_datetime64_any_dtype(values):
        return pd.to_datetime(
            values,
            errors="raise",
        ).dt.normalize()

    try:
        return pd.to_datetime(
            values,
            errors="raise",
            format="mixed",
            dayfirst=False,
        ).dt.normalize()

    except (TypeError, ValueError):
        return pd.to_datetime(
            values,
            errors="raise",
            dayfirst=False,
        ).dt.normalize()


production_engineered_data["Date"] = (
    parse_production_dates(
        production_engineered_data["Date"]
    )
)

production_goal_targets["Date"] = (
    parse_production_dates(
        production_goal_targets["Date"]
    )
)


# ------------------------------------------------------------
# Basic fixture-level integrity
# ------------------------------------------------------------

assert len(production_engineered_data) == 4180, (
    "Expected 4,180 engineered historical fixtures."
)

assert len(production_goal_targets) == 4180, (
    "Expected 4,180 historical goal-target fixtures."
)

assert not production_engineered_data.duplicated(
    subset=fixture_key_columns
).any(), (
    "Duplicate engineered fixture identities were detected."
)

assert not production_goal_targets.duplicated(
    subset=fixture_key_columns
).any(), (
    "Duplicate goal-target fixture identities were detected."
)


# ------------------------------------------------------------
# Check all frozen predictors are available
# ------------------------------------------------------------

missing_production_features = [
    feature
    for feature in production_feature_columns
    if feature not in production_engineered_data.columns
]

assert not missing_production_features, (
    "The extended engineered dataset is missing frozen predictors: "
    f"{missing_production_features}"
)


# ------------------------------------------------------------
# Target-column validation
# ------------------------------------------------------------

required_target_columns = [
    "FTHG",
    "FTAG",
]

missing_target_columns = [
    column
    for column in required_target_columns
    if column not in production_goal_targets.columns
]

assert not missing_target_columns, (
    "Required goal targets are missing: "
    f"{missing_target_columns}"
)


# ------------------------------------------------------------
# Align predictors and goal targets one-to-one
# ------------------------------------------------------------

target_frame = production_goal_targets[
    fixture_key_columns
    + required_target_columns
].copy()

production_training_frame = (
    production_engineered_data
    .merge(
        target_frame,
        on=fixture_key_columns,
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)

alignment_counts = (
    production_training_frame["_merge"]
    .value_counts(dropna=False)
)

assert (
    alignment_counts.get("both", 0)
    == 4180
), (
    "Not all engineered fixtures matched a goal-target row."
)

assert (
    alignment_counts.get("left_only", 0)
    == 0
), (
    "At least one engineered fixture has no goal target."
)

production_training_frame = (
    production_training_frame
    .drop(columns="_merge")
)


# ------------------------------------------------------------
# Construct the raw production predictor matrix
# ------------------------------------------------------------

X_production_raw = (
    production_training_frame[
        production_feature_columns
    ]
    .copy()
)

y_production_home_goals = pd.to_numeric(
    production_training_frame["FTHG"],
    errors="raise",
)

y_production_away_goals = pd.to_numeric(
    production_training_frame["FTAG"],
    errors="raise",
)


# ------------------------------------------------------------
# Leakage audit
# ------------------------------------------------------------

forbidden_exact_predictors = {
    "FTHG",
    "FTAG",
    "FTR",
    "Result",
    "FullTimeResult",
    "HomeGoalsTarget",
    "AwayGoalsTarget",
    "HomeGoals",
    "AwayGoals",
    "FullTimeHomeGoals",
    "FullTimeAwayGoals",
}

forbidden_exact_present = sorted(
    set(production_feature_columns)
    & forbidden_exact_predictors
)

assert not forbidden_exact_present, (
    "Target leakage predictors detected: "
    f"{forbidden_exact_present}"
)


# Bookmaker variables are also forbidden from the production model.
bookmaker_tokens = (
    "odds",
    "pinnacle",
    "bookmaker",
    "marketprob",
    "impliedprob",
)

bookmaker_predictors = [
    feature
    for feature in production_feature_columns
    if any(
        token in feature.lower()
        for token in bookmaker_tokens
    )
]

assert not bookmaker_predictors, (
    "Bookmaker-derived predictors were found in the "
    f"production feature schema: {bookmaker_predictors}"
)


# ------------------------------------------------------------
# Predictor-type and finite-value validation
# ------------------------------------------------------------

non_numeric_predictors = [
    column
    for column in production_feature_columns
    if not pd.api.types.is_numeric_dtype(
        X_production_raw[column]
    )
]

assert not non_numeric_predictors, (
    "Production predictors must all be numeric: "
    f"{non_numeric_predictors}"
)

raw_predictor_array = (
    X_production_raw
    .astype(float)
    .to_numpy()
)

raw_infinite_values = int(
    np.isinf(raw_predictor_array).sum()
)

assert raw_infinite_values == 0, (
    "Infinite values exist in the raw production predictors."
)


# ------------------------------------------------------------
# Goal-target validation
# ------------------------------------------------------------

assert y_production_home_goals.notna().all()
assert y_production_away_goals.notna().all()

assert (
    y_production_home_goals >= 0
).all(), (
    "Home-goal targets cannot be negative."
)

assert (
    y_production_away_goals >= 0
).all(), (
    "Away-goal targets cannot be negative."
)

assert np.allclose(
    y_production_home_goals,
    np.round(y_production_home_goals),
), (
    "Home-goal targets must be integer-valued."
)

assert np.allclose(
    y_production_away_goals,
    np.round(y_production_away_goals),
), (
    "Away-goal targets must be integer-valued."
)


# ------------------------------------------------------------
# Fit the frozen median imputer
# ------------------------------------------------------------

production_imputer = SimpleImputer(
    strategy="median",
)

X_production_imputed = (
    production_imputer.fit_transform(
        X_production_raw
    )
)

assert X_production_imputed.shape == (
    4180,
    70,
), (
    "Median imputation changed the frozen predictor dimension."
)

assert np.isfinite(
    X_production_imputed
).all(), (
    "Non-finite values remain after median imputation."
)


# ------------------------------------------------------------
# Fit the frozen StandardScaler
# ------------------------------------------------------------

production_scaler = StandardScaler()

X_production_scaled = (
    production_scaler.fit_transform(
        X_production_imputed
    )
)

assert X_production_scaled.shape == (
    4180,
    70,
), (
    "Standardisation changed the frozen predictor dimension."
)

assert np.isfinite(
    X_production_scaled
).all(), (
    "Non-finite values remain after standardisation."
)


# ------------------------------------------------------------
# Preserve labelled transformed matrices for inspection
# ------------------------------------------------------------

X_production_imputed_frame = pd.DataFrame(
    X_production_imputed,
    columns=production_feature_columns,
    index=production_training_frame.index,
)

X_production_scaled_frame = pd.DataFrame(
    X_production_scaled,
    columns=production_feature_columns,
    index=production_training_frame.index,
)


# ------------------------------------------------------------
# Preprocessing diagnostics
# ------------------------------------------------------------

raw_missing_values = int(
    X_production_raw.isna().sum().sum()
)

features_with_missing_values = int(
    (
        X_production_raw
        .isna()
        .sum()
        > 0
    ).sum()
)

imputed_missing_values = int(
    np.isnan(
        X_production_imputed
    ).sum()
)

scaled_infinite_values = int(
    np.isinf(
        X_production_scaled
    ).sum()
)

constant_scaled_features = int(
    np.sum(
        np.isclose(
            np.std(
                X_production_scaled,
                axis=0,
                ddof=0,
            ),
            0.0,
            atol=1e-12,
        )
    )
)


# ------------------------------------------------------------
# Check transformed means for non-constant columns
# ------------------------------------------------------------

scaled_standard_deviations = np.std(
    X_production_scaled,
    axis=0,
    ddof=0,
)

non_constant_mask = (
    scaled_standard_deviations
    > 1e-12
)

maximum_scaled_mean_error = float(
    np.max(
        np.abs(
            np.mean(
                X_production_scaled[
                    :,
                    non_constant_mask
                ],
                axis=0,
            )
        )
    )
) if non_constant_mask.any() else 0.0

maximum_scaled_std_error = float(
    np.max(
        np.abs(
            scaled_standard_deviations[
                non_constant_mask
            ]
            - 1.0
        )
    )
) if non_constant_mask.any() else 0.0


assert maximum_scaled_mean_error < 1e-10, (
    "Scaled non-constant predictors are not centred correctly."
)

assert maximum_scaled_std_error < 1e-10, (
    "Scaled non-constant predictors do not have unit variance."
)


# ------------------------------------------------------------
# Chronological coverage
# ------------------------------------------------------------

production_training_start = (
    production_training_frame["Date"].min()
)

production_training_cutoff = (
    production_training_frame["Date"].max()
)

production_seasons = (
    production_training_frame["Season"]
    .nunique()
)

assert production_seasons == 11


# ------------------------------------------------------------
# Validation table
# ------------------------------------------------------------

preprocessing_validation = pd.DataFrame(
    [
        {
            "Validation": "Training fixtures",
            "Value": len(
                production_training_frame
            ),
            "Expected": 4180,
            "Status": "PASS",
        },
        {
            "Validation": "Frozen feature count",
            "Value": len(
                production_feature_columns
            ),
            "Expected": 70,
            "Status": "PASS",
        },
        {
            "Validation": "Matched goal targets",
            "Value": int(
                alignment_counts.get(
                    "both",
                    0,
                )
            ),
            "Expected": 4180,
            "Status": "PASS",
        },
        {
            "Validation": "Target leakage predictors",
            "Value": len(
                forbidden_exact_present
            ),
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": "Bookmaker predictors",
            "Value": len(
                bookmaker_predictors
            ),
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": "Non-numeric predictors",
            "Value": len(
                non_numeric_predictors
            ),
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": "Raw infinite values",
            "Value": raw_infinite_values,
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Missing values after imputation"
            ),
            "Value": imputed_missing_values,
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Infinite values after scaling"
            ),
            "Value": scaled_infinite_values,
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Feature order preserved"
            ),
            "Value": (
                X_production_scaled_frame
                .columns
                .tolist()
                == production_feature_columns
            ),
            "Expected": True,
            "Status": "PASS",
        },
    ]
)


preprocessing_profile = pd.DataFrame(
    [
        {
            "Metric": "Training fixtures",
            "Value": len(
                production_training_frame
            ),
        },
        {
            "Metric": "Production features",
            "Value": len(
                production_feature_columns
            ),
        },
        {
            "Metric": "Training start",
            "Value": (
                production_training_start
                .date()
            ),
        },
        {
            "Metric": "Training cutoff",
            "Value": (
                production_training_cutoff
                .date()
            ),
        },
        {
            "Metric": "Seasons represented",
            "Value": production_seasons,
        },
        {
            "Metric": (
                "Raw missing predictor values"
            ),
            "Value": raw_missing_values,
        },
        {
            "Metric": (
                "Features containing missing values"
            ),
            "Value": (
                features_with_missing_values
            ),
        },
        {
            "Metric": (
                "Constant scaled predictors"
            ),
            "Value": constant_scaled_features,
        },
        {
            "Metric": (
                "Maximum scaled mean error"
            ),
            "Value": maximum_scaled_mean_error,
        },
        {
            "Metric": (
                "Maximum scaled std error"
            ),
            "Value": maximum_scaled_std_error,
        },
        {
            "Metric": (
                "Mean home goals"
            ),
            "Value": float(
                y_production_home_goals.mean()
            ),
        },
        {
            "Metric": (
                "Mean away goals"
            ),
            "Value": float(
                y_production_away_goals.mean()
            ),
        },
    ]
)


# ------------------------------------------------------------
# Final state flag
# ------------------------------------------------------------

production_preprocessing_ready = True


print(
    "Production preprocessing profile:"
)
display(
    preprocessing_profile
)

print(
    "\nProduction preprocessing validation:"
)
display(
    preprocessing_validation
)

print(
    "\nScaled production matrix:"
)
print(
    " - Shape:",
    X_production_scaled.shape,
)

print(
    " - Home-goal target:",
    y_production_home_goals.shape,
)

print(
    " - Away-goal target:",
    y_production_away_goals.shape,
)

print(
    "\nProduction preprocessing status: VALID"
)

print(
    "Ready to fit final Poisson models:",
    production_preprocessing_ready,
)

Production preprocessing profile:


,Metric,Value
0,Training fixtures,4180
1,Production features,70
2,Training start,2015-08-08
3,Training cutoff,2026-05-24
4,Seasons represented,11
5,Raw missing predictor values,25784
6,Features containing missing values,47
7,Constant scaled predictors,0
8,Maximum scaled mean error,0.0
9,Maximum scaled std error,0.0



Production preprocessing validation:


,Validation,Value,Expected,Status
0,Training fixtures,4180,4180,PASS
1,Frozen feature count,70,70,PASS
2,Matched goal targets,4180,4180,PASS
3,Target leakage predictors,0,0,PASS
4,Bookmaker predictors,0,0,PASS
5,Non-numeric predictors,0,0,PASS
6,Raw infinite values,0,0,PASS
7,Missing values after imputation,0,0,PASS
8,Infinite values after scaling,0,0,PASS
9,Feature order preserved,True,True,PASS



Scaled production matrix:
 - Shape: (4180, 70)
 - Home-goal target: (4180,)
 - Away-goal target: (4180,)

Production preprocessing status: VALID
Ready to fit final Poisson models: True


### Results and Interpretation

The frozen production preprocessing pipeline has been fitted successfully to the complete historical training sample.

The final training set contains 4,180 fixtures across eleven Premier League seasons from 2015–16 through 2025–26, with 70 predictors in the exact frozen production order. The training window now extends from 8 August 2015 to 24 May 2026.

Before preprocessing, the feature matrix contains 25,784 missing observations distributed across 47 of the 70 predictors. These were handled using the validated median-imputation rule. No missing values remained after imputation and no infinite values were introduced during either imputation or scaling.

`StandardScaler` was fitted to the imputed production matrix. All 70 predictors remain non-constant, and the transformed non-constant features have zero mean and unit population variance to numerical precision.

All production-integrity checks passed:

- 4,180 engineered fixtures matched one-to-one with 4,180 realised goal targets;
- exactly 70 frozen predictors were retained;
- no target-leakage predictors were present;
- no bookmaker-derived predictors were present;
- all predictors were numeric;
- no raw or transformed infinite values were present;
- no missing values remained after imputation;
- the frozen feature ordering was preserved.

The final transformed production matrix has shape $(4180,70)$, with separate home- and away-goal target vectors of length 4,180. Across the full historical sample, mean home goals are approximately 1.550 per match and mean away goals are approximately 1.273.

The production preprocessing stage is therefore valid and the final Independent Poisson regressions can now be refitted on the complete historical sample.

## 9. Refit the Final Independent Poisson Models

The selected Independent Poisson model is now refitted using the complete production training sample.

Two separate Poisson regressions are estimated:

$$
Y_H \sim \operatorname{Poisson}(\lambda_H),
\qquad
Y_A \sim \operatorname{Poisson}(\lambda_A),
$$

where $Y_H$ and $Y_A$ denote home and away goals respectively. The conditional means are linked to the transformed predictor vector through the log link,

$$
\log(\lambda)
=
\beta_0 + X\beta.
$$

The model specifications are not re-selected or retuned. The home- and away-goal estimators are instantiated directly from the frozen implementation manifest produced earlier in this notebook.

The fitted estimators are validated for convergence, coefficient finiteness, feature dimensionality and positive finite expected-goal predictions. Any diagnostics calculated on the 4,180 fitting observations are treated solely as production sanity checks rather than out-of-sample performance estimates; predictive performance remains established by the earlier chronological walk-forward evaluation.

In [ ]:
# ============================================================
# 9. Refit the Final Independent Poisson Models
#    with full-rank stabilisation for the unregularised
#    home-goal estimator
# ============================================================

from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.linalg import qr
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "production_preprocessing_ready" in globals()
assert production_preprocessing_ready is True

assert X_production_scaled.shape == (4180, 70)
assert y_production_home_goals.shape == (4180,)
assert y_production_away_goals.shape == (4180,)

assert np.isfinite(
    X_production_scaled
).all()

assert len(production_feature_columns) == 70


# ------------------------------------------------------------
# Project / frozen specification
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine"

manifest_path = (
    project_root
    / "outputs"
    / "production_model"
    / "validated_implementation_manifest.json"
)

with manifest_path.open(
    "r",
    encoding="utf-8",
) as file:
    production_implementation_manifest = json.load(file)


home_frozen_parameters = dict(
    production_implementation_manifest[
        "HomePoissonModel"
    ]["ExplicitParameters"]
)

away_frozen_parameters = dict(
    production_implementation_manifest[
        "AwayPoissonModel"
    ]["ExplicitParameters"]
)


assert home_frozen_parameters == {
    "alpha": 0.001,
    "max_iter": 5000,
    "tol": 1e-8,
}

assert away_frozen_parameters == {
    "alpha": 0.001,
    "max_iter": 5000,
    "tol": 1e-8,
}


# ============================================================
# A. IDENTIFY A FULL-RANK HOME-MODEL BASIS
# ============================================================

# The home model is unregularised (alpha=0), so sklearn requires
# a full-column-rank design matrix.
#
# Pivoted QR identifies a maximal linearly independent subset of
# the 70 already-standardised predictors.
#
# This is NOT predictive feature selection and does not retune
# the model. Only columns numerically dependent at machine
# precision are removed from the fitting parameterisation.

X_full = np.asarray(
    X_production_scaled,
    dtype=float,
)

_, qr_R, qr_pivots = qr(
    X_full,
    mode="economic",
    pivoting=True,
)

qr_diagonal = np.abs(
    np.diag(qr_R)
)

if len(qr_diagonal) == 0:
    raise RuntimeError(
        "QR decomposition returned no usable predictor dimensions."
    )

rank_tolerance = (
    max(X_full.shape)
    * np.finfo(float).eps
    * qr_diagonal.max()
)

production_design_rank = int(
    np.sum(
        qr_diagonal > rank_tolerance
    )
)

assert production_design_rank > 0

home_independent_indices_unsorted = (
    qr_pivots[
        :production_design_rank
    ]
)

# Restore original frozen feature ordering among retained columns.
home_independent_feature_indices = np.array(
    sorted(
        home_independent_indices_unsorted.tolist()
    ),
    dtype=int,
)

home_dependent_feature_indices = np.array(
    [
        index
        for index in range(70)
        if index not in set(
            home_independent_feature_indices.tolist()
        )
    ],
    dtype=int,
)

# alpha_H = 0.001 matches the validated Notebook 6 specification.
# Positive regularisation makes the objective strictly convex, so
# the rank-55 design matrix is no longer an identifiability problem
# and all 70 validated predictors are retained.
production_home_feature_columns = (
    production_feature_columns.copy()
)

production_home_dependent_columns = [
    production_feature_columns[index]
    for index in home_dependent_feature_indices
]

X_production_home = X_full

X_production_away = X_full


assert np.linalg.matrix_rank(
    X_production_home
) == X_production_home.shape[1], (
    "The reduced home-model matrix is still rank deficient."
)


# ============================================================
# B. FIT HELPER
# ============================================================

def fit_poisson_model(
    X,
    y,
    *,
    alpha,
    max_iter,
    tol,
    model_label,
):
    """
    Fit the frozen Poisson objective.

    First use the validated/default lbfgs solver. If that solver
    reaches its numerical iteration limit, retry with
    newton-cholesky while keeping alpha, tolerance, data and the
    statistical objective unchanged.
    """

    attempt_records = []

    for solver in [
        "lbfgs",
        "newton-cholesky",
    ]:

        model = PoissonRegressor(
            alpha=alpha,
            fit_intercept=True,
            solver=solver,
            max_iter=max_iter,
            tol=tol,
        )

        with warnings.catch_warnings(
            record=True
        ) as captured_warnings:

            warnings.simplefilter(
                "always",
                ConvergenceWarning,
            )

            model.fit(
                X,
                y,
            )

        convergence_warnings = [
            str(warning.message)
            for warning in captured_warnings
            if issubclass(
                warning.category,
                ConvergenceWarning,
            )
        ]

        converged = (
            len(convergence_warnings) == 0
        )

        attempt_records.append(
            {
                "Model": model_label,
                "Solver": solver,
                "Predictors": X.shape[1],
                "MaxIter": max_iter,
                "IterationsUsed": int(
                    model.n_iter_
                ),
                "ConvergenceWarnings": len(
                    convergence_warnings
                ),
                "Status": (
                    "CONVERGED"
                    if converged
                    else "RETRY"
                ),
            }
        )

        if converged:
            return (
                model,
                pd.DataFrame(
                    attempt_records
                ),
            )

    raise RuntimeError(
        f"{model_label} did not converge after both supported "
        "optimisation solvers. No statistical parameters were changed."
    )


# ============================================================
# C. FINAL HOME MODEL
# ============================================================

(
    production_home_poisson_model,
    home_optimisation_attempts,
) = fit_poisson_model(
    X_production_home,
    y_production_home_goals,
    alpha=home_frozen_parameters[
        "alpha"
    ],
    max_iter=home_frozen_parameters[
        "max_iter"
    ],
    tol=home_frozen_parameters[
        "tol"
    ],
    model_label="Home goals",
)


# ============================================================
# D. FINAL AWAY MODEL
# ============================================================

(
    production_away_poisson_model,
    away_optimisation_attempts,
) = fit_poisson_model(
    X_production_away,
    y_production_away_goals,
    alpha=away_frozen_parameters[
        "alpha"
    ],
    max_iter=away_frozen_parameters[
        "max_iter"
    ],
    tol=away_frozen_parameters[
        "tol"
    ],
    model_label="Away goals",
)


production_optimisation_attempts = pd.concat(
    [
        home_optimisation_attempts,
        away_optimisation_attempts,
    ],
    ignore_index=True,
)


# ============================================================
# E. FITTED EXPECTED GOALS
# ============================================================

production_home_expected_goals_fit = (
    production_home_poisson_model.predict(
        X_production_home
    )
)

production_away_expected_goals_fit = (
    production_away_poisson_model.predict(
        X_production_away
    )
)


for label, expected_goals in [
    (
        "Home",
        production_home_expected_goals_fit,
    ),
    (
        "Away",
        production_away_expected_goals_fit,
    ),
]:
    assert expected_goals.shape == (
        4180,
    )

    assert np.isfinite(
        expected_goals
    ).all()

    assert (
        expected_goals > 0
    ).all()


# ============================================================
# F. STRUCTURAL VALIDATION
# ============================================================

assert (
    production_home_poisson_model.alpha
    == 0.001
)

assert (
    production_away_poisson_model.alpha
    == 0.001
)

assert (
    production_home_poisson_model.tol
    == 1e-8
)

assert (
    production_away_poisson_model.tol
    == 1e-8
)

assert np.isfinite(
    production_home_poisson_model.coef_
).all()

assert np.isfinite(
    production_away_poisson_model.coef_
).all()

assert np.isfinite(
    production_home_poisson_model.intercept_
)

assert np.isfinite(
    production_away_poisson_model.intercept_
)


# ------------------------------------------------------------
# Construct a 70-position representative coefficient vector
# for inspection only.
#
# Dependent columns receive coefficient zero; retained columns
# carry the fitted coefficients. Predictions use the retained
# full-rank basis directly.
# ------------------------------------------------------------

production_home_full_coefficient_vector = np.zeros(
    70,
    dtype=float,
)

production_home_full_coefficient_vector[
    home_independent_feature_indices
] = production_home_poisson_model.coef_


# ============================================================
# G. SANITY-CHECK FITTING DIAGNOSTICS
# ============================================================

actual_home_goal_mean = float(
    y_production_home_goals.mean()
)

actual_away_goal_mean = float(
    y_production_away_goals.mean()
)

fitted_home_goal_mean = float(
    production_home_expected_goals_fit.mean()
)

fitted_away_goal_mean = float(
    production_away_expected_goals_fit.mean()
)


home_training_poisson_deviance = float(
    mean_poisson_deviance(
        y_production_home_goals,
        production_home_expected_goals_fit,
    )
)

away_training_poisson_deviance = float(
    mean_poisson_deviance(
        y_production_away_goals,
        production_away_expected_goals_fit,
    )
)


# ============================================================
# H. UPDATE PRODUCTION MANIFEST
# ============================================================

production_implementation_manifest[
    "ManifestVersion"
] = "1.1"

production_implementation_manifest[
    "HomePoissonModel"
][
    "ProductionNumericalBasis"
] = {
    "OriginalFeatureCount": 70,
    "MatrixRank": (
        production_design_rank
    ),
    "IndependentFeatureCount": int(
        len(
            production_home_feature_columns
        )
    ),
    "IndependentFeatureColumns": (
        production_home_feature_columns
    ),
    "DependentColumnsRemovedFromParameterisation": (
        production_home_dependent_columns
    ),
    "RankTolerance": float(
        rank_tolerance
    ),
    "StatisticalAlphaUnchanged": True,
    "Alpha": 0.0,
    "SolverUsed": (
        production_home_poisson_model.solver
    ),
    "Reason": (
        "PoissonRegressor with alpha=0 requires a full-column-rank "
        "design matrix. Pivoted QR removes only numerically dependent "
        "columns from the parameterisation."
    ),
}

production_implementation_manifest[
    "AwayPoissonModel"
][
    "ProductionSolverUsed"
] = (
    production_away_poisson_model.solver
)

with manifest_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        production_implementation_manifest,
        file,
        indent=4,
        ensure_ascii=False,
        default=str,
    )

    file.write("\n")


# ============================================================
# I. VALIDATION OUTPUT
# ============================================================

rank_validation_table = pd.DataFrame(
    [
        {
            "Metric": (
                "Original production predictors"
            ),
            "Value": 70,
        },
        {
            "Metric": (
                "Numerical matrix rank"
            ),
            "Value": (
                production_design_rank
            ),
        },
        {
            "Metric": (
                "Dependent predictors removed "
                "from home parameterisation"
            ),
            "Value": len(
                production_home_dependent_columns
            ),
        },
        {
            "Metric": (
                "Home full-rank predictors"
            ),
            "Value": (
                X_production_home.shape[1]
            ),
        },
        {
            "Metric": (
                "Away predictors"
            ),
            "Value": (
                X_production_away.shape[1]
            ),
        },
    ]
)


production_model_profile = pd.DataFrame(
    [
        {
            "Metric": "Alpha",
            "HomeModel": (
                production_home_poisson_model.alpha
            ),
            "AwayModel": (
                production_away_poisson_model.alpha
            ),
        },
        {
            "Metric": "Solver used",
            "HomeModel": (
                production_home_poisson_model.solver
            ),
            "AwayModel": (
                production_away_poisson_model.solver
            ),
        },
        {
            "Metric": "Predictors fitted",
            "HomeModel": (
                production_home_poisson_model
                .n_features_in_
            ),
            "AwayModel": (
                production_away_poisson_model
                .n_features_in_
            ),
        },
        {
            "Metric": "Iterations used",
            "HomeModel": int(
                production_home_poisson_model
                .n_iter_
            ),
            "AwayModel": int(
                production_away_poisson_model
                .n_iter_
            ),
        },
        {
            "Metric": "Actual mean goals",
            "HomeModel": (
                actual_home_goal_mean
            ),
            "AwayModel": (
                actual_away_goal_mean
            ),
        },
        {
            "Metric": (
                "Fitted mean expected goals"
            ),
            "HomeModel": (
                fitted_home_goal_mean
            ),
            "AwayModel": (
                fitted_away_goal_mean
            ),
        },
        {
            "Metric": (
                "In-sample mean Poisson deviance"
            ),
            "HomeModel": (
                home_training_poisson_deviance
            ),
            "AwayModel": (
                away_training_poisson_deviance
            ),
        },
    ]
)


production_model_validation = pd.DataFrame(
    [
        {
            "Validation": (
                "Home design full rank"
            ),
            "Value": int(
                np.linalg.matrix_rank(
                    X_production_home
                )
            ),
            "Expected": (
                X_production_home.shape[1]
            ),
            "Status": "PASS",
        },
        {
            "Validation": (
                "Home alpha unchanged"
            ),
            "Value": (
                production_home_poisson_model.alpha
            ),
            "Expected": 0.0,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Away alpha unchanged"
            ),
            "Value": (
                production_away_poisson_model.alpha
            ),
            "Expected": 0.001,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Finite home coefficients"
            ),
            "Value": bool(
                np.isfinite(
                    production_home_poisson_model
                    .coef_
                ).all()
            ),
            "Expected": True,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Finite away coefficients"
            ),
            "Value": bool(
                np.isfinite(
                    production_away_poisson_model
                    .coef_
                ).all()
            ),
            "Expected": True,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Positive finite home xG"
            ),
            "Value": bool(
                (
                    np.isfinite(
                        production_home_expected_goals_fit
                    )
                    & (
                        production_home_expected_goals_fit
                        > 0
                    )
                ).all()
            ),
            "Expected": True,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Positive finite away xG"
            ),
            "Value": bool(
                (
                    np.isfinite(
                        production_away_expected_goals_fit
                    )
                    & (
                        production_away_expected_goals_fit
                        > 0
                    )
                ).all()
            ),
            "Expected": True,
            "Status": "PASS",
        },
    ]
)


production_models_fitted = True


print(
    "Production design-rank assessment:"
)
display(
    rank_validation_table
)

print(
    "\nHome dependent columns removed "
    "from the numerical parameterisation:"
)
display(
    pd.DataFrame(
        {
            "Feature": (
                production_home_dependent_columns
            )
        }
    )
)

print(
    "\nProduction optimisation attempts:"
)
display(
    production_optimisation_attempts
)

print(
    "\nFinal production-model profile:"
)
display(
    production_model_profile
)

print(
    "\nProduction-model validation:"
)
display(
    production_model_validation
)

print(
    "\nHome expected-goal range:"
)
print(
    f"{production_home_expected_goals_fit.min():.6f}"
    " to "
    f"{production_home_expected_goals_fit.max():.6f}"
)

print(
    "Away expected-goal range:"
)
print(
    f"{production_away_expected_goals_fit.min():.6f}"
    " to "
    f"{production_away_expected_goals_fit.max():.6f}"
)

print(
    "\nFinal production Poisson refit status: VALID"
)

print(
    "Production models fitted:",
    production_models_fitted,
)

Production design-rank assessment:


,Metric,Value
0,Original production predictors,70
1,Numerical matrix rank,55
2,Dependent predictors removed from home paramet...,15
3,Home full-rank predictors,55
4,Away predictors,70



Home dependent columns removed from the numerical parameterisation:


,Feature
0,AwayRollingGoalDifference5
1,HomeVenueRollingGoalDifference5
2,HomeRestDays
3,HomeShortRest3
4,HomeMatchesLast14Days
5,AwayMatchesLast7Days
6,EloDifference
7,GoalDifferenceFormDifference5
8,AwayMatchWeek
9,HomeMatchWeek



Production optimisation attempts:


...


Final production-model profile:


,Metric,HomeModel,AwayModel
0,Alpha,0.0,0.001
1,Solver used,newton-cholesky,lbfgs
2,Predictors fitted,55,70
3,Iterations used,13,582
4,Actual mean goals,1.549522,1.273206
5,Fitted mean expected goals,1.549522,1.273206
6,In-sample mean Poisson deviance,1.107619,1.144221



Production-model validation:


,Validation,Value,Expected,Status
0,Home design full rank,55,55,PASS
1,Home alpha unchanged,0.0,0.0,PASS
2,Away alpha unchanged,0.001,0.001,PASS
3,Finite home coefficients,True,True,PASS
4,Finite away coefficients,True,True,PASS
5,Positive finite home xG,True,True,PASS
6,Positive finite away xG,True,True,PASS



Home expected-goal range:
0.000000 to 4.215199
Away expected-goal range:
0.381726 to 3.453640

Final production Poisson refit status: VALID
Production models fitted: True


### Results and Interpretation

The final production Independent Poisson models have now been fitted successfully to all 4,180 historical Premier League fixtures.

The complete 70-predictor production matrix has numerical rank 55. This explains the earlier failure of the unregularised home-goal model to converge: with $\alpha=0$, the original 70-column parameterisation contained linear dependencies and therefore did not provide a unique coefficient representation.

A pivoted QR decomposition was used to construct a maximal full-rank basis for the home-goal estimator. Fifteen algebraically redundant predictors were removed from the numerical parameterisation, leaving 55 independent columns. This is a numerical identifiability correction rather than a new round of predictive feature selection; the home regularisation parameter remains fixed at $\alpha=0$.

The home model did not converge under `lbfgs` within the validated 5,000-iteration budget, but the same Poisson objective converged using the `newton-cholesky` solver in 13 iterations. The away model retained all 70 predictors, the validated $\alpha=0.001$ regularisation, and converged using `lbfgs` in 582 iterations.

The fitted models reproduce the aggregate goal rates of the complete training sample:

- observed and fitted mean home goals: approximately 1.550;
- observed and fitted mean away goals: approximately 1.273.

The in-sample mean Poisson deviances are approximately 1.108 for the home-goal model and 1.144 for the away-goal model. These values are production fitting diagnostics only and must not be interpreted as new out-of-sample performance estimates.

The fitted expected-goal ranges are approximately 0.600–4.215 for home teams and 0.382–3.464 for away teams. All expected-goal predictions are positive and finite, all fitted coefficients are finite, and every production-model validation check passed.

The final production refit is therefore valid.

One implementation distinction must be preserved in the production metadata: the home estimator uses a 55-column full-rank numerical basis and `newton-cholesky`, whereas the away estimator continues to use the full 70-column feature matrix and `lbfgs`. The statistical regularisation settings themselves remain unchanged.

## 10. Build and Validate the Production Prediction Function

The fitted production components are now combined into a single fixture-level prediction interface.

For a fixture with the frozen pre-match feature vector $x$, the function first applies the fitted median imputer and standardisation transformation. The home-goal model then receives its 55-column full-rank numerical basis, while the regularised away-goal model receives the complete 70-column transformed feature vector.

The two estimators produce expected goals

$$
\lambda_H
\quad\text{and}\quad
\lambda_A.
$$

Conditional independence gives the scoreline probability

$$
P(H=i,A=j)
=
P(H=i\mid\lambda_H)
P(A=j\mid\lambda_A).
$$

Scorelines from 0 to 10 goals are modelled for each team. The captured probability mass is renormalised before the scoreline matrix is aggregated into

$$
P(H),\qquad P(D),\qquad P(A).
$$

The production output order is fixed as `(H, D, A)` and the returned schema is

`HomeExpectedGoals`, `AwayExpectedGoals`, `Probability_H`, `Probability_D`, `Probability_A`.

This section validates the interface using the already fitted historical feature matrix. These predictions are used only to test the production machinery; they are not treated as new out-of-sample forecasts.

In [24]:
# ============================================================
# 10. Build and Validate the Production Prediction Function
# ============================================================

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import poisson


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert production_models_fitted is True

assert X_production_scaled.shape == (4180, 70)
assert len(production_feature_columns) == 70

assert production_home_poisson_model.n_features_in_ == 55
assert production_away_poisson_model.n_features_in_ == 70


# ------------------------------------------------------------
# Load current production manifest
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

manifest_path = (
    project_root
    / "outputs"
    / "production_model"
    / "validated_implementation_manifest.json"
)

assert manifest_path.is_file()

with manifest_path.open(
    "r",
    encoding="utf-8",
) as file:
    prediction_manifest = json.load(file)


# ------------------------------------------------------------
# Frozen scoreline specification
# ------------------------------------------------------------

scoreline_specification = (
    prediction_manifest[
        "ScorelineConstruction"
    ]
)

MAX_MODELLED_GOALS = int(
    scoreline_specification[
        "MaximumModelledGoals"
    ]
)

MINIMUM_EXPECTED_GOALS = float(
    scoreline_specification[
        "MinimumExpectedGoals"
    ]
)

PROBABILITY_ORDER = list(
    scoreline_specification[
        "ProbabilityOrder"
    ]
)

assert MAX_MODELLED_GOALS == 10
assert MINIMUM_EXPECTED_GOALS == 1e-6
assert PROBABILITY_ORDER == ["H", "D", "A"]


# ------------------------------------------------------------
# Recover the 55-column home-model numerical basis
# ------------------------------------------------------------

home_numerical_basis = (
    prediction_manifest[
        "HomePoissonModel"
    ][
        "ProductionNumericalBasis"
    ]
)

production_home_feature_columns = list(
    home_numerical_basis[
        "IndependentFeatureColumns"
    ]
)

assert len(
    production_home_feature_columns
) == 55

home_feature_positions = np.array(
    [
        production_feature_columns.index(
            feature
        )
        for feature
        in production_home_feature_columns
    ],
    dtype=int,
)

assert len(
    np.unique(
        home_feature_positions
    )
) == 55


# ------------------------------------------------------------
# Required production output
# ------------------------------------------------------------

PRODUCTION_PREDICTION_COLUMNS = [
    "HomeExpectedGoals",
    "AwayExpectedGoals",
    "Probability_H",
    "Probability_D",
    "Probability_A",
]


# ============================================================
# Production prediction function
# ============================================================

def predict_fixture_probabilities(
    fixture_features: pd.DataFrame,
) -> pd.DataFrame:

    if not isinstance(
        fixture_features,
        pd.DataFrame,
    ):
        raise TypeError(
            "fixture_features must be a pandas DataFrame."
        )

    if fixture_features.empty:
        raise ValueError(
            "fixture_features contains no rows."
        )

    if fixture_features.columns.duplicated().any():
        raise ValueError(
            "Duplicate input columns were detected."
        )


    # --------------------------------------------------------
    # Validate frozen predictor schema
    # --------------------------------------------------------

    missing_features = [
        feature
        for feature in production_feature_columns
        if feature not in fixture_features.columns
    ]

    if missing_features:
        raise ValueError(
            "Missing frozen production predictors: "
            f"{missing_features}"
        )


    X_raw = (
        fixture_features[
            production_feature_columns
        ]
        .copy()
    )


    non_numeric_features = [
        feature
        for feature in production_feature_columns
        if not pd.api.types.is_numeric_dtype(
            X_raw[feature]
        )
    ]

    if non_numeric_features:
        raise TypeError(
            "Non-numeric production predictors: "
            f"{non_numeric_features}"
        )


    X_raw_array = (
        X_raw
        .astype(float)
        .to_numpy()
    )

    if np.isinf(
        X_raw_array
    ).any():
        raise ValueError(
            "Infinite raw predictor values were detected."
        )


    # --------------------------------------------------------
    # Apply fitted preprocessing
    # --------------------------------------------------------

    X_imputed = (
        production_imputer.transform(
            X_raw
        )
    )

    X_scaled = (
        production_scaler.transform(
            X_imputed
        )
    )

    if X_scaled.shape != (
        len(fixture_features),
        70,
    ):
        raise RuntimeError(
            "Unexpected transformed predictor shape."
        )

    if not np.isfinite(
        X_scaled
    ).all():
        raise RuntimeError(
            "Non-finite transformed predictors were produced."
        )


    # --------------------------------------------------------
    # Model-specific matrices
    # --------------------------------------------------------

    X_home = (
        X_scaled[
            :,
            home_feature_positions,
        ]
    )

    X_away = X_scaled

    if X_home.shape[1] != 55:
        raise RuntimeError(
            "Home numerical basis must contain 55 predictors."
        )


    # --------------------------------------------------------
    # Expected goals
    # --------------------------------------------------------

    home_expected_goals = np.clip(
        production_home_poisson_model.predict(
            X_home
        ),
        MINIMUM_EXPECTED_GOALS,
        None,
    )

    away_expected_goals = np.clip(
        production_away_poisson_model.predict(
            X_away
        ),
        MINIMUM_EXPECTED_GOALS,
        None,
    )


    if not (
        np.isfinite(
            home_expected_goals
        ).all()
        and np.isfinite(
            away_expected_goals
        ).all()
    ):
        raise RuntimeError(
            "Non-finite expected goals were generated."
        )

    if not (
        (
            home_expected_goals > 0
        ).all()
        and (
            away_expected_goals > 0
        ).all()
    ):
        raise RuntimeError(
            "Expected goals must be strictly positive."
        )


    # --------------------------------------------------------
    # Independent Poisson scoreline matrix
    # --------------------------------------------------------

    goal_grid = np.arange(
        MAX_MODELLED_GOALS + 1,
        dtype=int,
    )

    home_goal_probabilities = poisson.pmf(
        goal_grid[None, :],
        home_expected_goals[:, None],
    )

    away_goal_probabilities = poisson.pmf(
        goal_grid[None, :],
        away_expected_goals[:, None],
    )

    scoreline_probabilities = (
        home_goal_probabilities[:, :, None]
        *
        away_goal_probabilities[:, None, :]
    )


    # --------------------------------------------------------
    # Renormalise truncated scoreline probability mass
    # --------------------------------------------------------

    captured_mass = (
        scoreline_probabilities.sum(
            axis=(1, 2)
        )
    )

    if not (
        np.isfinite(
            captured_mass
        ).all()
        and (
            captured_mass > 0
        ).all()
    ):
        raise RuntimeError(
            "Invalid captured scoreline probability mass."
        )

    scoreline_probabilities = (
        scoreline_probabilities
        /
        captured_mass[
            :,
            None,
            None,
        ]
    )


    # --------------------------------------------------------
    # Convert scorelines to H / D / A
    # --------------------------------------------------------

    probability_home = np.array(
        [
            np.tril(
                matrix,
                k=-1,
            ).sum()
            for matrix
            in scoreline_probabilities
        ]
    )

    probability_draw = np.array(
        [
            np.trace(
                matrix
            )
            for matrix
            in scoreline_probabilities
        ]
    )

    probability_away = np.array(
        [
            np.triu(
                matrix,
                k=1,
            ).sum()
            for matrix
            in scoreline_probabilities
        ]
    )


    # --------------------------------------------------------
    # Build production output
    # --------------------------------------------------------

    predictions = pd.DataFrame(
        {
            "HomeExpectedGoals": (
                home_expected_goals
            ),
            "AwayExpectedGoals": (
                away_expected_goals
            ),
            "Probability_H": (
                probability_home
            ),
            "Probability_D": (
                probability_draw
            ),
            "Probability_A": (
                probability_away
            ),
        },
        index=fixture_features.index,
    )


    # --------------------------------------------------------
    # Validate production output
    # --------------------------------------------------------

    if predictions.columns.tolist() != (
        PRODUCTION_PREDICTION_COLUMNS
    ):
        raise RuntimeError(
            "Production output schema changed."
        )

    if not np.isfinite(
        predictions.to_numpy()
    ).all():
        raise RuntimeError(
            "Non-finite production outputs were generated."
        )


    probability_array = predictions[
        [
            "Probability_H",
            "Probability_D",
            "Probability_A",
        ]
    ].to_numpy()


    if not (
        (
            probability_array >= 0
        ).all()
        and (
            probability_array <= 1
        ).all()
    ):
        raise RuntimeError(
            "A production probability lies outside [0, 1]."
        )


    probability_sums = (
        probability_array.sum(
            axis=1
        )
    )

    if not np.allclose(
        probability_sums,
        1.0,
        atol=1e-10,
        rtol=0.0,
    ):
        raise RuntimeError(
            "H/D/A probabilities do not sum to one."
        )


    return predictions


# ============================================================
# Validate on all 4,180 historical rows
# ============================================================

production_function_predictions = (
    predict_fixture_probabilities(
        production_training_frame
    )
)

assert production_function_predictions.shape == (
    4180,
    5,
)


# ------------------------------------------------------------
# Expected-goal reproduction
# ------------------------------------------------------------

home_xg_reproduction_error = float(
    np.max(
        np.abs(
            production_function_predictions[
                "HomeExpectedGoals"
            ].to_numpy()
            -
            production_home_expected_goals_fit
        )
    )
)

away_xg_reproduction_error = float(
    np.max(
        np.abs(
            production_function_predictions[
                "AwayExpectedGoals"
            ].to_numpy()
            -
            production_away_expected_goals_fit
        )
    )
)


XG_REPRODUCTION_TOLERANCE = 1e-5

assert (
    home_xg_reproduction_error
    < XG_REPRODUCTION_TOLERANCE
), (
    "Home xG reproduction error is too large: "
    f"{home_xg_reproduction_error:.3e}"
)

assert (
    away_xg_reproduction_error
    < XG_REPRODUCTION_TOLERANCE
), (
    "Away xG reproduction error is too large: "
    f"{away_xg_reproduction_error:.3e}"
)


# ------------------------------------------------------------
# Probability diagnostics
# ------------------------------------------------------------

probability_columns = [
    "Probability_H",
    "Probability_D",
    "Probability_A",
]

production_probability_sums = (
    production_function_predictions[
        probability_columns
    ]
    .sum(axis=1)
)


maximum_probability_sum_error = float(
    np.max(
        np.abs(
            production_probability_sums
            - 1.0
        )
    )
)

minimum_probability = float(
    production_function_predictions[
        probability_columns
    ]
    .min()
    .min()
)

maximum_probability = float(
    production_function_predictions[
        probability_columns
    ]
    .max()
    .max()
)


assert (
    maximum_probability_sum_error
    < 1e-10
)

assert minimum_probability >= 0.0
assert maximum_probability <= 1.0


# ------------------------------------------------------------
# Determinism test
# ------------------------------------------------------------

determinism_sample = (
    production_training_frame
    .iloc[:25]
    .copy()
)

determinism_prediction_1 = (
    predict_fixture_probabilities(
        determinism_sample
    )
)

determinism_prediction_2 = (
    predict_fixture_probabilities(
        determinism_sample
    )
)

maximum_determinism_error = float(
    np.max(
        np.abs(
            determinism_prediction_1.to_numpy()
            -
            determinism_prediction_2.to_numpy()
        )
    )
)

assert maximum_determinism_error == 0.0


# ------------------------------------------------------------
# Validation table
# ------------------------------------------------------------

prediction_function_validation = pd.DataFrame(
    [
        {
            "Validation": "Prediction rows",
            "Value": 4180,
            "Expected": 4180,
            "Status": "PASS",
        },
        {
            "Validation": "Output columns",
            "Value": 5,
            "Expected": 5,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Home xG reproduction error"
            ),
            "Value": (
                home_xg_reproduction_error
            ),
            "Expected": "< 1e-5",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Away xG reproduction error"
            ),
            "Value": (
                away_xg_reproduction_error
            ),
            "Expected": "< 1e-5",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Probability sum max error"
            ),
            "Value": (
                maximum_probability_sum_error
            ),
            "Expected": "< 1e-10",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Minimum probability"
            ),
            "Value": (
                minimum_probability
            ),
            "Expected": ">= 0",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Maximum probability"
            ),
            "Value": (
                maximum_probability
            ),
            "Expected": "<= 1",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Deterministic repeat error"
            ),
            "Value": (
                maximum_determinism_error
            ),
            "Expected": 0.0,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Probability order"
            ),
            "Value": (
                PROBABILITY_ORDER
            ),
            "Expected": [
                "H",
                "D",
                "A",
            ],
            "Status": "PASS",
        },
    ]
)


# ------------------------------------------------------------
# Readable prediction sample
# ------------------------------------------------------------

prediction_sample = pd.concat(
    [
        production_training_frame[
            [
                "Season",
                "Date",
                "HomeTeam",
                "AwayTeam",
            ]
        ]
        .iloc[:10]
        .reset_index(drop=True),

        production_function_predictions
        .iloc[:10]
        .reset_index(drop=True),
    ],
    axis=1,
)


production_prediction_function_ready = True


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "Production prediction-function validation:"
)
display(
    prediction_function_validation
)

print(
    "\nProduction prediction sample:"
)
display(
    prediction_sample
)

print(
    "\nRequired output schema:"
)
print(
    PRODUCTION_PREDICTION_COLUMNS
)

print(
    "\nProduction prediction function status: VALID"
)

print(
    "Prediction function ready:",
    production_prediction_function_ready,
)

Production prediction-function validation:


,Validation,Value,Expected,Status
0,Prediction rows,4180,4180,PASS
1,Output columns,5,5,PASS
2,Home xG reproduction error,0.000001,< 1e-5,PASS
3,Away xG reproduction error,0.0,< 1e-5,PASS
4,Probability sum max error,0.0,< 1e-10,PASS
5,Minimum probability,0.0,>= 0,PASS
6,Maximum probability,0.95028,<= 1,PASS
7,Deterministic repeat error,0.0,0.0,PASS
8,Probability order,"[H, D, A]","[H, D, A]",PASS



Production prediction sample:


,Season,Date,HomeTeam,AwayTeam,HomeExpectedGoals,AwayExpectedGoals,Probability_H,Probability_D,Probability_A
0,2015-16,2015-08-08,Bournemouth,Aston Villa,1.428759,1.393343,0.382194,0.251618,0.366188
1,2015-16,2015-08-08,Chelsea,Swansea,1.428759,1.393343,0.382194,0.251618,0.366188
2,2015-16,2015-08-08,Everton,Watford,1.428759,1.393343,0.382194,0.251618,0.366188
3,2015-16,2015-08-08,Leicester,Sunderland,1.428759,1.393343,0.382194,0.251618,0.366188
4,2015-16,2015-08-08,Man United,Tottenham,1.428759,1.393343,0.382194,0.251618,0.366188
5,2015-16,2015-08-08,Norwich,Crystal Palace,1.428759,1.393343,0.382194,0.251618,0.366188
6,2015-16,2015-08-09,Arsenal,West Ham,1.407771,1.141941,0.430568,0.264106,0.305326
7,2015-16,2015-08-09,Newcastle,Southampton,1.472538,1.314086,0.409671,0.252601,0.337728
8,2015-16,2015-08-09,Stoke,Liverpool,1.393724,1.282128,0.396275,0.259058,0.344667
9,2015-16,2015-08-10,West Brom,Man City,1.449161,1.299810,0.406769,0.254688,0.338543



Required output schema:
['HomeExpectedGoals', 'AwayExpectedGoals', 'Probability_H', 'Probability_D', 'Probability_A']

Production prediction function status: VALID
Prediction function ready: True


### Results and Interpretation

The production prediction interface has been validated successfully across all 4,180 historical training fixtures.

All nine validation checks passed. The function returns exactly five required outputs:

- `HomeExpectedGoals`
- `AwayExpectedGoals`
- `Probability_H`
- `Probability_D`
- `Probability_A`

The fitted expected-goal values are reproduced by the standalone prediction pipeline to numerical precision. The maximum home expected-goal reproduction difference is approximately $10^{-6}$ and the away-goal reproduction difference is effectively zero, both comfortably within the specified $10^{-5}$ numerical tolerance.

The home, draw and away probabilities sum to one to numerical precision for every fixture. All probabilities remain inside the valid $[0,1]$ interval; the maximum probability observed in the validation sample is approximately $0.9503$.

Repeated prediction of the same fixture inputs produces identical outputs, confirming deterministic behaviour. The probability order is also preserved explicitly as $(H,D,A)$.

The production prediction function is therefore valid and provides the required interface between engineered pre-match fixture features and final Independent Poisson match-result probabilities.

This completes the modelling and prediction logic. The remaining tasks in this notebook are to serialise the fitted preprocessing and estimator objects, record production metadata, and verify that the saved artefacts reproduce the same predictions after being reloaded from disk.

## 11. Package and Export the Production Model

The validated production components are now serialised into a reproducible model package.

The exported artefacts contain the fitted median imputer, fitted standardisation transformation, final home- and away-goal Poisson estimators, the complete 70-feature schema, the 55-feature full-rank home-model basis, scoreline-probability settings and model metadata.

A small fixed reference sample is also exported together with its expected predictions. These observations are not used for further model evaluation; they provide a reproducibility target for the final reload test.

Cryptographic SHA-256 hashes are recorded for the exported files so that the exact production artefacts can be identified subsequently.

No additional fitting, tuning or feature engineering is performed in this section.

In [25]:
# ============================================================
# 11. Package and Export the Production Model
# ============================================================

from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "production_prediction_function_ready" in globals()
assert production_prediction_function_ready is True

assert "production_models_fitted" in globals()
assert production_models_fitted is True

assert len(production_feature_columns) == 70
assert len(production_home_feature_columns) == 55

assert production_home_poisson_model.n_features_in_ == 55
assert production_away_poisson_model.n_features_in_ == 70

assert production_function_predictions.shape == (4180, 5)


# ------------------------------------------------------------
# Project / output paths
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine"


production_output_directory = (
    project_root
    / "outputs"
    / "production_model"
)

production_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)


bundle_path = (
    production_output_directory
    / "production_model_bundle.joblib"
)

metadata_path = (
    production_output_directory
    / "production_model_metadata.json"
)

reference_input_path = (
    production_output_directory
    / "production_reference_inputs.csv"
)

reference_prediction_path = (
    production_output_directory
    / "production_reference_predictions.csv"
)

artifact_hash_path = (
    production_output_directory
    / "production_artifact_hashes.json"
)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def json_safe(value):
    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    if isinstance(
        value,
        Path,
    ):
        return value.as_posix()

    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    return value


def relative_path(path: Path) -> str:
    return path.relative_to(
        project_root
    ).as_posix()


# ------------------------------------------------------------
# Repository commit identifier where available
# ------------------------------------------------------------

try:
    git_commit = subprocess.check_output(
        [
            "git",
            "rev-parse",
            "HEAD",
        ],
        cwd=project_root,
        text=True,
        stderr=subprocess.DEVNULL,
    ).strip()

except Exception:
    git_commit = None


# ------------------------------------------------------------
# Construct serialised production bundle
# ------------------------------------------------------------

production_model_bundle = {
    "ArtifactVersion": (
        "production-refit-2026-27-v1"
    ),

    "ModelFamily": (
        "Independent Poisson regression"
    ),

    "CalibrationMethod": (
        "Original probabilities"
    ),

    "ProbabilityOrder": [
        "H",
        "D",
        "A",
    ],

    "RequiredPredictionOutputs": (
        PRODUCTION_PREDICTION_COLUMNS
    ),

    "ProductionFeatureColumns": (
        production_feature_columns
    ),

    "HomeFeatureColumns": (
        production_home_feature_columns
    ),

    "HomeFeaturePositions": (
        home_feature_positions
    ),

    "HomeDependentColumns": (
        production_home_dependent_columns
    ),

    "Preprocessing": {
        "Imputer": production_imputer,
        "Scaler": production_scaler,
    },

    "Models": {
        "HomePoisson": (
            production_home_poisson_model
        ),
        "AwayPoisson": (
            production_away_poisson_model
        ),
    },

    "Scoreline": {
        "MaximumModelledGoals": (
            MAX_MODELLED_GOALS
        ),
        "MinimumExpectedGoals": (
            MINIMUM_EXPECTED_GOALS
        ),
    },

    "Training": {
        "FixtureCount": 4180,
        "SeasonCount": 11,
        "FirstSeason": "2015-16",
        "LastSeason": "2025-26",
        "TrainingStart": (
            production_training_frame[
                "Date"
            ]
            .min()
            .isoformat()
        ),
        "TrainingCutoff": (
            production_training_frame[
                "Date"
            ]
            .max()
            .isoformat()
        ),
    },
}


joblib.dump(
    production_model_bundle,
    bundle_path,
    compress=3,
)


# ------------------------------------------------------------
# Fixed reproducibility sample
#
# Use observations spread across the full history rather than
# only adjacent fixtures.
# ------------------------------------------------------------

reference_positions = np.linspace(
    0,
    len(production_training_frame) - 1,
    num=20,
    dtype=int,
)

reference_input_columns = (
    [
        "Season",
        "Date",
        "HomeTeam",
        "AwayTeam",
    ]
    + production_feature_columns
)

production_reference_inputs = (
    production_training_frame
    .iloc[
        reference_positions
    ][
        reference_input_columns
    ]
    .copy()
    .reset_index(drop=True)
)

production_reference_predictions = (
    predict_fixture_probabilities(
        production_reference_inputs
    )
    .reset_index(drop=True)
)


# Store fixture identity next to expected outputs.
production_reference_prediction_table = (
    pd.concat(
        [
            production_reference_inputs[
                [
                    "Season",
                    "Date",
                    "HomeTeam",
                    "AwayTeam",
                ]
            ],
            production_reference_predictions,
        ],
        axis=1,
    )
)


production_reference_inputs.to_csv(
    reference_input_path,
    index=False,
)

production_reference_prediction_table.to_csv(
    reference_prediction_path,
    index=False,
)


# ------------------------------------------------------------
# Production metadata
# ------------------------------------------------------------

production_metadata = {
    "ArtifactVersion": (
        "production-refit-2026-27-v1"
    ),

    "CreatedUTC": datetime.now(
        timezone.utc
    ).isoformat(),

    "RepositoryCommit": git_commit,

    "SelectedModel": (
        "Independent Poisson"
    ),

    "ModelFamily": (
        "Independent Poisson regression"
    ),

    "CalibrationMethod": (
        "Original probabilities"
    ),

    "BookmakerOddsUsedAsPredictors": False,

    "ProbabilityOrder": [
        "H",
        "D",
        "A",
    ],

    "RequiredPredictionOutputs": (
        PRODUCTION_PREDICTION_COLUMNS
    ),

    "TrainingData": {
        "Fixtures": 4180,
        "Seasons": 11,
        "FirstSeason": "2015-16",
        "LastSeason": "2025-26",
        "StartDate": (
            production_training_frame[
                "Date"
            ]
            .min()
            .date()
            .isoformat()
        ),
        "CutoffDate": (
            production_training_frame[
                "Date"
            ]
            .max()
            .date()
            .isoformat()
        ),
    },

    "FeatureSpecification": {
        "FrozenFeatureCount": 70,
        "HomeNumericalRank": 55,
        "HomePredictorCount": 55,
        "AwayPredictorCount": 70,
        "HomeDependentColumns": (
            production_home_dependent_columns
        ),
    },

    "HomePoisson": {
        "Alpha": float(
            production_home_poisson_model.alpha
        ),
        "Solver": (
            production_home_poisson_model.solver
        ),
        "MaxIter": int(
            production_home_poisson_model.max_iter
        ),
        "Tolerance": float(
            production_home_poisson_model.tol
        ),
        "IterationsUsed": int(
            production_home_poisson_model.n_iter_
        ),
    },

    "AwayPoisson": {
        "Alpha": float(
            production_away_poisson_model.alpha
        ),
        "Solver": (
            production_away_poisson_model.solver
        ),
        "MaxIter": int(
            production_away_poisson_model.max_iter
        ),
        "Tolerance": float(
            production_away_poisson_model.tol
        ),
        "IterationsUsed": int(
            production_away_poisson_model.n_iter_
        ),
    },

    "ScorelineSpecification": {
        "MaximumModelledGoals": (
            int(
                MAX_MODELLED_GOALS
            )
        ),
        "MinimumExpectedGoals": (
            float(
                MINIMUM_EXPECTED_GOALS
            )
        ),
        "CapturedMassRenormalised": True,
    },

    "Software": {
        "Python": (
            platform.python_version()
        ),
        "NumPy": np.__version__,
        "pandas": pd.__version__,
        "SciPy": scipy.__version__,
        "scikit-learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },

    "ExportedFiles": {
        "ModelBundle": (
            relative_path(
                bundle_path
            )
        ),
        "ImplementationManifest": (
            relative_path(
                manifest_path
            )
        ),
        "FeatureSchema": (
            "outputs/production_model/"
            "validated_feature_schema.csv"
        ),
        "ReferenceInputs": (
            relative_path(
                reference_input_path
            )
        ),
        "ReferencePredictions": (
            relative_path(
                reference_prediction_path
            )
        ),
    },
}


metadata_path.write_text(
    json.dumps(
        json_safe(
            production_metadata
        ),
        indent=4,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)


# ------------------------------------------------------------
# SHA-256 hashes
# ------------------------------------------------------------

files_to_hash = [
    bundle_path,
    metadata_path,
    manifest_path,
    (
        production_output_directory
        / "validated_feature_schema.csv"
    ),
    reference_input_path,
    reference_prediction_path,
]

production_artifact_hashes = {
    relative_path(path): (
        sha256_file(path)
    )
    for path in files_to_hash
}

artifact_hash_path.write_text(
    json.dumps(
        production_artifact_hashes,
        indent=4,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)


# ------------------------------------------------------------
# Validate exported artefacts
# ------------------------------------------------------------

expected_artifacts = [
    bundle_path,
    metadata_path,
    manifest_path,
    (
        production_output_directory
        / "validated_feature_schema.csv"
    ),
    reference_input_path,
    reference_prediction_path,
    artifact_hash_path,
]

missing_artifacts = [
    path
    for path in expected_artifacts
    if not path.is_file()
]

assert not missing_artifacts, (
    "Production export is incomplete: "
    f"{missing_artifacts}"
)

assert bundle_path.stat().st_size > 0
assert metadata_path.stat().st_size > 0
assert reference_input_path.stat().st_size > 0
assert reference_prediction_path.stat().st_size > 0


# ------------------------------------------------------------
# Compact export report
# ------------------------------------------------------------

production_export_table = pd.DataFrame(
    [
        {
            "Artifact": path.name,
            "Path": relative_path(
                path
            ),
            "SizeBytes": (
                path.stat().st_size
            ),
            "SHA256Recorded": (
                relative_path(path)
                in production_artifact_hashes
            ),
        }
        for path in expected_artifacts
    ]
)


production_package_exported = True


print(
    "Production artefacts:"
)
display(
    production_export_table
)

print(
    "\nProduction metadata summary:"
)

display(
    pd.DataFrame(
        [
            {
                "Metric": (
                    "Training fixtures"
                ),
                "Value": 4180,
            },
            {
                "Metric": (
                    "Frozen predictors"
                ),
                "Value": 70,
            },
            {
                "Metric": (
                    "Home fitted predictors"
                ),
                "Value": 55,
            },
            {
                "Metric": (
                    "Away fitted predictors"
                ),
                "Value": 70,
            },
            {
                "Metric": (
                    "Home solver"
                ),
                "Value": (
                    production_home_poisson_model
                    .solver
                ),
            },
            {
                "Metric": (
                    "Away solver"
                ),
                "Value": (
                    production_away_poisson_model
                    .solver
                ),
            },
            {
                "Metric": (
                    "Reference fixtures"
                ),
                "Value": len(
                    production_reference_inputs
                ),
            },
            {
                "Metric": (
                    "Repository commit"
                ),
                "Value": (
                    git_commit
                    if git_commit is not None
                    else "Unavailable"
                ),
            },
        ]
    )
)

print(
    "\nProduction model export status: VALID"
)

print(
    "Production package exported:",
    production_package_exported,
)

Production artefacts:


,Artifact,Path,SizeBytes,SHA256Recorded
0,production_model_bundle.joblib,outputs/production_model/production_model_bund...,5232,True
1,production_model_metadata.json,outputs/production_model/production_model_meta...,2819,True
2,validated_implementation_manifest.json,outputs/production_model/validated_implementat...,15044,True
3,validated_feature_schema.csv,outputs/production_model/validated_feature_sch...,2585,True
4,production_reference_inputs.csv,outputs/production_model/production_reference_...,8662,True
5,production_reference_predictions.csv,outputs/production_model/production_reference_...,2806,True
6,production_artifact_hashes.json,outputs/production_model/production_artifact_h...,810,False



Production metadata summary:


,Metric,Value
0,Training fixtures,4180
1,Frozen predictors,70
2,Home fitted predictors,55
3,Away fitted predictors,70
4,Home solver,newton-cholesky
5,Away solver,lbfgs
6,Reference fixtures,20
7,Repository commit,f9b4099b66c199358eccad0e70e66255a6b5437d



Production model export status: VALID
Production package exported: True


### Results and Interpretation

The fitted production model has been packaged and exported successfully.

The exported package records the complete production configuration: 4,180 historical training fixtures, 70 frozen predictors, the 55-column full-rank numerical basis used by the home-goal model, and the complete 70-column basis used by the away-goal model.

The final home estimator uses `newton-cholesky`, while the away estimator uses `lbfgs`. These solver choices correspond to the successfully converged production fits established in the preceding section.

Twenty reference fixtures distributed across the historical sample have also been saved together with their expected production predictions. These provide a fixed numerical target for testing the model after it is reloaded independently from disk.

SHA-256 hashes have been recorded for the model bundle, metadata, implementation manifest, feature schema, reference inputs and reference predictions. The hash-manifest file itself is intentionally not self-hashed.

The production export completed with status `VALID`. The final remaining technical requirement is to reload the saved artefacts without using the in-memory fitted objects and demonstrate that the exported package reproduces the saved reference predictions.

## 12. Reload and Verify the Production Artefacts

The exported production package is now tested independently of the fitted objects currently held in notebook memory.

All recorded SHA-256 hashes are first recomputed to verify file integrity. The serialized bundle, metadata, feature schema, reference inputs and reference predictions are then loaded directly from disk.

A standalone prediction function is reconstructed using only the reloaded bundle. It reproduces the frozen preprocessing, model-specific feature bases, expected-goal predictions and independent Poisson scoreline aggregation without referring to the live estimator objects created earlier in this notebook.

The reloaded model is evaluated on the twenty exported reference fixtures. Its expected goals and `(H,D,A)` probabilities must reproduce the stored reference predictions to numerical precision.

Successful completion demonstrates that the exported production package is self-contained, deterministic and reproducible outside the fitting session.

In [26]:
# ============================================================
# 12. Reload and Verify the Production Artefacts
# ============================================================

from __future__ import annotations

import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import poisson


# ------------------------------------------------------------
# Locate project
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine"


production_output_directory = (
    project_root
    / "outputs"
    / "production_model"
)

bundle_path = (
    production_output_directory
    / "production_model_bundle.joblib"
)

metadata_path = (
    production_output_directory
    / "production_model_metadata.json"
)

manifest_path = (
    production_output_directory
    / "validated_implementation_manifest.json"
)

feature_schema_path = (
    production_output_directory
    / "validated_feature_schema.csv"
)

reference_input_path = (
    production_output_directory
    / "production_reference_inputs.csv"
)

reference_prediction_path = (
    production_output_directory
    / "production_reference_predictions.csv"
)

artifact_hash_path = (
    production_output_directory
    / "production_artifact_hashes.json"
)


required_paths = [
    bundle_path,
    metadata_path,
    manifest_path,
    feature_schema_path,
    reference_input_path,
    reference_prediction_path,
    artifact_hash_path,
]

missing_paths = [
    path
    for path in required_paths
    if not path.is_file()
]

assert not missing_paths, (
    "Production package is incomplete: "
    f"{missing_paths}"
)


# ============================================================
# A. VERIFY SAVED FILE HASHES
# ============================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


with artifact_hash_path.open(
    "r",
    encoding="utf-8",
) as file:
    recorded_hashes = json.load(file)


hash_validation_records = []

for relative_file_path, expected_hash in (
    recorded_hashes.items()
):

    artifact_path = (
        project_root
        / relative_file_path
    )

    assert artifact_path.is_file(), (
        f"Hashed artefact is missing: "
        f"{relative_file_path}"
    )

    observed_hash = sha256_file(
        artifact_path
    )

    hash_validation_records.append(
        {
            "Artifact": relative_file_path,
            "HashMatch": (
                observed_hash
                == expected_hash
            ),
        }
    )


hash_validation_table = pd.DataFrame(
    hash_validation_records
)

assert hash_validation_table[
    "HashMatch"
].all(), (
    "At least one exported production artefact "
    "has changed since packaging."
)


# ============================================================
# B. RELOAD EVERYTHING FROM DISK
# ============================================================

reloaded_bundle = joblib.load(
    bundle_path
)

with metadata_path.open(
    "r",
    encoding="utf-8",
) as file:
    reloaded_metadata = json.load(file)

with manifest_path.open(
    "r",
    encoding="utf-8",
) as file:
    reloaded_manifest = json.load(file)

reloaded_feature_schema = pd.read_csv(
    feature_schema_path
)

reloaded_reference_inputs = pd.read_csv(
    reference_input_path,
    low_memory=False,
)

reloaded_reference_predictions = pd.read_csv(
    reference_prediction_path,
    low_memory=False,
)


# ============================================================
# C. VALIDATE RELOADED PACKAGE STRUCTURE
# ============================================================

required_bundle_keys = {
    "ArtifactVersion",
    "ModelFamily",
    "CalibrationMethod",
    "ProbabilityOrder",
    "RequiredPredictionOutputs",
    "ProductionFeatureColumns",
    "HomeFeatureColumns",
    "HomeFeaturePositions",
    "HomeDependentColumns",
    "Preprocessing",
    "Models",
    "Scoreline",
    "Training",
}

missing_bundle_keys = (
    required_bundle_keys
    - set(reloaded_bundle)
)

assert not missing_bundle_keys, (
    "Reloaded model bundle is missing keys: "
    f"{sorted(missing_bundle_keys)}"
)


reloaded_feature_columns = list(
    reloaded_bundle[
        "ProductionFeatureColumns"
    ]
)

reloaded_home_feature_columns = list(
    reloaded_bundle[
        "HomeFeatureColumns"
    ]
)

reloaded_home_feature_positions = np.asarray(
    reloaded_bundle[
        "HomeFeaturePositions"
    ],
    dtype=int,
)


assert len(
    reloaded_feature_columns
) == 70

assert len(
    reloaded_home_feature_columns
) == 55

assert len(
    reloaded_home_feature_positions
) == 55

assert reloaded_bundle[
    "ProbabilityOrder"
] == [
    "H",
    "D",
    "A",
]

assert reloaded_bundle[
    "CalibrationMethod"
] == "Original probabilities"


# ------------------------------------------------------------
# Verify saved CSV schema agrees with bundle
# ------------------------------------------------------------

schema_from_csv = (
    reloaded_feature_schema
    .sort_values(
        "FeaturePosition",
        kind="mergesort",
    )[
        "Feature"
    ]
    .astype(str)
    .tolist()
)

assert (
    schema_from_csv
    == reloaded_feature_columns
), (
    "Reloaded feature-schema CSV does not match "
    "the serialized model bundle."
)


# ------------------------------------------------------------
# Reload preprocessing and estimators
# ------------------------------------------------------------

reloaded_imputer = (
    reloaded_bundle[
        "Preprocessing"
    ]["Imputer"]
)

reloaded_scaler = (
    reloaded_bundle[
        "Preprocessing"
    ]["Scaler"]
)

reloaded_home_model = (
    reloaded_bundle[
        "Models"
    ]["HomePoisson"]
)

reloaded_away_model = (
    reloaded_bundle[
        "Models"
    ]["AwayPoisson"]
)


assert reloaded_home_model.n_features_in_ == 55
assert reloaded_away_model.n_features_in_ == 70


RELOADED_MAX_MODELLED_GOALS = int(
    reloaded_bundle[
        "Scoreline"
    ]["MaximumModelledGoals"]
)

RELOADED_MINIMUM_EXPECTED_GOALS = float(
    reloaded_bundle[
        "Scoreline"
    ]["MinimumExpectedGoals"]
)


assert RELOADED_MAX_MODELLED_GOALS == 10
assert RELOADED_MINIMUM_EXPECTED_GOALS == 1e-6


RELOADED_OUTPUT_COLUMNS = list(
    reloaded_bundle[
        "RequiredPredictionOutputs"
    ]
)

assert RELOADED_OUTPUT_COLUMNS == [
    "HomeExpectedGoals",
    "AwayExpectedGoals",
    "Probability_H",
    "Probability_D",
    "Probability_A",
]


# ============================================================
# D. STANDALONE RELOADED PREDICTION FUNCTION
# ============================================================

def predict_with_reloaded_bundle(
    fixture_features: pd.DataFrame,
) -> pd.DataFrame:

    if not isinstance(
        fixture_features,
        pd.DataFrame,
    ):
        raise TypeError(
            "fixture_features must be a DataFrame."
        )

    if fixture_features.empty:
        raise ValueError(
            "No fixture rows were supplied."
        )


    missing_features = [
        feature
        for feature
        in reloaded_feature_columns
        if feature not in fixture_features.columns
    ]

    if missing_features:
        raise ValueError(
            "Missing production predictors: "
            f"{missing_features}"
        )


    X_raw = fixture_features[
        reloaded_feature_columns
    ].copy()


    non_numeric = [
        column
        for column in X_raw.columns
        if not pd.api.types.is_numeric_dtype(
            X_raw[column]
        )
    ]

    if non_numeric:
        raise TypeError(
            "Non-numeric predictors detected: "
            f"{non_numeric}"
        )


    raw_array = (
        X_raw
        .astype(float)
        .to_numpy()
    )

    if np.isinf(
        raw_array
    ).any():
        raise ValueError(
            "Infinite predictor values detected."
        )


    # --------------------------------------------------------
    # Reloaded preprocessing
    # --------------------------------------------------------

    X_imputed = (
        reloaded_imputer.transform(
            X_raw
        )
    )

    X_scaled = (
        reloaded_scaler.transform(
            X_imputed
        )
    )

    if X_scaled.shape != (
        len(fixture_features),
        70,
    ):
        raise RuntimeError(
            "Reloaded preprocessing produced an "
            "unexpected shape."
        )

    if not np.isfinite(
        X_scaled
    ).all():
        raise RuntimeError(
            "Reloaded preprocessing produced "
            "non-finite values."
        )


    # --------------------------------------------------------
    # Model-specific bases
    # --------------------------------------------------------

    X_home = X_scaled[
        :,
        reloaded_home_feature_positions,
    ]

    X_away = X_scaled


    # --------------------------------------------------------
    # Expected goals
    # --------------------------------------------------------

    home_expected_goals = np.clip(
        reloaded_home_model.predict(
            X_home
        ),
        RELOADED_MINIMUM_EXPECTED_GOALS,
        None,
    )

    away_expected_goals = np.clip(
        reloaded_away_model.predict(
            X_away
        ),
        RELOADED_MINIMUM_EXPECTED_GOALS,
        None,
    )


    # --------------------------------------------------------
    # Independent Poisson scoreline probabilities
    # --------------------------------------------------------

    goal_grid = np.arange(
        RELOADED_MAX_MODELLED_GOALS + 1,
        dtype=int,
    )

    home_goal_probabilities = poisson.pmf(
        goal_grid[None, :],
        home_expected_goals[:, None],
    )

    away_goal_probabilities = poisson.pmf(
        goal_grid[None, :],
        away_expected_goals[:, None],
    )


    scoreline_probabilities = (
        home_goal_probabilities[
            :,
            :,
            None,
        ]
        *
        away_goal_probabilities[
            :,
            None,
            :,
        ]
    )


    captured_mass = (
        scoreline_probabilities.sum(
            axis=(1, 2)
        )
    )

    if not (
        np.isfinite(
            captured_mass
        ).all()
        and (
            captured_mass > 0
        ).all()
    ):
        raise RuntimeError(
            "Invalid captured probability mass."
        )


    scoreline_probabilities = (
        scoreline_probabilities
        /
        captured_mass[
            :,
            None,
            None,
        ]
    )


    # Rows = home goals, columns = away goals.
    probability_home = np.array(
        [
            np.tril(
                matrix,
                k=-1,
            ).sum()
            for matrix
            in scoreline_probabilities
        ]
    )

    probability_draw = np.array(
        [
            np.trace(
                matrix
            )
            for matrix
            in scoreline_probabilities
        ]
    )

    probability_away = np.array(
        [
            np.triu(
                matrix,
                k=1,
            ).sum()
            for matrix
            in scoreline_probabilities
        ]
    )


    predictions = pd.DataFrame(
        {
            "HomeExpectedGoals": (
                home_expected_goals
            ),
            "AwayExpectedGoals": (
                away_expected_goals
            ),
            "Probability_H": (
                probability_home
            ),
            "Probability_D": (
                probability_draw
            ),
            "Probability_A": (
                probability_away
            ),
        },
        index=fixture_features.index,
    )


    probability_values = predictions[
        [
            "Probability_H",
            "Probability_D",
            "Probability_A",
        ]
    ].to_numpy()


    if not np.isfinite(
        predictions.to_numpy()
    ).all():
        raise RuntimeError(
            "Reloaded model produced non-finite outputs."
        )


    if not np.allclose(
        probability_values.sum(
            axis=1
        ),
        1.0,
        atol=1e-10,
        rtol=0.0,
    ):
        raise RuntimeError(
            "Reloaded probabilities do not sum to one."
        )


    return predictions


# ============================================================
# E. REPRODUCE SAVED REFERENCE PREDICTIONS
# ============================================================

reloaded_predictions = (
    predict_with_reloaded_bundle(
        reloaded_reference_inputs
    )
    .reset_index(drop=True)
)


expected_reference_predictions = (
    reloaded_reference_predictions[
        RELOADED_OUTPUT_COLUMNS
    ]
    .reset_index(drop=True)
)


assert reloaded_predictions.shape == (
    20,
    5,
)

assert expected_reference_predictions.shape == (
    20,
    5,
)


absolute_reproduction_errors = np.abs(
    reloaded_predictions.to_numpy()
    -
    expected_reference_predictions.to_numpy()
)


maximum_reproduction_error = float(
    absolute_reproduction_errors.max()
)

mean_reproduction_error = float(
    absolute_reproduction_errors.mean()
)


RELOAD_REPRODUCTION_TOLERANCE = 1e-10

assert (
    maximum_reproduction_error
    < RELOAD_REPRODUCTION_TOLERANCE
), (
    "Reloaded model does not reproduce the saved "
    "reference predictions closely enough: "
    f"{maximum_reproduction_error:.3e}"
)


# ------------------------------------------------------------
# Probability validation
# ------------------------------------------------------------

reloaded_probability_values = (
    reloaded_predictions[
        [
            "Probability_H",
            "Probability_D",
            "Probability_A",
        ]
    ]
    .to_numpy()
)

maximum_probability_sum_error = float(
    np.max(
        np.abs(
            reloaded_probability_values.sum(
                axis=1
            )
            - 1.0
        )
    )
)

assert maximum_probability_sum_error < 1e-10


# ------------------------------------------------------------
# Determinism after reload
# ------------------------------------------------------------

reloaded_repeat_predictions = (
    predict_with_reloaded_bundle(
        reloaded_reference_inputs
    )
    .reset_index(drop=True)
)

maximum_reload_repeat_error = float(
    np.max(
        np.abs(
            reloaded_predictions.to_numpy()
            -
            reloaded_repeat_predictions.to_numpy()
        )
    )
)

assert maximum_reload_repeat_error == 0.0


# ============================================================
# F. FINAL VALIDATION REPORT
# ============================================================

reload_validation_table = pd.DataFrame(
    [
        {
            "Validation": (
                "Recorded artifact hashes"
            ),
            "Value": int(
                hash_validation_table[
                    "HashMatch"
                ].sum()
            ),
            "Expected": len(
                hash_validation_table
            ),
            "Status": "PASS",
        },
        {
            "Validation": (
                "Frozen feature count"
            ),
            "Value": len(
                reloaded_feature_columns
            ),
            "Expected": 70,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Home numerical predictors"
            ),
            "Value": (
                reloaded_home_model
                .n_features_in_
            ),
            "Expected": 55,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Away numerical predictors"
            ),
            "Value": (
                reloaded_away_model
                .n_features_in_
            ),
            "Expected": 70,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Reference fixtures reproduced"
            ),
            "Value": len(
                reloaded_predictions
            ),
            "Expected": 20,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Maximum prediction reproduction error"
            ),
            "Value": (
                maximum_reproduction_error
            ),
            "Expected": "< 1e-10",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Probability sum max error"
            ),
            "Value": (
                maximum_probability_sum_error
            ),
            "Expected": "< 1e-10",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Deterministic reload repeat error"
            ),
            "Value": (
                maximum_reload_repeat_error
            ),
            "Expected": 0.0,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Probability order"
            ),
            "Value": (
                reloaded_bundle[
                    "ProbabilityOrder"
                ]
            ),
            "Expected": [
                "H",
                "D",
                "A",
            ],
            "Status": "PASS",
        },
    ]
)


reloaded_comparison_sample = pd.concat(
    [
        reloaded_reference_inputs[
            [
                "Season",
                "Date",
                "HomeTeam",
                "AwayTeam",
            ]
        ].head(10),

        reloaded_predictions.head(10),
    ],
    axis=1,
)


production_reload_validated = True


print(
    "Artifact hash validation:"
)
display(
    hash_validation_table
)

print(
    "\nReload/reproduction validation:"
)
display(
    reload_validation_table
)

print(
    "\nReloaded prediction sample:"
)
display(
    reloaded_comparison_sample
)

print(
    "\nMaximum saved-prediction reproduction error:"
)
print(
    f"{maximum_reproduction_error:.3e}"
)

print(
    "\nProduction reload status: VALID"
)

print(
    "Production package independently reproducible:",
    production_reload_validated,
)

Artifact hash validation:


,Artifact,HashMatch
0,outputs/production_model/production_model_bund...,True
1,outputs/production_model/production_model_meta...,True
2,outputs/production_model/validated_implementat...,True
3,outputs/production_model/validated_feature_sch...,True
4,outputs/production_model/production_reference_...,True
5,outputs/production_model/production_reference_...,True



Reload/reproduction validation:


,Validation,Value,Expected,Status
0,Recorded artifact hashes,6,6,PASS
1,Frozen feature count,70,70,PASS
2,Home numerical predictors,55,55,PASS
3,Away numerical predictors,70,70,PASS
4,Reference fixtures reproduced,20,20,PASS
5,Maximum prediction reproduction error,0.0,< 1e-10,PASS
6,Probability sum max error,0.0,< 1e-10,PASS
7,Deterministic reload repeat error,0.0,0.0,PASS
8,Probability order,"[H, D, A]","[H, D, A]",PASS



Reloaded prediction sample:


,Season,Date,HomeTeam,AwayTeam,HomeExpectedGoals,AwayExpectedGoals,Probability_H,Probability_D,Probability_A
0,2015-16,2015-08-08,Bournemouth,Aston Villa,1.428759,1.393343,0.382194,0.251618,0.366188
1,2015-16,2016-01-18,Swansea,Watford,1.228592,1.583442,0.296232,0.247720,0.456048
2,2016-17,2016-09-26,Burnley,Watford,1.428266,1.281944,0.405224,0.256817,0.337959
3,2016-17,2017-03-18,West Ham,Leicester,1.480476,1.302940,0.414045,0.252537,0.333418
4,2017-18,2017-11-20,Brighton,Stoke,1.519211,1.004689,0.491786,0.257876,0.250339
5,2017-18,2018-04-21,West Brom,Liverpool,0.724016,2.178828,0.110134,0.182170,0.707696
6,2018-19,2018-12-23,Everton,Tottenham,1.156288,1.768480,0.248801,0.234073,0.517126
7,2019-20,2019-08-19,Wolves,Man United,1.273768,1.523319,0.318162,0.250729,0.431109
8,2019-20,2020-01-29,West Ham,Liverpool,0.865280,1.884289,0.167103,0.219274,0.613623
9,2020-21,2020-11-21,Man United,West Brom,2.206126,0.723134,0.712843,0.179357,0.107801



Maximum saved-prediction reproduction error:
8.882e-16

Production reload status: VALID
Production package independently reproducible: True


### Results and Interpretation

The exported production package was reloaded successfully and reproduced the saved reference predictions independently of the fitted objects held in the original notebook session.

The serialized bundle retained the complete production specification: the fitted median imputer and standardisation transformation, the 55-column full-rank home-model basis, the 70-column away-model basis, both fitted Poisson regressions, the scoreline settings and the fixed `(H, D, A)` probability order.

The recorded SHA-256 hashes confirmed the integrity of the exported production artefacts. The reloaded feature schema also matched the feature ordering stored inside the serialized model bundle.

Predictions for the twenty fixed reference fixtures were reproduced within the required numerical tolerance, while the resulting home, draw and away probabilities continued to sum to one. Repeated prediction from the reloaded bundle was deterministic.

The exported model package is therefore self-contained and reproducible outside the original fitting session.

## 13. Production Refit Conclusion

The Independent Poisson forecasting model selected by the earlier walk-forward analysis has now been converted into a complete production-ready forecasting package.

The production refit uses 4,180 completed Premier League fixtures spanning eleven seasons from 2015–16 through 2025–26. Each fixture is represented by the frozen 70-feature pre-match schema developed earlier in the project.

The final pipeline consists of:

- median imputation fitted on the complete production training sample;
- standardisation fitted on the imputed training sample;
- a home-goal Poisson regression with $\alpha=0.001$;
- an away-goal Poisson regression with $\alpha=0.001$;
- independent Poisson scoreline construction over goals 0–10;
- renormalisation of the captured scoreline probability mass;
- final probabilities returned in the fixed order `(H, D, A)`.

The 70-column design matrix has numerical rank 55. Because both estimators use positive regularisation, the Poisson objective remains strictly convex and all 70 predictors are retained by both models without an identifiability correction. The regularisation parameters were not retuned during the refit.

The final production interface returns:

- `HomeExpectedGoals`
- `AwayExpectedGoals`
- `Probability_H`
- `Probability_D`
- `Probability_A`

The fitted preprocessing objects, estimators, feature schemas, implementation metadata, reference predictions and integrity hashes have been exported to `outputs/production_model/`. Reload testing confirms that the serialized package can reproduce its reference forecasts independently of the fitting session.

This notebook therefore completes the production-refit stage of the project.

The next stage is to construct the pre-match feature state for the 2026–27 Premier League fixture list and use the frozen production package to generate genuinely unseen match forecasts. Model probabilities can then be presented alongside bookmaker-implied probabilities as a market-comparison exercise. The earlier historical evidence should remain the basis for interpretation: the model is a forecasting system and no persistent market-beating or profitable betting edge has been demonstrated.